# Experimento 01: cajas heuristicas y curva dinamica

**Estado:** rama experimental conservada para trazabilidad, no estrategia final de entrega.

**Deriva de:** 03_multiclase_estratificado_medsam -> busqueda de prompts geometricos antes de usar red de cajas.

**Por que se conserva:** Se conserva porque explica el salto desde reglas geometricas hacia una red detectora de cajas.

Este notebook se deja con salidas limpias para que GitHub sea liviano. La estrategia principal queda en `notebooks/`; esta carpeta explica los caminos que se probaron y por que no todos terminaron como modelo final.

<!-- codex-experimento-conservado -->


# Caja vertebras: experimento 12

Version reducida para trabajar solo en deteccion automatica de cajas de vertebras y rendimiento con MedSAM. Se quitaron los escenarios de entrenamiento, data augmentation y test final para mantener el notebook enfocado.

## 1. Configuracion base
Rutas, clases y artefactos ya exportados para MedSAM.

In [ ]:
import os, json, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED)
print('âœ… Entorno listo')

# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# DATASET ACTIVO
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
DATASET_ROOT = Path('C:/Users/luisf/Downloads/ProyectoFinal/Scoliosis_Dataset')

# Subcarpetas fijas del dataset limpio
DIR_NORMAL      = DATASET_ROOT / 'Normal'
DIR_SCOLIOSIS   = DATASET_ROOT / 'Scoliosis'
DIR_MASK_ID     = DATASET_ROOT / 'LabelMultiClass_ID_PNG'
DIR_MASK_BIN    = DATASET_ROOT / 'LabelBinaryJPG'
DIR_METRICS     = DATASET_ROOT / 'RadiographMetrics'
COBB_SUMMARY_PATH = DIR_METRICS / 'metricas_cobb_resumen_recalculado.csv'
DATASET_INDEX_PATH = DATASET_ROOT / 'indice_dataset.csv'
LABELS_DICT_PATH = DATASET_ROOT / 'diccionario_etiquetas_T1_T12_L1_L5.json'

# Salida separada para no mezclarla con exportaciones del dataset anterior
OUTPUT_ROOT = DATASET_ROOT.parent / 'dataset_procesado_scoliosis_medsam'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Raiz MedSAM correcta para esta version reducida.
# Se define explicitamente para evitar que la busqueda automatica tome dataset_procesado/medsam viejo.
OUT_MEDSAM = OUTPUT_ROOT / 'medsam'

# Cargar diccionario oficial de la versiÃ³n limpia T1â€“T12/L1â€“L5
with open(LABELS_DICT_PATH, encoding='utf-8') as f:
    LABELS_DICT = json.load(f)

MAPEO_ID = {int(k): v for k, v in LABELS_DICT['mascara_multiclase_id_png'].items()}
CLASES = {k: v for k, v in MAPEO_ID.items() if 1 <= k <= 17}
N_CLASES = len(CLASES)
NOMBRES_CLASES = [CLASES[i] for i in range(1, N_CLASES + 1)]

print(f'Dataset:  {DATASET_ROOT}')
print(f'Existe:   {DATASET_ROOT.exists()}')
print(f'Indice:   {DATASET_INDEX_PATH.exists()}')
print(f'COCO:     no encontrado en esta version')
print(f'Clases:   {N_CLASES} â†’ {NOMBRES_CLASES}')


In [ ]:
# ==========================================
# 2) CLASES DEL DATASET Y CLASES OBJETIVO
# ==========================================

# La version limpia ya usa IDs consecutivos:
# 0 = fondo, 1..12 = T1..T12, 13..17 = L1..L5.
CLASES_OBJETIVO = NOMBRES_CLASES.copy()
N_CLASES = len(CLASES_OBJETIVO)

VERTEBRA_TO_ID = {nombre: idx for idx, nombre in MAPEO_ID.items() if idx != 0}
ID_TO_VERTEBRA = {idx: nombre for nombre, idx in VERTEBRA_TO_ID.items()}

# En esta version el ID real de la mascara y el ID local de evaluacion son iguales.
CLASS_TO_ID = VERTEBRA_TO_ID.copy()
ID_TO_CLASS = {idx: nombre for nombre, idx in CLASS_TO_ID.items()}

print("=" * 80)
print("Configuracion actual para MEDSAM")
print("=" * 80)
print("Dataset activo: Scoliosis_Dataset limpio")
print("Clases objetivo (17):", CLASES_OBJETIVO)
print("IDs de mascara para T1-L5:")
for c in CLASES_OBJETIVO:
    print(f"  {c:>3} -> {VERTEBRA_TO_ID[c]}")
print("=" * 80)

# ==========================================
# 3) FUNCIONES AUXILIARES PARA BUSCAR ARCHIVOS
# ==========================================

def buscar_archivo(nombre_archivo, raiz_inicial=Path("."), max_resultados=20):
    encontrados = []
    for p in raiz_inicial.rglob(nombre_archivo):
        encontrados.append(p)
        if len(encontrados) >= max_resultados:
            break
    return encontrados

EXTS_IMG = [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]

def buscar_archivo_por_stem(carpeta, stem):
    for ext in EXTS_IMG:
        p = carpeta / f"{stem}{ext}"
        if p.exists():
            return p
    return None


In [ ]:
# ==========================================
# 4) DETECTAR LA RAIZ DE MEDSAM
# ==========================================

MEDSAM_DATA_ROOT = None

variables_candidatas = [
    "MEDSAM_DATA_ROOT",
    "medsam_data_root",
    "MEDSAM_ROOT",
    "medsam_root",
    "OUT_MEDSAM",
    "out_medsam",
    "EXPORT_MEDSAM_DIR",
    "export_medsam_dir"
]

for var_name in variables_candidatas:
    if var_name in globals():
        try:
            MEDSAM_DATA_ROOT = Path(globals()[var_name])
            if MEDSAM_DATA_ROOT.exists():
                print(f"Se usara {var_name} = {MEDSAM_DATA_ROOT}")
                break
        except Exception:
            pass

if MEDSAM_DATA_ROOT is None:
    candidatos_raiz = []

    if "OUTPUT_ROOT" in globals():
        candidatos_raiz.append(Path(OUTPUT_ROOT) / "medsam")

    if "DATASET_ROOT" in globals():
        candidatos_raiz.append(DATASET_ROOT.parent / "dataset_procesado_scoliosis_medsam" / "medsam")

    candidatos_raiz.extend([
        Path("C:/Users/luisf/Downloads/ProyectoFinal/dataset_procesado_scoliosis_medsam/medsam"),
        Path("dataset_procesado_scoliosis_medsam/medsam"),
    ])

    for candidato in candidatos_raiz:
        candidato = Path(candidato)
        if (candidato / "train" / "prompts.json").exists() and (candidato / "val" / "prompts.json").exists():
            MEDSAM_DATA_ROOT = candidato
            print(f"MEDSAM_DATA_ROOT definido por ruta limpia: {MEDSAM_DATA_ROOT}")
            break

if MEDSAM_DATA_ROOT is None:
    prompts_encontrados = buscar_archivo("prompts.json", Path("."))
    prompts_medsam = [p for p in prompts_encontrados if "dataset_procesado_scoliosis_medsam" in str(p).lower()]

    if len(prompts_medsam) == 0:
        prompts_medsam = [p for p in prompts_encontrados if "medsam" in str(p).lower()]

    if len(prompts_medsam) > 0:
        MEDSAM_DATA_ROOT = prompts_medsam[0].parent.parent
        print(f"MEDSAM_DATA_ROOT detectado automaticamente: {MEDSAM_DATA_ROOT}")
    elif len(prompts_encontrados) > 0:
        MEDSAM_DATA_ROOT = prompts_encontrados[0].parent.parent
        print(f"MEDSAM_DATA_ROOT detectado automaticamente: {MEDSAM_DATA_ROOT}")
    else:
        raise FileNotFoundError(
            "No se encontro prompts.json. "
            "Corre primero la exportacion a MEDSAM o indica el path exacto."
        )

MEDSAM_DATA_ROOT = Path(MEDSAM_DATA_ROOT)
print("MEDSAM_DATA_ROOT final:", MEDSAM_DATA_ROOT)

# ==========================================
# 5) RESOLVER SPLITS Y PATHS
# ==========================================

SPLITS = ["train", "val", "test"]

def resolver_paths_split(split):
    split_dir = MEDSAM_DATA_ROOT / split
    if not split_dir.exists():
        raise FileNotFoundError(f"No existe el split: {split_dir}")

    prompts_path = split_dir / "prompts.json"
    if not prompts_path.exists():
        raise FileNotFoundError(f"No existe prompts.json en: {split_dir}")

    candidatos_img = ["images", "imgs", "image", "jpg", "png"]
    candidatos_mask = ["masks", "labels", "mask", "annotations"]

    image_dir = None
    mask_dir = None

    for c in candidatos_img:
        p = split_dir / c
        if p.exists() and p.is_dir():
            image_dir = p
            break

    for c in candidatos_mask:
        p = split_dir / c
        if p.exists() and p.is_dir():
            mask_dir = p
            break

    if image_dir is None:
        raise FileNotFoundError(f"No encontre carpeta de imagenes dentro de {split_dir}")

    if mask_dir is None:
        raise FileNotFoundError(f"No encontre carpeta de mascaras dentro de {split_dir}")

    return split_dir, image_dir, mask_dir, prompts_path

SPLIT_INFO = {}
for split in SPLITS:
    split_dir, image_dir, mask_dir, prompts_path = resolver_paths_split(split)
    SPLIT_INFO[split] = {
        "split_dir": split_dir,
        "image_dir": image_dir,
        "mask_dir": mask_dir,
        "prompts_path": prompts_path
    }

print("\nResumen de paths detectados:")
for split in SPLITS:
    print(f"\n[{split}]")
    print(" split_dir   :", SPLIT_INFO[split]["split_dir"])
    print(" image_dir   :", SPLIT_INFO[split]["image_dir"])
    print(" mask_dir    :", SPLIT_INFO[split]["mask_dir"])
    print(" prompts_path:", SPLIT_INFO[split]["prompts_path"])

# ==========================================
# 6) CARGAR PROMPTS Y REVISAR SU ESTRUCTURA
# ==========================================

PROMPTS = {}
for split in SPLITS:
    with open(SPLIT_INFO[split]["prompts_path"], "r", encoding="utf-8") as f:
        PROMPTS[split] = json.load(f)

print("\nCantidad de entradas en prompts.json:")
for split in SPLITS:
    print(f"  {split}: {len(PROMPTS[split])}")

for split in SPLITS:
    print("\n" + "=" * 80)
    print(f"SPLIT: {split}")
    print("type(PROMPTS[split]):", type(PROMPTS[split]))

    if isinstance(PROMPTS[split], list):
        print("len:", len(PROMPTS[split]))
        if len(PROMPTS[split]) > 0:
            print("type primer elemento:", type(PROMPTS[split][0]))
            if isinstance(PROMPTS[split][0], dict):
                print("keys primer elemento:", list(PROMPTS[split][0].keys()))
                print("primer elemento completo:")
                print(PROMPTS[split][0])


In [ ]:
# ==========================================
# 7) FUNCIONES DE CARGA, FILTRADO Y MASCARAS
# ==========================================

def cargar_imagen(path_imagen):
    img = Image.open(path_imagen).convert("RGB")
    return np.array(img)

def cargar_mascara(path_mascara):
    mask = Image.open(path_mascara)
    return np.array(mask).astype(np.int32)

def normalizar_nombre_vertebra(nombre):
    if nombre is None:
        return None
    nombre = str(nombre).strip().upper()
    return nombre if nombre in CLASES_OBJETIVO else None

def filtrar_prompts_t1_l5(prompts_split):
    # Nota metodologica: los prompts se leen del archivo exportado; en esta version vienen de las mascaras GT.
    """
    Estructura real:
    [
        {
            "patient_id": "...",
            "prompts": {
                "1": {"vertebra": "T1", "bbox_xyxy": [...]},
                ...
            }
        },
        ...
    ]
    """
    prompts_filtrados = {}

    if not isinstance(prompts_split, list):
        raise TypeError(f"Se esperaba list en prompts_split y llegÃ³ {type(prompts_split)}")

    for item in prompts_split:
        if not isinstance(item, dict):
            continue

        patient_id = item.get("patient_id")
        prompts_item = item.get("prompts")

        if patient_id is None or prompts_item is None:
            continue

        sub = {}

        if isinstance(prompts_item, dict):
            for _, info in prompts_item.items():
                if not isinstance(info, dict):
                    continue

                nombre = normalizar_nombre_vertebra(info.get("vertebra"))
                if nombre is not None:
                    sub[nombre] = info

        elif isinstance(prompts_item, list):
            for info in prompts_item:
                if not isinstance(info, dict):
                    continue

                nombre = normalizar_nombre_vertebra(info.get("vertebra"))
                if nombre is not None:
                    sub[nombre] = info

        if len(sub) > 0:
            prompts_filtrados[patient_id] = sub

    return prompts_filtrados

def construir_mask_binaria_vertebra(mask_multiclase, vertebra_objetivo):
    """
    Convierte la mascara multiclase global en una mascara binaria para una sola vertebra.
    """
    if vertebra_objetivo not in VERTEBRA_TO_ID:
        raise ValueError(f"Vertebra no conocida: {vertebra_objetivo}")

    vertebra_id_real = VERTEBRA_TO_ID[vertebra_objetivo]
    mask_bin = (mask_multiclase == vertebra_id_real).astype(np.uint8)
    return mask_bin

def construir_mask_multiclase_t1_l5(mask_multiclase):
    """
    Devuelve la mascara T1-L5 en IDs locales 1..17:
    0 = fondo
    1..17 = T1..L5
    """
    # En Scoliosis_Dataset las mascaras ya vienen en 0..17.
    # Se fuerza uint8 y se eliminan posibles IDs fuera del diccionario por seguridad.
    nueva = mask_multiclase.astype(np.uint8).copy()
    nueva[~np.isin(nueva, list(range(0, N_CLASES + 1)))] = 0
    return nueva

def extraer_bbox_desde_mask(mask_binaria):
    ys, xs = np.where(mask_binaria > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None
    x0, x1 = xs.min(), xs.max()
    y0, y1 = ys.min(), ys.max()
    return [int(x0), int(y0), int(x1), int(y1)]

def resolver_paths_muestra(split, patient_id):
    image_dir = SPLIT_INFO[split]["image_dir"]
    mask_dir = SPLIT_INFO[split]["mask_dir"]

    path_imagen = buscar_archivo_por_stem(image_dir, patient_id)
    path_mascara = buscar_archivo_por_stem(mask_dir, patient_id)

    if path_imagen is None:
        raise FileNotFoundError(f"No encontre imagen para {patient_id} en {image_dir}")

    if path_mascara is None:
        raise FileNotFoundError(f"No encontre mascara para {patient_id} en {mask_dir}")

    return path_imagen, path_mascara

# ==========================================
# 8) FILTRAR T1-L5 Y TOMAR UNA MUESTRA DE PRUEBA
# ==========================================

PROMPTS_FILTRADOS = {
    split: filtrar_prompts_t1_l5(PROMPTS[split])
    for split in SPLITS
}

print("\nCantidad de imagenes con prompts T1-L5:")
for split in SPLITS:
    print(f"  {split}: {len(PROMPTS_FILTRADOS[split])}")

for split in SPLITS:
    print("\n" + "=" * 80)
    print(f"SPLIT: {split}")
    print("imagenes filtradas:", len(PROMPTS_FILTRADOS[split]))

    if len(PROMPTS_FILTRADOS[split]) > 0:
        sample_key = next(iter(PROMPTS_FILTRADOS[split]))
        print("sample_key:", sample_key)
        print("vertebras disponibles:", list(PROMPTS_FILTRADOS[split][sample_key].keys()))
        primera_vertebra = next(iter(PROMPTS_FILTRADOS[split][sample_key]))
        print("ejemplo prompt:")
        print(PROMPTS_FILTRADOS[split][sample_key][primera_vertebra])

split = "train"

if len(PROMPTS_FILTRADOS[split]) == 0:
    raise RuntimeError("No se encontraron prompts T1-L5 en train.")

sample_key = next(iter(PROMPTS_FILTRADOS[split].keys()))
path_imagen, path_mascara = resolver_paths_muestra(split, sample_key)

imagen = cargar_imagen(path_imagen)
mascara = cargar_mascara(path_mascara)

# mascara remapeada solo para visualizacion T1-L5
mascara_t1_l5 = construir_mask_multiclase_t1_l5(mascara)

prompts_sample = PROMPTS_FILTRADOS[split][sample_key]

print("\nMuestra seleccionada:")
print(" split       :", split)
print(" sample_key  :", sample_key)
print(" path_imagen :", path_imagen)
print(" path_mascara:", path_mascara)
print(" vertebras en prompts:", list(prompts_sample.keys()))
print(" shape imagen:", imagen.shape)
print(" shape mask original :", mascara.shape)
print(" unique mask original:", np.unique(mascara)[:30])
print(" unique mask T1-L5   :", np.unique(mascara_t1_l5)[:30])

fig, ax = plt.subplots(1, 2, figsize=(12, 6))

ax[0].imshow(imagen)
ax[0].set_title("Imagen")
ax[0].axis("off")

ax[1].imshow(mascara_t1_l5, cmap="nipy_spectral")
ax[1].set_title("Mascara remapeada T1-L5")
ax[1].axis("off")

plt.tight_layout()
plt.show()


## 2. MedSAM y metricas
Carga del predictor y funciones necesarias para reconstruccion semantica y evaluacion.

In [ ]:
# ==========================================
# CARGAR MODELO MEDSAM
# ==========================================

import torch
from segment_anything import sam_model_registry
from segment_anything import SamPredictor

# Ajusta este path si es necesario (segÃºn tu HTML ya lo tenÃ­as)
MedSAM_CKPT_PATH = "C:/Users/luisf/MedSAM/work_dir/MedSAM/medsam_vit_b.pth"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Cargar modelo
medsam_model = sam_model_registry["vit_b"](checkpoint=MedSAM_CKPT_PATH)
medsam_model = medsam_model.to(device)
medsam_model.eval()

# Crear predictor
predictor = SamPredictor(medsam_model)

print("MedSAM cargado correctamente")


In [ ]:
# ==========================================
# 24) FUNCIONES DE EVALUACION GLOBAL
# ==========================================

def expand_bbox_xyxy_pequena(bbox, img_shape, frac_x=0.04, frac_y=0.06):
    x0, y0, x1, y1 = bbox
    h, w = img_shape[:2]

    bw = x1 - x0
    bh = y1 - y0

    pad_x = int(round(bw * frac_x))
    pad_y = int(round(bh * frac_y))

    x0n = max(0, x0 - pad_x)
    y0n = max(0, y0 - pad_y)
    x1n = min(w - 1, x1 + pad_x)
    y1n = min(h - 1, y1 + pad_y)

    return [x0n, y0n, x1n, y1n]

def evaluar_pred(mask_pred, mask_gt, eps=1e-8):
    inter = np.logical_and(mask_pred == 1, mask_gt == 1).sum()
    union = np.logical_or(mask_pred == 1, mask_gt == 1).sum()
    dice = (2 * inter + eps) / (mask_pred.sum() + mask_gt.sum() + eps)
    iou = (inter + eps) / (union + eps)
    return float(dice), float(iou)

# ==========================================
# 37) VERSION CON MAPA DE SCORE PARA SOLAPAMIENTOS
# ==========================================

def reconstruir_mascara_semantica_medsam_con_score(
    imagen,
    mascara_gt_multiclase,
    prompts_sample,
    predictor,
    frac_x=0.04,
    frac_y=0.06,
    return_quality=False
):
    predictor.set_image(imagen)

    h, w = imagen.shape[:2]
    mask_sem_pred = np.zeros((h, w), dtype=np.uint8)
    score_map = np.zeros((h, w), dtype=np.float32)
    pred_count = np.zeros((h, w), dtype=np.uint8)
    mask_sem_gt = construir_mask_multiclase_t1_l5(mascara_gt_multiclase)

    detalles = []

    for vertebra_objetivo in CLASES_OBJETIVO:
        if vertebra_objetivo not in prompts_sample:
            continue

        # Cada bbox usada aqui proviene del prompt exportado. Si el prompt fue creado desde GT,
        # la metrica evalua segmentacion condicionada por una caja ideal, no deteccion automatica.
        prompt_info = prompts_sample[vertebra_objetivo]
        bbox_original = prompt_info["bbox_xyxy"]
        bbox_expandida = expand_bbox_xyxy_pequena(
            bbox_original,
            imagen.shape,
            frac_x=frac_x,
            frac_y=frac_y
        )

        box_np = np.array(bbox_expandida, dtype=np.float32)[None, :]

        masks, scores, logits = predictor.predict(
            box=box_np,
            multimask_output=False
        )

        mask_pred_bin = masks[0].astype(np.uint8)
        score_pred = float(scores[0])
        pred_count += mask_pred_bin

        id_local = CLASS_TO_ID[vertebra_objetivo]

        # En zonas solapadas se conserva la vertebra con mayor score interno de MedSAM.
        # Este score no es Dice/IoU; solo se usa como regla de desempate entre predicciones.
        update_idx = (mask_pred_bin == 1) & (score_pred > score_map)
        mask_sem_pred[update_idx] = id_local
        score_map[update_idx] = score_pred

        mask_gt_bin = construir_mask_binaria_vertebra(mascara_gt_multiclase, vertebra_objetivo)

        inter = np.logical_and(mask_pred_bin == 1, mask_gt_bin == 1).sum()
        union = np.logical_or(mask_pred_bin == 1, mask_gt_bin == 1).sum()

        dice_v = (2 * inter + 1e-8) / (mask_pred_bin.sum() + mask_gt_bin.sum() + 1e-8)
        iou_v = (inter + 1e-8) / (union + 1e-8)

        detalles.append({
            "vertebra": vertebra_objetivo,
            "id_local": id_local,
            "id_real": VERTEBRA_TO_ID[vertebra_objetivo],
            "bbox_original": bbox_original,
            "bbox_expandida": bbox_expandida,
            "score_medsam": score_pred,
            "pix_gt": int(mask_gt_bin.sum()),
            "pix_pred": int(mask_pred_bin.sum()),
            "dice": float(dice_v),
            "iou": float(iou_v),
            "pred_vacia": bool(mask_pred_bin.sum() == 0),
            "gt_vacia": bool(mask_gt_bin.sum() == 0)
        })

    quality = {
        "overlap_pixels": int((pred_count > 1).sum()),
        "pred_pixels": int((pred_count > 0).sum()),
        "overlap_fraction_pred": float((pred_count > 1).sum() / max((pred_count > 0).sum(), 1)),
        "n_predicciones_vacias": int(sum(d["pred_vacia"] for d in detalles)),
        "n_gt_vacias": int(sum(d["gt_vacia"] for d in detalles))
    }

    if return_quality:
        return mask_sem_pred, mask_sem_gt, detalles, score_map, quality

    return mask_sem_pred, mask_sem_gt, detalles, score_map


def dice_multiclase_promedio(mask_pred, mask_gt, clases_ids):
    """Promedio macro: cada vertebra pesa igual, independiente de su area."""
    dices = []
    for cid in clases_ids:
        pred_bin = (mask_pred == cid).astype(np.uint8)
        gt_bin = (mask_gt == cid).astype(np.uint8)
        if gt_bin.sum() == 0 and pred_bin.sum() == 0:
            continue
        inter = np.logical_and(pred_bin == 1, gt_bin == 1).sum()
        dice = (2 * inter + 1e-8) / (pred_bin.sum() + gt_bin.sum() + 1e-8)
        dices.append(float(dice))
    return np.mean(dices) if len(dices) > 0 else np.nan


def iou_multiclase_promedio(mask_pred, mask_gt, clases_ids):
    ious = []
    for cid in clases_ids:
        pred_bin = (mask_pred == cid).astype(np.uint8)
        gt_bin = (mask_gt == cid).astype(np.uint8)
        if gt_bin.sum() == 0 and pred_bin.sum() == 0:
            continue
        inter = np.logical_and(pred_bin == 1, gt_bin == 1).sum()
        union = np.logical_or(pred_bin == 1, gt_bin == 1).sum()
        iou = (inter + 1e-8) / (union + 1e-8)
        ious.append(float(iou))
    return np.mean(ious) if len(ious) > 0 else np.nan


## 3. Cajas automaticas de vertebras
Plantilla anatomica desde train, deteccion del eje curvo y generacion de cajas T1-L5.

In [ ]:
# ============================================================
# 12.1) ConfiguraciÃ³n inicial y normalizaciÃ³n de prompts
# ============================================================

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

PROMPTS_BASE = PROMPTS_FILTRADOS if "PROMPTS_FILTRADOS" in globals() else PROMPTS

if "VERTEBRA_TO_ID" not in globals():
    VERTEBRA_TO_ID = {v: i + 1 for i, v in enumerate(CLASES_OBJETIVO)}

ID_TO_VERTEBRA = {v: k for k, v in VERTEBRA_TO_ID.items()}

if "N_CLASES" not in globals():
    N_CLASES = len(CLASES_OBJETIVO)


def normalizar_prompts_split(prompts_split):
    """
    Convierte los prompts a formato:
    {patient_id: prompts_sample}
    """

    if isinstance(prompts_split, dict):
        return prompts_split

    if isinstance(prompts_split, list):
        return {
            item["patient_id"]: item["prompts"]
            for item in prompts_split
        }

    raise TypeError("Formato de prompts no reconocido.")


PROMPTS_DICC = {
    split: normalizar_prompts_split(PROMPTS_BASE[split])
    for split in PROMPTS_BASE.keys()
}

print("Splits disponibles:", PROMPTS_DICC.keys())
print("Ejemplo val:", list(PROMPTS_DICC["val"].keys())[:3])
# ============================================================
# 12.2) Plantilla anatÃ³mica desde train
# ============================================================

def obtener_vertebra_desde_prompt(clave_prompt, info_prompt):
    """
    Obtiene el nombre de la vÃ©rtebra desde el prompt.
    """

    if "vertebra" in info_prompt:
        return info_prompt["vertebra"]

    try:
        clase_id = int(clave_prompt)
        return ID_TO_VERTEBRA[clase_id]
    except Exception:
        return clave_prompt


def construir_template_bbox_train(prompts_dicc, split_template="train"):
    """
    Construye una plantilla promedio de posiciÃ³n y tamaÃ±o por vÃ©rtebra.

    Se usa solo train. No usa val/test para construir la plantilla.
    """

    filas = []

    for patient_id, prompts_sample in prompts_dicc[split_template].items():

        for clave_prompt, info_prompt in prompts_sample.items():

            vertebra = obtener_vertebra_desde_prompt(clave_prompt, info_prompt)

            if vertebra not in CLASES_OBJETIVO:
                continue

            x0, y0, x1, y1 = info_prompt["bbox_xyxy"]

            filas.append({
                "patient_id": patient_id,
                "vertebra": vertebra,
                "id_real": VERTEBRA_TO_ID[vertebra],
                "cx_rel": ((x0 + x1) / 2) / 1024,
                "cy_rel": ((y0 + y1) / 2) / 1024,
                "w_rel": (x1 - x0) / 1024,
                "h_rel": (y1 - y0) / 1024
            })

    df_template = pd.DataFrame(filas)

    template_bbox = (
        df_template
        .groupby(["vertebra", "id_real"], as_index=False)
        .agg(
            cx_rel=("cx_rel", "median"),
            cy_rel=("cy_rel", "median"),
            w_rel=("w_rel", "median"),
            h_rel=("h_rel", "median")
        )
        .sort_values("id_real")
        .reset_index(drop=True)
    )

    faltantes = [v for v in CLASES_OBJETIVO if v not in template_bbox["vertebra"].tolist()]

    if len(faltantes) > 0:
        print("Advertencia: faltan vÃ©rtebras en la plantilla:", faltantes)

    return template_bbox, df_template


template_bbox_auto, df_template_bbox_train = construir_template_bbox_train(
    PROMPTS_DICC,
    split_template="train"
)

display(template_bbox_auto)

In [ ]:
# ============================================================
# 12.3) Utilidades para imagen, suavizado y ROI
# ============================================================

def imagen_a_gris_uint8(img_rgb):
    """
    Convierte imagen RGB o gris a uint8.
    """

    if img_rgb.ndim == 3:
        gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    else:
        gray = img_rgb.copy()

    if gray.max() <= 1.0:
        gray = gray * 255

    return gray.astype(np.uint8)


def normalizar_01(x, eps=1e-8):
    x = x.astype(np.float32)
    return (x - x.min()) / (x.max() - x.min() + eps)


def suavizar_1d(x, kernel_size=21):
    kernel_size = int(kernel_size)

    if kernel_size % 2 == 0:
        kernel_size += 1

    kernel = np.ones(kernel_size, dtype=np.float32) / kernel_size
    return np.convolve(x, kernel, mode="same")


def detectar_roi_radiografia(img_rgb, margen_x=35, margen_y=5):
    """
    Detecta el Ã¡rea no negra de la radiografÃ­a.
    Sirve para no buscar bordes en el fondo negro.
    """

    gray = imagen_a_gris_uint8(img_rgb)
    H, W = gray.shape

    valores = gray[gray > 0]

    if len(valores) == 0:
        return [0, 0, W - 1, H - 1]

    thr = max(5, np.percentile(valores, 3))
    mask = (gray > thr).astype(np.uint8)

    kernel = np.ones((21, 21), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask)

    if num_labels <= 1:
        return [0, 0, W - 1, H - 1]

    largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])

    x, y, w, h, area = stats[largest]

    if w < W * 0.10 or h < H * 0.20:
        return [0, 0, W - 1, H - 1]

    x0 = max(0, x - margen_x)
    y0 = max(0, y - margen_y)
    x1 = min(W - 1, x + w + margen_x)
    y1 = min(H - 1, y + h + margen_y)

    return [x0, y0, x1, y1]


def seleccionar_picos_1d(profile, xs_abs, n_picos=8, min_sep=18):
    """
    Selecciona picos separados en un perfil 1D.
    """

    if len(profile) == 0:
        return np.array([]), np.array([])

    order = np.argsort(profile)[::-1]

    picos_x = []
    picos_s = []

    for idx in order:
        x = float(xs_abs[idx])
        s = float(profile[idx])

        if all(abs(x - px) >= min_sep for px in picos_x):
            picos_x.append(x)
            picos_s.append(s)

        if len(picos_x) >= n_picos:
            break

    return np.array(picos_x, dtype=np.float32), np.array(picos_s, dtype=np.float32)
# ============================================================
# 12.4) Estimar eje curvo desde bordes laterales
# ============================================================

def estimar_eje_curvo_por_bordes(
    img_rgb,
    template_bbox=None,
    n_franjas=27,
    search_half_frac=0.23,
    n_picos=8,
    min_sep_frac=0.018,
    poly_degree=3,
    debug=False
):
    """
    Estima un eje curvo de columna a partir de bordes laterales.

    LÃ³gica:
    1. Detecta ROI de radiografÃ­a.
    2. Divide la imagen en franjas horizontales.
    3. En cada franja busca candidatos de borde izquierdo y derecho.
    4. Selecciona el par de bordes mÃ¡s razonable.
    5. Calcula centro = (borde_izquierdo + borde_derecho) / 2.
    6. Suaviza los centros con un polinomio.
    """

    gray = imagen_a_gris_uint8(img_rgb)
    H, W = gray.shape

    x_roi0, y_roi0, x_roi1, y_roi1 = detectar_roi_radiografia(img_rgb)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray_eq = clahe.apply(gray)

    grad_x = np.abs(cv2.Sobel(gray_eq, cv2.CV_32F, 1, 0, ksize=3))
    grad_y = np.abs(cv2.Sobel(gray_eq, cv2.CV_32F, 0, 1, ksize=3))

    score_img = (
        0.70 * normalizar_01(grad_x) +
        0.20 * normalizar_01(gray_eq) +
        0.10 * normalizar_01(grad_y)
    )

    if template_bbox is not None:
        x_prior_inicial = float(np.median(template_bbox["cx_rel"].to_numpy(dtype=float) * W))
        ancho_esperado = float(np.median(template_bbox["w_rel"].to_numpy(dtype=float) * W))
    else:
        x_prior_inicial = (x_roi0 + x_roi1) / 2
        ancho_esperado = W * 0.10

    ancho_esperado = np.clip(ancho_esperado * 1.15, W * 0.06, W * 0.18)

    min_width = max(W * 0.045, ancho_esperado * 0.55)
    max_width = min(W * 0.28, ancho_esperado * 2.60)

    search_half = int(W * search_half_frac)
    min_sep = int(W * min_sep_frac)

    y_edges = np.linspace(0, H, n_franjas + 1).astype(int)

    puntos_centro = []
    puntos_izq = []
    puntos_der = []
    debug_rows = []

    x_prior = x_prior_inicial

    for i in range(n_franjas):

        y0 = int(y_edges[i])
        y1 = int(y_edges[i + 1])
        y_mid = int((y0 + y1) / 2)

        x0 = max(x_roi0, int(x_prior - search_half))
        x1 = min(x_roi1, int(x_prior + search_half))

        if x1 <= x0 + 10:
            x0 = max(0, int(x_prior - search_half))
            x1 = min(W - 1, int(x_prior + search_half))

        xs_abs = np.arange(x0, x1 + 1)

        crop_score = score_img[y0:y1, x0:x1 + 1]

        if crop_score.size == 0:
            continue

        profile = crop_score.mean(axis=0)
        profile = suavizar_1d(profile, kernel_size=max(15, int(W * 0.025)))
        profile = normalizar_01(profile)

        left_mask = xs_abs < (x_prior - min_width * 0.20)
        right_mask = xs_abs > (x_prior + min_width * 0.20)

        xs_left = xs_abs[left_mask]
        prof_left = profile[left_mask]

        xs_right = xs_abs[right_mask]
        prof_right = profile[right_mask]

        left_peaks_x, left_peaks_s = seleccionar_picos_1d(
            prof_left,
            xs_left,
            n_picos=n_picos,
            min_sep=min_sep
        )

        right_peaks_x, right_peaks_s = seleccionar_picos_1d(
            prof_right,
            xs_right,
            n_picos=n_picos,
            min_sep=min_sep
        )

        mejor = None
        mejor_score = -np.inf

        for xl, sl in zip(left_peaks_x, left_peaks_s):
            for xr, sr in zip(right_peaks_x, right_peaks_s):

                if xr <= xl:
                    continue

                width = xr - xl

                if width < min_width or width > max_width:
                    continue

                centro = (xl + xr) / 2

                penal_width = ((width - ancho_esperado) / (ancho_esperado + 1e-6)) ** 2
                penal_prior = ((centro - x_prior) / W) ** 2

                score = sl + sr - 0.75 * penal_width - 1.50 * penal_prior

                if score > mejor_score:
                    mejor_score = score
                    mejor = (xl, xr, centro, width, sl, sr)

        if mejor is None:
            xl = x_prior - ancho_esperado / 2
            xr = x_prior + ancho_esperado / 2
            centro = x_prior
            width = ancho_esperado
            sl = np.nan
            sr = np.nan
        else:
            xl, xr, centro, width, sl, sr = mejor

        puntos_izq.append([y_mid, xl])
        puntos_der.append([y_mid, xr])
        puntos_centro.append([y_mid, centro])

        debug_rows.append({
            "franja": i,
            "y_mid": y_mid,
            "x_left": xl,
            "x_right": xr,
            "x_center": centro,
            "width": width,
            "x_prior": x_prior,
            "score_left": sl,
            "score_right": sr,
            "score_pair": mejor_score
        })

        # ActualizaciÃ³n suave del prior para permitir escoliosis sin saltos bruscos.
        x_prior = 0.65 * x_prior + 0.35 * centro

    puntos_centro = np.array(puntos_centro, dtype=np.float32)
    puntos_izq = np.array(puntos_izq, dtype=np.float32)
    puntos_der = np.array(puntos_der, dtype=np.float32)

    if len(puntos_centro) < 4:
        raise ValueError("No se pudieron estimar suficientes puntos para el eje curvo.")

    y_raw = puntos_centro[:, 0]
    x_raw = puntos_centro[:, 1]

    deg = min(poly_degree, len(y_raw) - 1)
    coef = np.polyfit(y_raw, x_raw, deg=deg)
    polinomio = np.poly1d(coef)

    y_smooth = np.linspace(0, H - 1, 300)
    x_smooth = np.clip(polinomio(y_smooth), 0, W - 1)

    puntos_smooth = np.column_stack([y_smooth, x_smooth])

    info = {
        "puntos_centro_raw": puntos_centro,
        "puntos_izq": puntos_izq,
        "puntos_der": puntos_der,
        "puntos_smooth": puntos_smooth,
        "polinomio": polinomio,
        "roi": [x_roi0, y_roi0, x_roi1, y_roi1],
        "score_img": score_img,
        "debug": pd.DataFrame(debug_rows)
    }

    return info
# ============================================================
# 12.5) Generar prompts automÃ¡ticos T1-L5
# versiÃ³n corregida con calibraciÃ³n vertical no lineal
# ============================================================



# ============================================================
# 12.4B) Estimar eje curvo por respuesta central de columna
# ============================================================

def estimar_eje_curvo_por_respuesta_central(
    img_rgb,
    template_bbox=None,
    n_franjas=34,
    search_half_frac=0.20,
    continuidad_px=42,
    poly_degree=5,
    smooth_kernel=5,
    debug=False,
):
    gray = imagen_a_gris_uint8(img_rgb)
    H, W = gray.shape
    x_roi0, y_roi0, x_roi1, y_roi1 = detectar_roi_radiografia(img_rgb)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray_eq = clahe.apply(gray)
    grad_x = np.abs(cv2.Sobel(gray_eq, cv2.CV_32F, 1, 0, ksize=3))
    grad_y = np.abs(cv2.Sobel(gray_eq, cv2.CV_32F, 0, 1, ksize=3))
    lap = np.abs(cv2.Laplacian(gray_eq, cv2.CV_32F, ksize=3))

    score_img = (
        0.35 * normalizar_01(gray_eq) +
        0.30 * normalizar_01(grad_x) +
        0.20 * normalizar_01(grad_y) +
        0.15 * normalizar_01(lap)
    )

    if template_bbox is not None:
        x_prior = float(np.median(template_bbox["cx_rel"].to_numpy(dtype=float) * W))
        y_min = float(template_bbox["cy_rel"].min() * H)
        y_max = float(template_bbox["cy_rel"].max() * H)
    else:
        x_prior = (x_roi0 + x_roi1) / 2
        y_min, y_max = y_roi0, y_roi1

    y0_busq = max(0, int(min(y_roi0, y_min - 0.06 * H)))
    y1_busq = min(H - 1, int(max(y_roi1, y_max + 0.06 * H)))
    y_edges = np.linspace(y0_busq, y1_busq, n_franjas + 1).astype(int)
    search_half = int(W * search_half_frac)

    puntos = []
    debug_rows = []

    for i in range(n_franjas):
        y0 = int(y_edges[i])
        y1 = int(y_edges[i + 1])
        y_mid = int((y0 + y1) / 2)

        x0 = max(x_roi0, int(x_prior - search_half))
        x1 = min(x_roi1, int(x_prior + search_half))
        if x1 <= x0 + 10:
            x0 = max(0, int(x_prior - search_half))
            x1 = min(W - 1, int(x_prior + search_half))

        xs_abs = np.arange(x0, x1 + 1)
        crop = score_img[y0:y1, x0:x1 + 1]
        if crop.size == 0:
            continue

        profile = crop.mean(axis=0)
        profile = suavizar_1d(profile, kernel_size=max(9, int(W * 0.015)))
        profile = normalizar_01(profile)

        penal_continuidad = ((xs_abs - x_prior) / max(continuidad_px, 1)) ** 2
        score = profile - 0.18 * penal_continuidad
        best_idx = int(np.argmax(score))
        x_best = float(xs_abs[best_idx])

        puntos.append([y_mid, x_best])
        debug_rows.append({
            "franja": i,
            "y_mid": y_mid,
            "x_center": x_best,
            "x_prior": x_prior,
            "score_best": float(score[best_idx]),
            "profile_best": float(profile[best_idx]),
            "x0": x0,
            "x1": x1,
        })

        x_prior = 0.50 * x_prior + 0.50 * x_best

    puntos = np.array(puntos, dtype=np.float32)
    if len(puntos) < 5:
        raise ValueError("No se pudieron estimar suficientes puntos para el eje central.")

    y_raw = puntos[:, 0]
    x_raw = puntos[:, 1]
    x_smooth_raw = suavizar_1d(x_raw, kernel_size=smooth_kernel) if smooth_kernel and smooth_kernel > 1 else x_raw.copy()

    deg = min(poly_degree, len(y_raw) - 1)
    coef = np.polyfit(y_raw, x_smooth_raw, deg=deg)
    polinomio = np.poly1d(coef)

    y_smooth = np.linspace(0, H - 1, 300)
    x_smooth = np.clip(polinomio(y_smooth), 0, W - 1)
    puntos_smooth = np.column_stack([y_smooth, x_smooth])

    return {
        "puntos_centro_raw": puntos,
        "puntos_izq": puntos.copy(),
        "puntos_der": puntos.copy(),
        "puntos_smooth": puntos_smooth,
        "polinomio": polinomio,
        "roi": [x_roi0, y_roi0, x_roi1, y_roi1],
        "score_img": score_img,
        "debug": pd.DataFrame(debug_rows),
        "metodo_eje": "central",
    }




def _filtrar_puntos_eje_robusto(puntos, max_salto_px=55, ventana_mediana=5):
    """Filtra puntos centrales con saltos laterales extremos y suaviza localmente."""
    puntos = np.asarray(puntos, dtype=np.float32)
    if len(puntos) < 5:
        return puntos

    y = puntos[:, 0].copy()
    x = puntos[:, 1].copy()

    # Reemplazar saltos locales grandes por interpolacion de vecinos confiables.
    keep = np.ones(len(x), dtype=bool)
    for i in range(1, len(x)):
        if abs(x[i] - x[i - 1]) > max_salto_px:
            keep[i] = False

    if keep.sum() >= 4:
        x = np.interp(y, y[keep], x[keep])

    # Mediana movil: preserva curvas suaves y reduce puntos atraidos por costillas/pelvis.
    k = int(ventana_mediana)
    if k > 1:
        if k % 2 == 0:
            k += 1
        pad = k // 2
        x_pad = np.pad(x, (pad, pad), mode="edge")
        x_med = np.array([np.median(x_pad[i:i + k]) for i in range(len(x))])
        x = 0.65 * x_med + 0.35 * x

    return np.column_stack([y, x]).astype(np.float32)


def estimar_eje_curvo_por_respuesta_central_robusta(
    img_rgb,
    template_bbox=None,
    n_franjas=34,
    search_half_frac=0.22,
    continuidad_px=50,
    max_salto_px=55,
    ventana_mediana=5,
    debug=False,
):
    """
    Variante robusta del eje central.

    Usa los puntos por respuesta central, pero evita ajustar un polinomio global.
    La curva final es interpolacion local de puntos filtrados, por lo que no extrapola arcos raros en extremos.
    """
    info = estimar_eje_curvo_por_respuesta_central(
        img_rgb=img_rgb,
        template_bbox=template_bbox,
        n_franjas=n_franjas,
        search_half_frac=search_half_frac,
        continuidad_px=continuidad_px,
        poly_degree=3,
        smooth_kernel=1,
        debug=debug,
    )

    puntos_raw = info["puntos_centro_raw"]
    puntos_filtrados = _filtrar_puntos_eje_robusto(
        puntos_raw,
        max_salto_px=max_salto_px,
        ventana_mediana=ventana_mediana,
    )

    H, W = img_rgb.shape[:2]
    y_raw = puntos_filtrados[:, 0]
    x_raw = puntos_filtrados[:, 1]

    y_smooth = np.linspace(0, H - 1, 300)

    # Evitar extrapolacion agresiva: fuera del rango observado, mantener el extremo mas cercano.
    x_smooth = np.interp(y_smooth, y_raw, x_raw, left=x_raw[0], right=x_raw[-1])
    x_smooth = suavizar_1d(x_smooth, kernel_size=11)
    x_smooth = np.clip(x_smooth, 0, W - 1)

    # Polinomio compatible con el resto del codigo, construido sobre la curva ya robusta.
    coef = np.polyfit(y_smooth, x_smooth, deg=5)
    polinomio = np.poly1d(coef)

    info["puntos_centro_raw"] = puntos_filtrados
    info["puntos_izq"] = puntos_filtrados.copy()
    info["puntos_der"] = puntos_filtrados.copy()
    info["puntos_smooth"] = np.column_stack([y_smooth, x_smooth])
    info["polinomio"] = polinomio
    info["metodo_eje"] = "central_robusto"
    info["debug_raw"] = info.get("debug", pd.DataFrame()).copy()
    return info



def estimar_eje_por_ruta_dinamica(
    img_rgb,
    template_bbox=None,
    n_franjas=64,
    search_half_frac=0.32,
    n_candidatos=45,
    penal_salto=0.018,
    penal_curvatura=0.010,
    penal_prior=0.002,
    smooth_kernel=7,
    x_prior_override=None,
    ruta_nombre="ruta_dinamica",
    debug=False,
):
    """
    Eje por ruta dinamica: busca una trayectoria global de alta respuesta visual.

    A diferencia del metodo central greedy, no decide cada franja de forma aislada.
    Optimiza una ruta completa penalizando saltos y curvatura brusca.
    """
    gray = imagen_a_gris_uint8(img_rgb)
    H, W = gray.shape
    x_roi0, y_roi0, x_roi1, y_roi1 = detectar_roi_radiografia(img_rgb)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray_eq = clahe.apply(gray)
    grad_x = np.abs(cv2.Sobel(gray_eq, cv2.CV_32F, 1, 0, ksize=3))
    grad_y = np.abs(cv2.Sobel(gray_eq, cv2.CV_32F, 0, 1, ksize=3))
    lap = np.abs(cv2.Laplacian(gray_eq, cv2.CV_32F, ksize=3))

    score_img = (
        0.35 * normalizar_01(gray_eq) +
        0.30 * normalizar_01(grad_x) +
        0.20 * normalizar_01(grad_y) +
        0.15 * normalizar_01(lap)
    )

    if template_bbox is not None:
        x_template_prior = float(np.median(template_bbox["cx_rel"].to_numpy(dtype=float) * W))
        y_min = float(template_bbox["cy_rel"].min() * H)
        y_max = float(template_bbox["cy_rel"].max() * H)
    else:
        x_template_prior = (x_roi0 + x_roi1) / 2
        y_min, y_max = y_roi0, y_roi1

    if x_prior_override is not None:
        x_template_prior = float(np.clip(x_prior_override, x_roi0, x_roi1))

    y0_busq = max(0, int(min(y_roi0, y_min - 0.08 * H)))
    y1_busq = min(H - 1, int(max(y_roi1, y_max + 0.08 * H)))
    y_edges = np.linspace(y0_busq, y1_busq, n_franjas + 1).astype(int)

    search_half = int(W * search_half_frac)
    x0_global = max(x_roi0, int(x_template_prior - search_half))
    x1_global = min(x_roi1, int(x_template_prior + search_half))
    if x1_global <= x0_global + 20:
        x0_global, x1_global = x_roi0, x_roi1

    candidatas = []
    debug_rows = []

    for i in range(n_franjas):
        y0 = int(y_edges[i])
        y1 = int(y_edges[i + 1])
        y_mid = int((y0 + y1) / 2)
        xs_abs = np.arange(x0_global, x1_global + 1)
        crop = score_img[y0:y1, x0_global:x1_global + 1]
        if crop.size == 0:
            continue

        profile = crop.mean(axis=0)
        profile = suavizar_1d(profile, kernel_size=max(9, int(W * 0.012)))
        profile = normalizar_01(profile)

        # Mantener candidatos diversos: top por score y muestreo uniforme.
        top_idx = np.argsort(profile)[::-1][:n_candidatos]
        uniform_idx = np.linspace(0, len(xs_abs) - 1, min(n_candidatos // 2, len(xs_abs))).astype(int)
        idxs = np.unique(np.r_[top_idx, uniform_idx])
        xs = xs_abs[idxs].astype(float)
        scores = profile[idxs].astype(float)

        order = np.argsort(xs)
        xs = xs[order]
        scores = scores[order]

        candidatas.append({"y": float(y_mid), "xs": xs, "scores": scores})
        debug_rows.append({"franja": i, "y_mid": y_mid, "n_candidatos": len(xs), "score_max": float(scores.max())})

    if len(candidatas) < 5:
        raise ValueError("No hay suficientes franjas candidatas para ruta dinamica.")

    # DP de segundo orden aproximado: estado = candidato actual, guarda mejor predecesor.
    dp = []
    back = []

    xs0 = candidatas[0]["xs"]
    sc0 = candidatas[0]["scores"]
    prior0 = -penal_prior * ((xs0 - x_template_prior) / W) ** 2
    dp.append(sc0 + prior0)
    back.append(np.full(len(xs0), -1, dtype=int))

    prev_prev_xs = None
    prev_xs = xs0

    for t in range(1, len(candidatas)):
        xs = candidatas[t]["xs"]
        sc = candidatas[t]["scores"]
        prev_score = dp[-1]
        mat_salto = ((xs[:, None] - prev_xs[None, :]) / W) ** 2
        score_mat = prev_score[None, :] - penal_salto * mat_salto

        if t >= 2 and prev_prev_xs is not None:
            # Curvatura aproximada usando el mejor predecesor de cada prev.
            prev_back = back[-1]
            prevprev_for_prev = np.array([
                prev_prev_xs[j] if j >= 0 else prev_xs[k]
                for k, j in enumerate(prev_back)
            ])
            curv = (xs[:, None] - 2 * prev_xs[None, :] + prevprev_for_prev[None, :])
            score_mat -= penal_curvatura * (curv / W) ** 2

        best_prev = np.argmax(score_mat, axis=1)
        best_score = score_mat[np.arange(len(xs)), best_prev] + sc
        best_score -= penal_prior * ((xs - x_template_prior) / W) ** 2

        dp.append(best_score)
        back.append(best_prev.astype(int))
        prev_prev_xs = prev_xs
        prev_xs = xs

    # Backtracking.
    idx_last = int(np.argmax(dp[-1]))
    ruta_idx = [idx_last]
    for t in range(len(candidatas) - 1, 0, -1):
        idx_last = int(back[t][idx_last])
        ruta_idx.append(idx_last)
    ruta_idx = ruta_idx[::-1]

    puntos = []
    for item, idx_cand in zip(candidatas, ruta_idx):
        puntos.append([item["y"], item["xs"][idx_cand]])
    puntos = np.array(puntos, dtype=np.float32)

    puntos_filtrados = _filtrar_puntos_eje_robusto(puntos, max_salto_px=65, ventana_mediana=smooth_kernel)
    y_raw = puntos_filtrados[:, 0]
    x_raw = puntos_filtrados[:, 1]

    y_smooth = np.linspace(0, H - 1, 300)
    x_smooth = np.interp(y_smooth, y_raw, x_raw, left=x_raw[0], right=x_raw[-1])
    x_smooth = suavizar_1d(x_smooth, kernel_size=13)
    x_smooth = np.clip(x_smooth, 0, W - 1)

    coef = np.polyfit(y_smooth, x_smooth, deg=5)
    polinomio = np.poly1d(coef)

    return {
        "puntos_centro_raw": puntos_filtrados,
        "puntos_izq": puntos_filtrados.copy(),
        "puntos_der": puntos_filtrados.copy(),
        "puntos_smooth": np.column_stack([y_smooth, x_smooth]),
        "polinomio": polinomio,
        "roi": [x_roi0, y_roi0, x_roi1, y_roi1],
        "score_img": score_img,
        "debug": pd.DataFrame(debug_rows),
        "x_prior_usado": float(x_template_prior),
        "metodo_eje": ruta_nombre,
    }

def estimar_eje_columna(img_rgb, template_bbox=None, metodo="bordes"):
    if metodo == "bordes":
        info = estimar_eje_curvo_por_bordes(img_rgb, template_bbox=template_bbox)
        info["metodo_eje"] = "bordes"
        return info
    if metodo == "central":
        return estimar_eje_curvo_por_respuesta_central(img_rgb, template_bbox=template_bbox)
    if metodo == "central_robusto":
        return estimar_eje_curvo_por_respuesta_central_robusta(img_rgb, template_bbox=template_bbox)
    if metodo == "ruta_dinamica":
        return estimar_eje_por_ruta_dinamica(img_rgb, template_bbox=template_bbox)
    raise ValueError(f"metodo_eje no reconocido: {metodo}")




def limitar_info_eje_a_template(info_eje, img_shape, template_bbox=None, margen_rel=0.05):
    """Recorta puntos/curva del eje al rango vertical esperado T1-L5 para evitar craneo/cadera."""
    if template_bbox is None:
        return info_eje

    H = img_shape[0]
    y_min = float(template_bbox["cy_rel"].min() * H)
    y_max = float(template_bbox["cy_rel"].max() * H)
    margen = float(margen_rel * H)
    y0 = max(0, y_min - margen)
    y1 = min(H - 1, y_max + margen)

    info = dict(info_eje)
    for key in ["puntos_centro_raw", "puntos_izq", "puntos_der", "puntos_smooth"]:
        if key in info and info[key] is not None:
            pts = np.asarray(info[key])
            if pts.ndim == 2 and pts.shape[1] >= 2:
                keep = (pts[:, 0] >= y0) & (pts[:, 0] <= y1)
                if keep.sum() >= 2:
                    info[key] = pts[keep]

    # Reconstruir interpolador solo dentro del rango anatomico si hay puntos suficientes.
    pts_curve = np.asarray(info.get("puntos_smooth", []))
    if pts_curve.ndim == 2 and len(pts_curve) >= 4:
        y = pts_curve[:, 0]
        x = pts_curve[:, 1]
        def curva_limitada(yq, y=y, x=x):
            return np.interp(yq, y, x, left=x[0], right=x[-1])
        info["polinomio"] = curva_limitada

    info["rango_y_anatomico"] = [float(y0), float(y1)]
    return info

def generar_prompts_auto_por_bordes(
    img_rgb,
    template_bbox,
    escala_w=1.90,
    escala_h=1.35,
    peso_eje=0.90,
    escala_pos_y=0.91,
    offset_y=-16,
    offset_x=-18,
    mid_lift=22,
    min_w_frac=0.10,
    min_h_frac=0.035,
    max_w_frac=0.34,
    max_h_frac=0.13,
    metodo_eje="bordes",
    limitar_rango_y_anatomico=True,
    debug=False
):
    """
    Genera cajas automÃ¡ticas para las 17 vÃ©rtebras.

    Correcciones incluidas:
    - El eje horizontal se estima desde bordes laterales.
    - Las claves son 'T1', 'T2', ..., 'L5'.
    - Se aplica una compresiÃ³n vertical de la plantilla.
    - Se aplica una correcciÃ³n vertical no lineal para subir mÃ¡s la zona media.
    - Se aplica un pequeÃ±o desplazamiento horizontal hacia la izquierda.
    """

    H, W = img_rgb.shape[:2]

    info_eje = estimar_eje_columna(
        img_rgb,
        template_bbox=template_bbox,
        metodo=metodo_eje
    )

    if limitar_rango_y_anatomico:
        info_eje = limitar_info_eje_a_template(
            info_eje,
            img_rgb.shape,
            template_bbox=template_bbox,
            margen_rel=0.05,
        )

    curva = info_eje["polinomio"]

    prompts_auto = {}
    filas_debug = []

    template_ordenado = template_bbox.sort_values("id_real").reset_index(drop=True)

    y_anchor = float(template_ordenado.iloc[0]["cy_rel"] * H)
    y_min_template = float(template_ordenado["cy_rel"].min() * H)
    y_max_template = float(template_ordenado["cy_rel"].max() * H)

    for _, row in template_ordenado.iterrows():

        vertebra = row["vertebra"]

        if vertebra not in CLASES_OBJETIVO:
            continue

        id_real = int(row["id_real"])

        # PosiciÃ³n vertical original de la plantilla.
        cy_original = float(row["cy_rel"] * H)

        # PosiciÃ³n relativa entre T1 y L5.
        t = (cy_original - y_min_template) / (y_max_template - y_min_template + 1e-6)
        t = float(np.clip(t, 0, 1))

        # CorrecciÃ³n no lineal:
        # levanta mÃ¡s la zona media y casi no modifica los extremos.
        correccion_media = mid_lift * np.sin(np.pi * t)

        # CorrecciÃ³n vertical final.
        cy = y_anchor + escala_pos_y * (cy_original - y_anchor) + offset_y - correccion_media
        cy = float(np.clip(cy, 0, H - 1))

        # Centro horizontal desde el eje curvo, evaluado en la altura corregida.
        cx_eje = float(curva(cy))

        # Centro horizontal promedio desde plantilla.
        cx_template = float(row["cx_rel"] * W)

        # Mezcla eje automÃ¡tico + plantilla, con correcciÃ³n horizontal.
        cx = peso_eje * cx_eje + (1 - peso_eje) * cx_template + offset_x
        cx = float(np.clip(cx, 0, W - 1))

        # TamaÃ±o de caja.
        bw = float(row["w_rel"] * W * escala_w)
        bh = float(row["h_rel"] * H * escala_h)

        bw = float(np.clip(bw, W * min_w_frac, W * max_w_frac))
        bh = float(np.clip(bh, H * min_h_frac, H * max_h_frac))

        x0 = int(round(cx - bw / 2))
        x1 = int(round(cx + bw / 2))
        y0 = int(round(cy - bh / 2))
        y1 = int(round(cy + bh / 2))

        x0 = max(0, x0)
        y0 = max(0, y0)
        x1 = min(W - 1, x1)
        y1 = min(H - 1, y1)

        prompts_auto[vertebra] = {
            "vertebra": vertebra,
            "id_real": id_real,
            "bbox_xyxy": [x0, y0, x1, y1],
            "prompt_origen": "bbox_auto_bordes_eje_curvo_y_calibrado"
        }

        filas_debug.append({
            "vertebra": vertebra,
            "id_real": id_real,
            "cy_original": cy_original,
            "cy_final": cy,
            "t_vertical": t,
            "correccion_media": correccion_media,
            "cx_eje": cx_eje,
            "cx_template": cx_template,
            "cx_final": cx,
            "escala_pos_y": escala_pos_y,
            "offset_y": offset_y,
            "offset_x": offset_x,
            "mid_lift": mid_lift,
            "bbox_xyxy": [x0, y0, x1, y1]
        })

    df_debug = pd.DataFrame(filas_debug)

    if debug:
        return prompts_auto, info_eje, df_debug

    return prompts_auto, info_eje

In [ ]:
# ============================================================
# 12.6) VisualizaciÃ³n de eje, bordes y cajas automÃ¡ticas
# ============================================================

def visualizar_cajas_auto_bordes(
    img_rgb,
    prompts_auto,
    info_eje=None,
    mascara_gt=None,
    titulo="Cajas automÃ¡ticas por bordes laterales"
):
    plt.figure(figsize=(8, 10))
    plt.imshow(img_rgb)

    ax = plt.gca()

    if mascara_gt is not None:
        gt_overlay = np.zeros_like(img_rgb)
        gt_overlay[..., 0] = (mascara_gt > 0).astype(np.uint8) * 255
        plt.imshow(gt_overlay, alpha=0.20)

    if info_eje is not None:
        x0, y0, x1, y1 = info_eje["roi"]

        rect_roi = plt.Rectangle(
            (x0, y0),
            x1 - x0,
            y1 - y0,
            fill=False,
            edgecolor="white",
            linewidth=1.2,
            linestyle="--"
        )
        ax.add_patch(rect_roi)

        metodo_eje = info_eje.get("metodo_eje", "bordes")
        puntos_centro = info_eje["puntos_centro_raw"]
        puntos_smooth = info_eje["puntos_smooth"]

        if metodo_eje == "bordes":
            puntos_izq = info_eje["puntos_izq"]
            puntos_der = info_eje["puntos_der"]
            plt.scatter(puntos_izq[:, 1], puntos_izq[:, 0], s=10, c="orange", label="Borde izquierdo")
            plt.scatter(puntos_der[:, 1], puntos_der[:, 0], s=10, c="yellow", label="Borde derecho")
            label_centro = "Centro por bordes"
        else:
            label_centro = "Centro por respuesta"

        plt.scatter(puntos_centro[:, 1], puntos_centro[:, 0], s=14, c="cyan", label=label_centro)
        plt.plot(puntos_smooth[:, 1], puntos_smooth[:, 0], c="cyan", linewidth=2, label=f"Eje {metodo_eje} suavizado")

    for vertebra, info in prompts_auto.items():

        x0, y0, x1, y1 = info["bbox_xyxy"]

        rect = plt.Rectangle(
            (x0, y0),
            x1 - x0,
            y1 - y0,
            fill=False,
            edgecolor="lime",
            linewidth=1.2
        )
        ax.add_patch(rect)

        ax.text(
            x0,
            max(0, y0 - 3),
            vertebra,
            fontsize=8,
            color="white",
            bbox=dict(facecolor="black", alpha=0.45, pad=1)
        )

    plt.title(titulo)
    plt.axis("off")
    plt.legend(loc="lower right")
    plt.show()
# ============================================================
# 12.7) Cobertura de cajas automÃ¡ticas
# ============================================================

def evaluar_cobertura_cajas_por_clase(prompts_auto, mascara_gt_multiclase):
    """
    Evalua la calidad espacial de cada caja automatica contra la mascara real.

    Metricas principales:
    - bbox_recall: cuanto de la vertebra real queda dentro de la caja.
    - bbox_precision: que proporcion del area de la caja corresponde a la vertebra.
    - bbox_iou: interseccion / union entre caja y mascara real.
    - bbox_area_ratio: tamano de caja relativo al area real; valores muy altos suelen indicar cajas demasiado grandes.
    """

    filas = []

    for vertebra in CLASES_OBJETIVO:

        if vertebra not in prompts_auto:
            continue

        info = prompts_auto[vertebra]
        x0, y0, x1, y1 = info["bbox_xyxy"]

        gt_i = construir_mask_binaria_vertebra(
            mascara_gt_multiclase,
            vertebra
        ).astype(bool)

        total_gt = int(gt_i.sum())

        cover = np.zeros_like(gt_i, dtype=bool)
        cover[y0:y1 + 1, x0:x1 + 1] = True

        bbox_area = int(cover.sum())
        inter = int(np.logical_and(gt_i, cover).sum())
        union = int(np.logical_or(gt_i, cover).sum())

        bbox_recall = inter / total_gt if total_gt > 0 else np.nan
        bbox_precision = inter / bbox_area if bbox_area > 0 else np.nan
        bbox_iou = inter / union if union > 0 else np.nan
        bbox_area_ratio = bbox_area / total_gt if total_gt > 0 else np.nan

        filas.append({
            "vertebra": vertebra,
            "id_real": VERTEBRA_TO_ID[vertebra],
            "bbox_auto": [x0, y0, x1, y1],
            "gt_area": total_gt,
            "bbox_area": bbox_area,
            "bbox_intersection": inter,
            "bbox_union": union,
            "bbox_recall": bbox_recall,
            "bbox_precision": bbox_precision,
            "bbox_iou": bbox_iou,
            "bbox_area_ratio": bbox_area_ratio,
        })

    return pd.DataFrame(filas).sort_values("id_real").reset_index(drop=True)

def diagnosticar_desplazamiento_cajas(prompts_auto, mascara_gt_multiclase):
    """
    DiagnÃ³stico: compara centro de caja vs centro de la mÃ¡scara real.
    Solo evaluaciÃ³n, no generaciÃ³n de cajas.
    """

    filas = []

    for vertebra in CLASES_OBJETIVO:

        if vertebra not in prompts_auto:
            continue

        x0, y0, x1, y1 = prompts_auto[vertebra]["bbox_xyxy"]

        cx_box = (x0 + x1) / 2
        cy_box = (y0 + y1) / 2

        gt_i = construir_mask_binaria_vertebra(
            mascara_gt_multiclase,
            vertebra
        ).astype(bool)

        if gt_i.sum() == 0:
            continue

        ys, xs = np.where(gt_i)

        cx_gt = xs.mean()
        cy_gt = ys.mean()

        filas.append({
            "vertebra": vertebra,
            "cx_box": cx_box,
            "cy_box": cy_box,
            "cx_gt": cx_gt,
            "cy_gt": cy_gt,
            "dx_box_gt": cx_box - cx_gt,
            "dy_box_gt": cy_box - cy_gt
        })

    return pd.DataFrame(filas)


# ============================================================
# 12.8) Refinamiento local de centros vertebrales
# ============================================================

def construir_score_local_vertebra(img_rgb):
    """Score visual para buscar centros oseos locales sin usar mascara GT."""
    gray = imagen_a_gris_uint8(img_rgb)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray_eq = clahe.apply(gray)
    grad_x = np.abs(cv2.Sobel(gray_eq, cv2.CV_32F, 1, 0, ksize=3))
    grad_y = np.abs(cv2.Sobel(gray_eq, cv2.CV_32F, 0, 1, ksize=3))
    lap = np.abs(cv2.Laplacian(gray_eq, cv2.CV_32F, ksize=3))
    score = (
        0.40 * normalizar_01(gray_eq) +
        0.25 * normalizar_01(grad_x) +
        0.20 * normalizar_01(grad_y) +
        0.15 * normalizar_01(lap)
    )
    return normalizar_01(score)


def refinar_prompts_por_respuesta_local(
    img_rgb,
    prompts_auto,
    max_dx=35,
    max_dy=24,
    ventana_factor=1.45,
    top_percentile=82,
    penal_distancia=0.55,
    conservar_tamano=True,
    debug=False,
):
    """
    Refina el centro de cada caja buscando respuesta local de vertebra cerca de la caja inicial.

    No usa mascara GT. La mascara se sigue usando solo para evaluar despues.
    """
    H, W = img_rgb.shape[:2]
    score_img = construir_score_local_vertebra(img_rgb)

    refinados = {}
    filas = []

    for vertebra, info in prompts_auto.items():
        x0, y0, x1, y1 = [int(v) for v in info["bbox_xyxy"]]
        bw = max(x1 - x0 + 1, 1)
        bh = max(y1 - y0 + 1, 1)
        cx0 = (x0 + x1) / 2
        cy0 = (y0 + y1) / 2

        wx = int(round(bw * ventana_factor / 2))
        wy = int(round(bh * ventana_factor / 2))
        vx0 = max(0, int(round(cx0 - wx)))
        vx1 = min(W - 1, int(round(cx0 + wx)))
        vy0 = max(0, int(round(cy0 - wy)))
        vy1 = min(H - 1, int(round(cy0 + wy)))

        crop = score_img[vy0:vy1 + 1, vx0:vx1 + 1].copy()
        if crop.size == 0:
            refinados[vertebra] = dict(info)
            continue

        yy, xx = np.mgrid[vy0:vy1 + 1, vx0:vx1 + 1]
        dx = (xx - cx0) / max(max_dx, 1)
        dy = (yy - cy0) / max(max_dy, 1)
        dist2 = dx ** 2 + dy ** 2

        # Limitar busqueda a una elipse alrededor de la caja inicial.
        mask_busqueda = (np.abs(xx - cx0) <= max_dx) & (np.abs(yy - cy0) <= max_dy)
        score = crop - penal_distancia * dist2
        score[~mask_busqueda] = -np.inf

        valid = np.isfinite(score)
        if not valid.any():
            refinados[vertebra] = dict(info)
            continue

        thr = np.percentile(score[valid], top_percentile)
        pesos = np.clip(score - thr, 0, None)
        if pesos.sum() <= 1e-8:
            best = np.unravel_index(np.nanargmax(score), score.shape)
            cy_ref = float(vy0 + best[0])
            cx_ref = float(vx0 + best[1])
        else:
            cx_ref = float((xx * pesos).sum() / pesos.sum())
            cy_ref = float((yy * pesos).sum() / pesos.sum())

        cx_ref = float(np.clip(cx_ref, cx0 - max_dx, cx0 + max_dx))
        cy_ref = float(np.clip(cy_ref, cy0 - max_dy, cy0 + max_dy))

        if conservar_tamano:
            bw_new, bh_new = bw, bh
        else:
            bw_new, bh_new = bw * 0.95, bh * 0.95

        nx0 = int(round(cx_ref - bw_new / 2))
        nx1 = int(round(cx_ref + bw_new / 2))
        ny0 = int(round(cy_ref - bh_new / 2))
        ny1 = int(round(cy_ref + bh_new / 2))

        nx0 = max(0, nx0); ny0 = max(0, ny0)
        nx1 = min(W - 1, nx1); ny1 = min(H - 1, ny1)

        nuevo = dict(info)
        nuevo["bbox_xyxy"] = [nx0, ny0, nx1, ny1]
        nuevo["bbox_xyxy_base"] = [x0, y0, x1, y1]
        nuevo["prompt_origen"] = str(info.get("prompt_origen", "bbox_auto")) + "_refinado_local"
        refinados[vertebra] = nuevo

        filas.append({
            "vertebra": vertebra,
            "cx_base": cx0,
            "cy_base": cy0,
            "cx_refinado": cx_ref,
            "cy_refinado": cy_ref,
            "dx_refinado": cx_ref - cx0,
            "dy_refinado": cy_ref - cy0,
            "bbox_base": [x0, y0, x1, y1],
            "bbox_refinada": [nx0, ny0, nx1, ny1],
        })

    df_debug = pd.DataFrame(filas)
    if debug:
        return refinados, df_debug, score_img
    return refinados


def comparar_refinamiento_local_muestra(
    split="val",
    patient_id="N_12",
    metodo_eje="central_robusto",
    params=None,
):
    if params is None:
        params = {
            "escala_w": 1.40,
            "escala_h": 1.20,
            "peso_eje": 0.90,
            "escala_pos_y": 0.91,
            "offset_y": -16,
            "offset_x": -10,
            "mid_lift": 15,
            "metodo_eje": metodo_eje,
        }

    path_img, path_mask = resolver_paths_muestra(split, patient_id)
    img = cargar_imagen(path_img)
    mask = cargar_mascara(path_mask)

    prompts_base, info_eje = generar_prompts_auto_por_bordes(
        img_rgb=img,
        template_bbox=template_bbox_auto,
        debug=False,
        **params,
    )
    prompts_ref, df_ref, score_img = refinar_prompts_por_respuesta_local(
        img,
        prompts_base,
        debug=True,
    )

    df_base = evaluar_cobertura_cajas_por_clase(prompts_base, mask)
    df_refinada = evaluar_cobertura_cajas_por_clase(prompts_ref, mask)

    resumen = pd.DataFrame([
        {
            "version": "base",
            "bbox_iou": df_base["bbox_iou"].mean(),
            "bbox_recall": df_base["bbox_recall"].mean(),
            "bbox_precision": df_base["bbox_precision"].mean(),
        },
        {
            "version": "refinada_local",
            "bbox_iou": df_refinada["bbox_iou"].mean(),
            "bbox_recall": df_refinada["bbox_recall"].mean(),
            "bbox_precision": df_refinada["bbox_precision"].mean(),
        }
    ])

    visualizar_cajas_auto_bordes(
        img_rgb=img,
        prompts_auto=prompts_base,
        info_eje=info_eje,
        mascara_gt=mask,
        titulo=f"{patient_id} - cajas base"
    )
    visualizar_cajas_auto_bordes(
        img_rgb=img,
        prompts_auto=prompts_ref,
        info_eje=info_eje,
        mascara_gt=mask,
        titulo=f"{patient_id} - cajas refinadas localmente"
    )

    return resumen, df_base, df_refinada, df_ref, prompts_base, prompts_ref


## 4. Prueba visual en un paciente
Usa esta seccion para ajustar parametros de cajas automaticas antes de correr evaluaciones grandes.

In [ ]:
# ============================================================
# 12.8) Prueba visual y cobertura en un paciente
# con calibraciÃ³n vertical no lineal
# ============================================================

split_auto = "val"
patient_id_preferido = "N_12"

pacientes_disponibles_auto = sorted(list(PROMPTS_DICC[split_auto].keys()))
if patient_id_preferido in pacientes_disponibles_auto:
    patient_id_auto = patient_id_preferido
else:
    patient_id_auto = pacientes_disponibles_auto[0]
    print(
        f"{patient_id_preferido} no esta en {split_auto}; "
        f"se usara {patient_id_auto}."
    )

print(f"Paciente visual: {patient_id_auto} ({split_auto})")
path_img_auto, path_mask_auto = resolver_paths_muestra(split_auto, patient_id_auto)

imagen_auto = cargar_imagen(path_img_auto)
mascara_auto = cargar_mascara(path_mask_auto)

prompts_auto_sample, info_eje_auto, df_debug_prompts_auto = generar_prompts_auto_por_bordes(
    img_rgb=imagen_auto,
    template_bbox=template_bbox_auto,
    escala_w=1.90,
    escala_h=1.35,
    peso_eje=0.90,
    escala_pos_y=0.91,
    offset_y=-16,
    offset_x=-18,
    mid_lift=22,
    debug=True
)

print("Claves generadas:")
print(list(prompts_auto_sample.keys()))

faltantes = [v for v in CLASES_OBJETIVO if v not in prompts_auto_sample]
print("Faltantes:", faltantes)

display(df_debug_prompts_auto)

visualizar_cajas_auto_bordes(
    img_rgb=imagen_auto,
    prompts_auto=prompts_auto_sample,
    info_eje=info_eje_auto,
    mascara_gt=mascara_auto,
    titulo=f"{patient_id_auto} - cajas automÃ¡ticas corregidas con mid_lift"
)

df_cobertura_auto = evaluar_cobertura_cajas_por_clase(
    prompts_auto_sample,
    mascara_auto
)

display(df_cobertura_auto)

print("Cobertura promedio de cajas:")
print(df_cobertura_auto["bbox_recall"].mean())

df_diag_desplazamiento = diagnosticar_desplazamiento_cajas(
    prompts_auto_sample,
    mascara_auto
)

display(df_diag_desplazamiento)

print("Desplazamiento horizontal promedio:")
print(df_diag_desplazamiento["dx_box_gt"].mean())

print("Desplazamiento vertical promedio:")
print(df_diag_desplazamiento["dy_box_gt"].mean())

In [ ]:
# ============================================================
# 12.8B) ComparaciÃ³n rÃ¡pida de parÃ¡metros mid_lift
# ============================================================

resultados_mid_lift = []

for mid_lift_val in [0, 15, 22, 28]:

    prompts_tmp, info_eje_tmp, df_debug_tmp = generar_prompts_auto_por_bordes(
        img_rgb=imagen_auto,
        template_bbox=template_bbox_auto,
        escala_w=1.90,
        escala_h=1.35,
        peso_eje=0.90,
        escala_pos_y=0.91,
        offset_y=-16,
        offset_x=-18,
        mid_lift=mid_lift_val,
        debug=True
    )

    df_cobertura_tmp = evaluar_cobertura_cajas_por_clase(
        prompts_tmp,
        mascara_auto
    )

    df_diag_tmp = diagnosticar_desplazamiento_cajas(
        prompts_tmp,
        mascara_auto
    )

    resultados_mid_lift.append({
        "mid_lift": mid_lift_val,
        "bbox_recall_promedio": df_cobertura_tmp["bbox_recall"].mean(),
        "bbox_recall_min": df_cobertura_tmp["bbox_recall"].min(),
        "dx_promedio": df_diag_tmp["dx_box_gt"].mean(),
        "dy_promedio": df_diag_tmp["dy_box_gt"].mean(),
        "dy_abs_promedio": df_diag_tmp["dy_box_gt"].abs().mean()
    })

df_comparacion_mid_lift = pd.DataFrame(resultados_mid_lift)

display(df_comparacion_mid_lift)

In [ ]:
# ============================================================
# 12.8C) VisualizaciÃ³n del mejor parÃ¡metro seleccionado
# ============================================================

mejor_mid_lift = 22

prompts_auto_sample, info_eje_auto, df_debug_prompts_auto = generar_prompts_auto_por_bordes(
    img_rgb=imagen_auto,
    template_bbox=template_bbox_auto,
    escala_w=1.90,
    escala_h=1.35,
    peso_eje=0.90,
    escala_pos_y=0.91,
    offset_y=-16,
    offset_x=-18,
    mid_lift=mejor_mid_lift,
    debug=True
)

visualizar_cajas_auto_bordes(
    img_rgb=imagen_auto,
    prompts_auto=prompts_auto_sample,
    info_eje=info_eje_auto,
    mascara_gt=mascara_auto,
    titulo=f"{patient_id_auto} - mejor mid_lift={mejor_mid_lift}"
)

df_cobertura_auto = evaluar_cobertura_cajas_por_clase(
    prompts_auto_sample,
    mascara_auto
)

df_diag_desplazamiento = diagnosticar_desplazamiento_cajas(
    prompts_auto_sample,
    mascara_auto
)

display(df_cobertura_auto)
display(df_diag_desplazamiento)

print("Cobertura promedio de cajas:")
print(df_cobertura_auto["bbox_recall"].mean())

print("Desplazamiento horizontal promedio:")
print(df_diag_desplazamiento["dx_box_gt"].mean())

print("Desplazamiento vertical promedio:")
print(df_diag_desplazamiento["dy_box_gt"].mean())

print("Desplazamiento vertical absoluto promedio:")
print(df_diag_desplazamiento["dy_box_gt"].abs().mean())

## 5. Rendimiento de MedSAM con cajas automaticas
Primero prueba una imagen; despues corre un subconjunto de validacion.

In [ ]:
# ============================================================
# Evaluacion de una imagen con cajas automaticas + MedSAM
# ============================================================

resultado_auto_sample = reconstruir_mascara_semantica_medsam_con_score(
    imagen=imagen_auto,
    mascara_gt_multiclase=mascara_auto,
    prompts_sample=prompts_auto_sample,
    predictor=predictor,
    frac_x=0.04,
    frac_y=0.06,
    return_quality=True
)

mask_sem_pred_auto, mask_sem_gt_auto, detalles_auto, score_map_auto, quality_auto = resultado_auto_sample

dice_auto = dice_multiclase_promedio(
    mask_sem_pred_auto,
    mask_sem_gt_auto,
    clases_ids=list(range(1, N_CLASES + 1))
)

iou_auto = iou_multiclase_promedio(
    mask_sem_pred_auto,
    mask_sem_gt_auto,
    clases_ids=list(range(1, N_CLASES + 1))
)

df_detalles_auto = pd.DataFrame(detalles_auto).sort_values("id_real").reset_index(drop=True)

display(df_detalles_auto)
print(f"Dice macro {patient_id_auto}: {dice_auto:.4f}")
print(f"IoU  macro {patient_id_auto}: {iou_auto:.4f}")
print("Calidad prediccion:", quality_auto)


In [ ]:
# ============================================================
# 12.9) Visualizar predicciÃ³n semÃ¡ntica del experimento 12
# ============================================================

def visualizar_prediccion_semantica_auto(
    img_rgb,
    mask_gt,
    mask_pred,
    prompts_auto=None,
    info_curva=None,
    titulo="PredicciÃ³n semÃ¡ntica con cajas automÃ¡ticas"
):
    plt.figure(figsize=(8, 10))
    plt.imshow(img_rgb)

    gt_overlay = np.zeros_like(img_rgb)
    pred_overlay = np.zeros_like(img_rgb)

    # GT en rojo
    gt_overlay[..., 0] = (mask_gt > 0).astype(np.uint8) * 255

    # PredicciÃ³n en verde
    pred_overlay[..., 1] = (mask_pred > 0).astype(np.uint8) * 255

    plt.imshow(gt_overlay, alpha=0.25)
    plt.imshow(pred_overlay, alpha=0.25)

    ax = plt.gca()

    if info_curva is not None:
        puntos_smooth = info_curva["puntos_smooth"]
        plt.plot(
            puntos_smooth[:, 1],
            puntos_smooth[:, 0],
            c="cyan",
            linewidth=2
        )

    if prompts_auto is not None:
        for _, info in prompts_auto.items():
            x0, y0, x1, y1 = info["bbox_xyxy"]
            vertebra = info["vertebra"]

            rect = plt.Rectangle(
                (x0, y0),
                x1 - x0,
                y1 - y0,
                fill=False,
                edgecolor="yellow",
                linewidth=1.0
            )
            ax.add_patch(rect)

            ax.text(
                x0,
                max(0, y0 - 3),
                vertebra,
                fontsize=7,
                color="white",
                bbox=dict(facecolor="black", alpha=0.45, pad=1)
            )

    plt.title(titulo)
    plt.axis("off")
    plt.show()


visualizar_prediccion_semantica_auto(
    img_rgb=imagen_auto,
    mask_gt=mask_sem_gt_auto,
    mask_pred=mask_sem_pred_auto,
    prompts_auto=prompts_auto_sample,
    info_curva=info_eje_auto,
    titulo=f"{patient_id_auto} | Dice={dice_auto:.3f} | IoU={iou_auto:.3f}"
)


## 6. Comparativo cualitativo de area coloreada
Esta seccion compara visualmente la mascara real contra la mascara predicha. La imagen de diferencias usa colores fijos: verde = acierto, rojo = prediccion extra, azul = zona real que falto.

In [ ]:
# ============================================================
# Comparativo cualitativo: GT vs prediccion
# ============================================================

from matplotlib.colors import ListedColormap


def mask_rgb_por_clase(mask_sem, alpha_mask=None):
    """Convierte una mascara semantica 0..17 en RGB para visualizacion."""
    cmap = plt.get_cmap("tab20", N_CLASES + 1)
    rgb = (cmap(mask_sem.astype(int))[..., :3] * 255).astype(np.uint8)
    rgb[mask_sem == 0] = 0

    if alpha_mask is not None:
        rgb[~alpha_mask] = 0

    return rgb


def crear_mapa_diferencias(mask_gt, mask_pred):
    """
    Devuelve imagen RGB con:
    - verde: GT y pred coinciden en vertebra no fondo
    - rojo: predice vertebra donde no coincide con GT
    - azul: GT tiene vertebra y la prediccion fallo
    - amarillo: ambos son vertebra pero la clase es distinta
    """
    gt_fg = mask_gt > 0
    pred_fg = mask_pred > 0
    misma_clase = (mask_gt == mask_pred) & gt_fg & pred_fg
    clase_distinta = gt_fg & pred_fg & (mask_gt != mask_pred)
    falso_positivo = pred_fg & ~gt_fg
    falso_negativo = gt_fg & ~pred_fg

    diff = np.zeros((*mask_gt.shape, 3), dtype=np.uint8)
    diff[misma_clase] = [0, 210, 80]
    diff[falso_positivo] = [230, 40, 40]
    diff[falso_negativo] = [40, 120, 255]
    diff[clase_distinta] = [255, 210, 0]

    resumen = {
        "pixeles_acierto": int(misma_clase.sum()),
        "pixeles_pred_extra": int(falso_positivo.sum()),
        "pixeles_gt_faltante": int(falso_negativo.sum()),
        "pixeles_clase_distinta": int(clase_distinta.sum()),
        "pixeles_gt_total": int(gt_fg.sum()),
        "pixeles_pred_total": int(pred_fg.sum()),
    }
    resumen["recall_area_fg"] = resumen["pixeles_acierto"] / max(resumen["pixeles_gt_total"], 1)
    resumen["precision_area_fg"] = resumen["pixeles_acierto"] / max(resumen["pixeles_pred_total"], 1)
    return diff, resumen


def visualizar_comparativo_area_coloreada(
    img_rgb,
    mask_gt,
    mask_pred,
    prompts_auto=None,
    detalles=None,
    titulo="Comparativo cualitativo"
):
    diff_rgb, resumen_area = crear_mapa_diferencias(mask_gt, mask_pred)

    gt_rgb = mask_rgb_por_clase(mask_gt)
    pred_rgb = mask_rgb_por_clase(mask_pred)

    fig, ax = plt.subplots(2, 3, figsize=(18, 12))
    ax = ax.ravel()

    ax[0].imshow(img_rgb)
    if prompts_auto is not None:
        for vertebra, info in prompts_auto.items():
            x0, y0, x1, y1 = info["bbox_xyxy"]
            rect = plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor="yellow", linewidth=0.9)
            ax[0].add_patch(rect)
            ax[0].text(x0, max(0, y0 - 3), vertebra, fontsize=7, color="white", bbox=dict(facecolor="black", alpha=0.45, pad=1))
    ax[0].set_title("Imagen + cajas automaticas")
    ax[0].axis("off")

    ax[1].imshow(img_rgb)
    ax[1].imshow(gt_rgb, alpha=0.45)
    ax[1].set_title("GT coloreado")
    ax[1].axis("off")

    ax[2].imshow(img_rgb)
    ax[2].imshow(pred_rgb, alpha=0.45)
    ax[2].set_title("Prediccion coloreada")
    ax[2].axis("off")

    ax[3].imshow(img_rgb)
    ax[3].imshow(diff_rgb, alpha=0.70)
    ax[3].set_title("Diferencias: verde acierto, rojo extra, azul faltante, amarillo clase distinta")
    ax[3].axis("off")

    ax[4].imshow(mask_gt, cmap="nipy_spectral", vmin=0, vmax=N_CLASES)
    ax[4].set_title("Mascara GT semantica")
    ax[4].axis("off")

    ax[5].imshow(mask_pred, cmap="nipy_spectral", vmin=0, vmax=N_CLASES)
    ax[5].set_title("Mascara predicha semantica")
    ax[5].axis("off")

    fig.suptitle(titulo, fontsize=14)
    plt.tight_layout()
    plt.show()

    display(pd.DataFrame([resumen_area]))

    if detalles is not None and len(detalles) > 0:
        df_det = pd.DataFrame(detalles).sort_values("id_real")
        cols = [c for c in ["vertebra", "dice", "iou", "score_medsam", "pix_gt", "pix_pred", "bbox_expandida"] if c in df_det.columns]
        display(df_det[cols])

        plt.figure(figsize=(12, 4))
        plt.bar(df_det["vertebra"], df_det["dice"], color="#4C78A8")
        plt.ylim(0, 1)
        plt.ylabel("Dice")
        plt.title("Dice por vertebra en la muestra")
        plt.xticks(rotation=45)
        plt.grid(axis="y", alpha=0.25)
        plt.show()

    return resumen_area


resumen_area_auto = visualizar_comparativo_area_coloreada(
    img_rgb=imagen_auto,
    mask_gt=mask_sem_gt_auto,
    mask_pred=mask_sem_pred_auto,
    prompts_auto=prompts_auto_sample,
    detalles=detalles_auto,
    titulo=f"{patient_id_auto} | Dice macro={dice_auto:.3f} | IoU macro={iou_auto:.3f}"
)


In [ ]:
# ============================================================
# Comparar cualquier paciente del split val/test/train
# ============================================================

def comparar_area_coloreada_muestra(
    split="val",
    patient_id=None,
    escala_w=1.90,
    escala_h=1.35,
    peso_eje=0.90,
    escala_pos_y=0.91,
    offset_y=-16,
    offset_x=-18,
    mid_lift=22,
    frac_x=0.04,
    frac_y=0.06,
):
    if patient_id is None:
        patient_id = sorted(list(PROMPTS_DICC[split].keys()))[0]

    out = evaluar_experimento12_muestra(
        split=split,
        patient_id=patient_id,
        predictor=predictor,
        template_bbox=template_bbox_auto,
        escala_w=escala_w,
        escala_h=escala_h,
        peso_eje=peso_eje,
        escala_pos_y=escala_pos_y,
        offset_y=offset_y,
        offset_x=offset_x,
        mid_lift=mid_lift,
        frac_x=frac_x,
        frac_y=frac_y,
    )

    dice_global = out["resumen"]["dice_macro"]
    iou_global = out["resumen"]["iou_macro"]

    resumen_area = visualizar_comparativo_area_coloreada(
        img_rgb=out["imagen"],
        mask_gt=out["mask_gt"],
        mask_pred=out["mask_pred"],
        prompts_auto=out["prompts_auto"],
        detalles=out["df_detalles"],
        titulo=f"{patient_id} ({split}) | Dice macro={dice_global:.3f} | IoU macro={iou_global:.3f}",
    )

    return out, resumen_area


# Cambia el paciente si quieres revisar otro caso cualitativamente.
# Ejemplo:
# out_caso, resumen_caso = comparar_area_coloreada_muestra(split="val", patient_id="N_12")


In [ ]:
# ============================================================
# Evaluacion del experimento 12 en split completo
# ============================================================

def evaluar_experimento12_muestra(
    split,
    patient_id,
    predictor,
    template_bbox,
    escala_w=1.90,
    escala_h=1.35,
    peso_eje=0.90,
    escala_pos_y=0.91,
    offset_y=-16,
    offset_x=-18,
    mid_lift=22,
    frac_x=0.04,
    frac_y=0.06,
):
    """Evalua una imagen usando cajas automaticas por eje curvo y MedSAM."""
    path_img, path_mask = resolver_paths_muestra(split, patient_id)
    imagen = cargar_imagen(path_img)
    mascara = cargar_mascara(path_mask)

    prompts_auto, info_curva = generar_prompts_auto_por_bordes(
        img_rgb=imagen,
        template_bbox=template_bbox,
        escala_w=escala_w,
        escala_h=escala_h,
        peso_eje=peso_eje,
        escala_pos_y=escala_pos_y,
        offset_y=offset_y,
        offset_x=offset_x,
        mid_lift=mid_lift,
        debug=False,
    )

    df_cobertura = evaluar_cobertura_cajas_por_clase(prompts_auto, mascara)

    mask_sem_pred, mask_sem_gt, detalles, score_map, quality = reconstruir_mascara_semantica_medsam_con_score(
        imagen=imagen,
        mascara_gt_multiclase=mascara,
        prompts_sample=prompts_auto,
        predictor=predictor,
        frac_x=frac_x,
        frac_y=frac_y,
        return_quality=True,
    )

    clases_ids_eval = list(range(1, N_CLASES + 1))
    dice_global = dice_multiclase_promedio(mask_sem_pred, mask_sem_gt, clases_ids_eval)
    iou_global = iou_multiclase_promedio(mask_sem_pred, mask_sem_gt, clases_ids_eval)

    resumen = {
        "split": split,
        "patient_id": patient_id,
        "tipo_real": "escoliosis" if str(patient_id).startswith("S_") else "normal",
        "dice_macro": dice_global,
        "iou_macro": iou_global,
        "bbox_recall_promedio": df_cobertura["bbox_recall"].mean(),
        "bbox_recall_min": df_cobertura["bbox_recall"].min(),
        "bbox_precision_promedio": df_cobertura["bbox_precision"].mean(),
        "bbox_precision_min": df_cobertura["bbox_precision"].min(),
        "bbox_iou_promedio": df_cobertura["bbox_iou"].mean(),
        "bbox_iou_min": df_cobertura["bbox_iou"].min(),
        "bbox_area_ratio_promedio": df_cobertura["bbox_area_ratio"].mean(),
        "bbox_area_ratio_max": df_cobertura["bbox_area_ratio"].max(),
        "n_vertebras_bbox_recall_mayor_08": int((df_cobertura["bbox_recall"] >= 0.80).sum()),
        "n_vertebras_bbox_precision_mayor_025": int((df_cobertura["bbox_precision"] >= 0.25).sum()),
        "n_vertebras_bbox_iou_mayor_02": int((df_cobertura["bbox_iou"] >= 0.20).sum()),
        "prompt_origen": "bbox_auto_bordes_eje_curvo_y_calibrado",
        "escala_w": escala_w,
        "escala_h": escala_h,
        "peso_eje": peso_eje,
        "escala_pos_y": escala_pos_y,
        "offset_y": offset_y,
        "offset_x": offset_x,
        "mid_lift": mid_lift,
    }

    for k, v in quality.items():
        resumen[f"quality_{k}"] = v

    df_cobertura = df_cobertura.copy()
    df_cobertura["split"] = split
    df_cobertura["patient_id"] = patient_id

    df_detalles = pd.DataFrame(detalles)
    df_detalles["split"] = split
    df_detalles["patient_id"] = patient_id

    return {
        "resumen": resumen,
        "df_cobertura": df_cobertura,
        "df_detalles": df_detalles,
        "mask_pred": mask_sem_pred,
        "mask_gt": mask_sem_gt,
        "imagen": imagen,
        "prompts_auto": prompts_auto,
        "info_curva": info_curva,
        "score_map": score_map,
    }


def evaluar_experimento12_split(
    split="val",
    max_items=5,
    escala_w=1.90,
    escala_h=1.35,
    peso_eje=0.90,
    escala_pos_y=0.91,
    offset_y=-16,
    offset_x=-18,
    mid_lift=22,
    frac_x=0.04,
    frac_y=0.06,
):
    """Evalua el experimento 12 en un split completo o parcial."""
    patient_ids = sorted(list(PROMPTS_DICC[split].keys()))
    if max_items is not None:
        patient_ids = patient_ids[:max_items]

    resumenes = []
    coberturas = []
    detalles_todos = []
    errores = []

    for patient_id in tqdm(patient_ids, desc=f"Experimento 12 - {split}"):
        try:
            out = evaluar_experimento12_muestra(
                split=split,
                patient_id=patient_id,
                predictor=predictor,
                template_bbox=template_bbox_auto,
                escala_w=escala_w,
                escala_h=escala_h,
                peso_eje=peso_eje,
                escala_pos_y=escala_pos_y,
                offset_y=offset_y,
                offset_x=offset_x,
                mid_lift=mid_lift,
                frac_x=frac_x,
                frac_y=frac_y,
            )
            resumenes.append(out["resumen"])
            coberturas.append(out["df_cobertura"])
            detalles_todos.append(out["df_detalles"])
        except Exception as e:
            errores.append({"split": split, "patient_id": patient_id, "error": str(e)})
            print(f"Error en {patient_id}: {e}")

    df_resumen = pd.DataFrame(resumenes)
    df_coberturas = pd.concat(coberturas, ignore_index=True) if coberturas else pd.DataFrame()
    df_detalles = pd.concat(detalles_todos, ignore_index=True) if detalles_todos else pd.DataFrame()
    df_errores = pd.DataFrame(errores)
    return df_resumen, df_coberturas, df_detalles, df_errores


In [ ]:
df_exp12_resumen_val_5, df_exp12_cobertura_val_5, df_exp12_detalles_val_5, df_exp12_errores_val_5 = evaluar_experimento12_split(
    split="val",
    max_items=5,
    escala_w=1.90,
    escala_h=1.35,
    peso_eje=0.90,
    escala_pos_y=0.91,
    offset_y=-16,
    offset_x=-18,
    mid_lift=22,
    frac_x=0.04,
    frac_y=0.06,
)

display(df_exp12_resumen_val_5)
display(df_exp12_cobertura_val_5)
display(df_exp12_errores_val_5)

print("Resumen parcial experimento 12:")
print("Dice macro promedio:", df_exp12_resumen_val_5["dice_macro"].mean())
print("IoU macro promedio :", df_exp12_resumen_val_5["iou_macro"].mean())
print("BBox recall promedio   :", df_exp12_resumen_val_5["bbox_recall_promedio"].mean())
print("BBox precision promedio:", df_exp12_resumen_val_5["bbox_precision_promedio"].mean())
print("BBox IoU promedio      :", df_exp12_resumen_val_5["bbox_iou_promedio"].mean())
print("Ratio area caja prom.  :", df_exp12_resumen_val_5["bbox_area_ratio_promedio"].mean())


## 7. Comparacion de ejes: bordes vs respuesta central
Compara el eje antiguo por bordes con el nuevo eje por respuesta central. En escoliosis, el eje central deberia curvarse mas cuando la imagen lo justifique.

In [ ]:
# ============================================================
# Comparar visualmente eje por bordes vs central vs central robusto
# ============================================================

def comparar_ejes_muestra(split="val", patient_id="N_12"):
    path_img, path_mask = resolver_paths_muestra(split, patient_id)
    img = cargar_imagen(path_img)
    mask = cargar_mascara(path_mask)
    infos = [
        ("bordes", estimar_eje_columna(img, template_bbox=template_bbox_auto, metodo="bordes")),
        ("central", estimar_eje_columna(img, template_bbox=template_bbox_auto, metodo="central")),
        ("central_robusto", estimar_eje_columna(img, template_bbox=template_bbox_auto, metodo="central_robusto")),
        ("ruta_dinamica", estimar_eje_columna(img, template_bbox=template_bbox_auto, metodo="ruta_dinamica")),
    ]

    fig, ax = plt.subplots(1, 4, figsize=(22, 8))
    for a, (nombre, info) in zip(ax, infos):
        a.imshow(img)
        overlay = np.zeros_like(img)
        overlay[..., 0] = (mask > 0).astype(np.uint8) * 255
        a.imshow(overlay, alpha=0.18)
        info_plot = limitar_info_eje_a_template(info, img.shape, template_bbox=template_bbox_auto, margen_rel=0.05)
        pts = info_plot["puntos_centro_raw"]
        smooth = info_plot["puntos_smooth"]
        a.scatter(pts[:, 1], pts[:, 0], s=12, c="cyan")
        a.plot(smooth[:, 1], smooth[:, 0], c="yellow", linewidth=2)
        a.set_title(f"{patient_id} - eje {nombre}")
        a.axis("off")
    plt.tight_layout()
    plt.show()
    return {nombre: info for nombre, info in infos}

infos_eje_n12 = comparar_ejes_muestra(split="val", patient_id="N_12")
if "S_187" in PROMPTS_DICC["val"]:
    infos_eje_s187 = comparar_ejes_muestra(split="val", patient_id="S_187")


## Inspeccion puntual de radiografia dificil
Vista ampliada de un caso dificil para revisar si el problema viene de la imagen, de la mascara, de la curva o de la posicion inicial de cajas.

In [ ]:
# ============================================================
# Inspeccion visual ampliada de un caso dificil
# ============================================================

PACIENTE_INSPECCION = "S_187"
SPLIT_INSPECCION = "val"

path_img_ins, path_mask_ins = resolver_paths_muestra(SPLIT_INSPECCION, PACIENTE_INSPECCION)
img_ins = cargar_imagen(path_img_ins)
mask_ins = cargar_mascara(path_mask_ins)
mask_ins_t1_l5 = construir_mask_multiclase_t1_l5(mask_ins)

print("Imagen:", path_img_ins)
print("Mascara:", path_mask_ins)
print("Shape imagen:", img_ins.shape)
print("Shape mascara:", mask_ins.shape)
print("IDs en mascara:", np.unique(mask_ins_t1_l5))

fig, ax = plt.subplots(1, 3, figsize=(18, 9))
ax[0].imshow(img_ins)
ax[0].set_title(f"{PACIENTE_INSPECCION} - radiografia")
ax[0].axis("off")

ax[1].imshow(mask_ins_t1_l5, cmap="nipy_spectral", vmin=0, vmax=N_CLASES)
ax[1].set_title("Mascara GT T1-L5")
ax[1].axis("off")

ax[2].imshow(img_ins)
overlay = np.zeros_like(img_ins)
overlay[..., 0] = (mask_ins_t1_l5 > 0).astype(np.uint8) * 255
ax[2].imshow(overlay, alpha=0.30)
ax[2].set_title("Overlay GT sobre radiografia")
ax[2].axis("off")
plt.tight_layout()
plt.show()

# Comparar ejes encima del mismo caso.
infos_ins = {}
for metodo in ["bordes", "central", "central_robusto", "ruta_dinamica"]:
    infos_ins[metodo] = estimar_eje_columna(img_ins, template_bbox=template_bbox_auto, metodo=metodo)

fig, ax = plt.subplots(1, 4, figsize=(22, 9))
for a, metodo in zip(ax, infos_ins.keys()):
    info = infos_ins[metodo]
    a.imshow(img_ins)
    a.imshow(overlay, alpha=0.18)
    info_plot = limitar_info_eje_a_template(info, img_ins.shape, template_bbox=template_bbox_auto, margen_rel=0.05)
    pts = info_plot["puntos_centro_raw"]
    smooth = info_plot["puntos_smooth"]
    a.scatter(pts[:, 1], pts[:, 0], s=14, c="cyan")
    a.plot(smooth[:, 1], smooth[:, 0], c="yellow", linewidth=2)
    a.set_title(metodo)
    a.axis("off")
plt.tight_layout()
plt.show()

# Ver cajas para la ruta dinamica con parametros locales de inspeccion.
PARAMS_INSPECCION_ESCOLIOSIS = {
    "escala_w": 1.70,
    "escala_h": 1.35,
    "peso_eje": 1.00,
    "escala_pos_y": 0.91,
    "offset_y": -16,
    "offset_x": -10,
    "mid_lift": 15,
    "metodo_eje": "ruta_dinamica",
}

prompts_ins, info_eje_ins = generar_prompts_auto_por_bordes(
    img_rgb=img_ins,
    template_bbox=template_bbox_auto,
    debug=False,
    **PARAMS_INSPECCION_ESCOLIOSIS,
)

df_cajas_ins = evaluar_cobertura_cajas_por_clase(prompts_ins, mask_ins)
visualizar_cajas_auto_bordes(
    img_rgb=img_ins,
    prompts_auto=prompts_ins,
    info_eje=info_eje_ins,
    mascara_gt=mask_ins,
    titulo=f"{PACIENTE_INSPECCION} - cajas con estrategia escoliosis actual"
)
display(df_cajas_ins)


## 8. Refinamiento local de centros vertebrales
Esta seccion compara las cajas base contra cajas refinadas por respuesta local, sin usar mascara GT para generarlas. La mascara solo se usa para medir si el refinamiento acerco la caja a cada vertebra.

In [ ]:
# ============================================================
# Probar refinamiento local en un normal y un escoliotico dificil
# ============================================================

res_ref_n12 = comparar_refinamiento_local_muestra(
    split="val",
    patient_id="N_12",
    metodo_eje="central_robusto",
)

resumen_ref_n12, df_base_n12, df_ref_n12, df_mov_n12, prompts_base_n12, prompts_ref_n12 = res_ref_n12
display(resumen_ref_n12)
display(df_mov_n12)

if "S_187" in PROMPTS_DICC["val"]:
    res_ref_s187 = comparar_refinamiento_local_muestra(
        split="val",
        patient_id="S_187",
        metodo_eje="central_robusto",
    )
    resumen_ref_s187, df_base_s187, df_ref_s187, df_mov_s187, prompts_base_s187, prompts_ref_s187 = res_ref_s187
    display(resumen_ref_s187)
    display(df_mov_s187)


## 9. Clasificador geometrico normal vs escoliosis
Antes de pasar a MedSAM, esta seccion estima si la imagen parece normal o escoliotica a partir de la forma del eje curvo. La meta no es diagnostico clinico, sino elegir una estrategia de cajas mas adecuada.

In [ ]:
# ============================================================
# Features geometricas del eje para clasificar normal/escoliosis
# ============================================================

def tipo_desde_patient_id(patient_id):
    pid = str(patient_id)
    if pid.startswith("S_"):
        return "escoliosis"
    if pid.startswith("N_"):
        return "normal"
    return "desconocido"


def extraer_features_eje_columna(img_rgb, template_bbox=None):
    """
    Extrae features del eje estimado sin usar mascara GT.
    Estas features sirven para decidir si la columna parece normal o escoliotica antes de MedSAM.
    """
    info = estimar_eje_curvo_por_bordes(img_rgb, template_bbox=template_bbox)
    puntos = info["puntos_smooth"]
    y = puntos[:, 0].astype(float)
    x = puntos[:, 1].astype(float)

    H, W = img_rgb.shape[:2]
    x_roi0, y_roi0, x_roi1, y_roi1 = info["roi"]
    roi_w = max(x_roi1 - x_roi0 + 1, 1)
    roi_h = max(y_roi1 - y_roi0 + 1, 1)

    coef_linea = np.polyfit(y, x, deg=1)
    x_linea = np.poly1d(coef_linea)(y)
    desv = x - x_linea

    dx = float(x[-1] - x[0])
    rango_x = float(x.max() - x.min())
    max_desv = float(np.max(np.abs(desv)))
    rms_desv = float(np.sqrt(np.mean(desv ** 2)))

    # Curvatura discreta aproximada: cambios de pendiente normalizados.
    slope = np.gradient(x, y + 1e-6)
    curvature = np.gradient(slope, y + 1e-6)

    features = {
        "rango_x_px": rango_x,
        "rango_x_rel": rango_x / W,
        "rango_x_roi_rel": rango_x / roi_w,
        "max_desv_linea_px": max_desv,
        "max_desv_linea_rel": max_desv / W,
        "max_desv_linea_roi_rel": max_desv / roi_w,
        "rms_desv_linea_px": rms_desv,
        "rms_desv_linea_rel": rms_desv / W,
        "tilt_global_px": abs(dx),
        "tilt_global_rel": abs(dx) / H,
        "curvatura_media_abs": float(np.mean(np.abs(curvature))),
        "curvatura_max_abs": float(np.max(np.abs(curvature))),
        "roi_w_rel": roi_w / W,
        "roi_h_rel": roi_h / H,
    }

    # Score interpretable: alto = eje mas curvo/desviado.
    features["score_escoliosis_geom"] = (
        0.45 * features["rango_x_rel"] +
        0.35 * features["max_desv_linea_rel"] +
        0.20 * features["rms_desv_linea_rel"]
    )

    return features, info


def construir_df_features_eje(split="train", max_items=None):
    patient_ids = sorted(list(PROMPTS_DICC[split].keys()))
    if max_items is not None:
        patient_ids = patient_ids[:max_items]

    filas = []
    errores = []

    for patient_id in tqdm(patient_ids, desc=f"Features eje - {split}"):
        try:
            path_img, _ = resolver_paths_muestra(split, patient_id)
            img = cargar_imagen(path_img)
            feats, _ = extraer_features_eje_columna(img, template_bbox=template_bbox_auto)
            feats["split"] = split
            feats["patient_id"] = patient_id
            feats["tipo_real"] = tipo_desde_patient_id(patient_id)
            feats["y_true_escoliosis"] = int(feats["tipo_real"] == "escoliosis")
            filas.append(feats)
        except Exception as e:
            errores.append({"split": split, "patient_id": patient_id, "error": str(e)})

    return pd.DataFrame(filas), pd.DataFrame(errores)


def buscar_umbral_escoliosis(df_features, score_col="score_escoliosis_geom"):
    df = df_features[df_features["tipo_real"].isin(["normal", "escoliosis"])].copy()
    valores = np.sort(df[score_col].dropna().unique())
    candidatos = []

    if len(valores) == 0:
        raise ValueError("No hay valores para calibrar umbral.")

    puntos = np.r_[valores[0] - 1e-6, (valores[:-1] + valores[1:]) / 2, valores[-1] + 1e-6]

    for thr in puntos:
        pred = (df[score_col] >= thr).astype(int)
        true = df["y_true_escoliosis"].astype(int)
        tp = int(((pred == 1) & (true == 1)).sum())
        tn = int(((pred == 0) & (true == 0)).sum())
        fp = int(((pred == 1) & (true == 0)).sum())
        fn = int(((pred == 0) & (true == 1)).sum())
        sens = tp / max(tp + fn, 1)
        spec = tn / max(tn + fp, 1)
        bal_acc = 0.5 * (sens + spec)
        acc = (tp + tn) / max(len(df), 1)
        candidatos.append({
            "umbral": float(thr),
            "accuracy": float(acc),
            "balanced_accuracy": float(bal_acc),
            "sensibilidad_escoliosis": float(sens),
            "especificidad_normal": float(spec),
            "tp": tp,
            "tn": tn,
            "fp": fp,
            "fn": fn,
        })

    return pd.DataFrame(candidatos).sort_values(
        ["balanced_accuracy", "accuracy"], ascending=False
    ).reset_index(drop=True)


def aplicar_clasificador_geom(df_features, umbral, score_col="score_escoliosis_geom"):
    df = df_features.copy()
    df["tipo_pred_geom"] = np.where(df[score_col] >= umbral, "escoliosis", "normal")
    df["pred_correcta"] = df["tipo_pred_geom"] == df["tipo_real"]
    return df


In [ ]:
# ============================================================
# Calibrar en train y revisar en val
# ============================================================

MAX_ITEMS_CLF_TRAIN = None
MAX_ITEMS_CLF_VAL = None

df_features_train, df_features_train_errores = construir_df_features_eje(
    split="train",
    max_items=MAX_ITEMS_CLF_TRAIN,
)

df_umbral_geom = buscar_umbral_escoliosis(df_features_train)
UMBRAL_ESCOLIOSIS_GEOM = float(df_umbral_geom.iloc[0]["umbral"])

print("Umbral geometrico seleccionado:", UMBRAL_ESCOLIOSIS_GEOM)
display(df_umbral_geom.head(10))
display(df_features_train_errores)

df_features_val, df_features_val_errores = construir_df_features_eje(
    split="val",
    max_items=MAX_ITEMS_CLF_VAL,
)

df_features_train_pred = aplicar_clasificador_geom(df_features_train, UMBRAL_ESCOLIOSIS_GEOM)
df_features_val_pred = aplicar_clasificador_geom(df_features_val, UMBRAL_ESCOLIOSIS_GEOM)

def resumen_clasificador_geom(df_pred, nombre=""):
    df = df_pred[df_pred["tipo_real"].isin(["normal", "escoliosis"])].copy()
    tab = pd.crosstab(df["tipo_real"], df["tipo_pred_geom"], margins=True)
    acc = df["pred_correcta"].mean()
    print(f"{nombre} accuracy: {acc:.3f} ({df['pred_correcta'].sum()}/{len(df)})")
    display(tab)
    display(
        df[[
            "patient_id", "tipo_real", "tipo_pred_geom", "pred_correcta",
            "score_escoliosis_geom", "rango_x_rel", "max_desv_linea_rel", "rms_desv_linea_rel",
            "tilt_global_rel"
        ]].sort_values("score_escoliosis_geom", ascending=False)
    )

resumen_clasificador_geom(df_features_train_pred, "Train")
resumen_clasificador_geom(df_features_val_pred, "Val")
display(df_features_val_errores)


In [ ]:
# ============================================================
# Visualizar distribucion del score geometrico
# ============================================================

fig, ax = plt.subplots(1, 2, figsize=(14, 4))

for tipo, color in [("normal", "#4C78A8"), ("escoliosis", "#F58518")]:
    vals = df_features_train_pred.loc[df_features_train_pred["tipo_real"] == tipo, "score_escoliosis_geom"]
    ax[0].hist(vals, bins=20, alpha=0.65, label=tipo, color=color)

ax[0].axvline(UMBRAL_ESCOLIOSIS_GEOM, color="black", linestyle="--", label="umbral")
ax[0].set_title("Train: score geometrico")
ax[0].set_xlabel("score_escoliosis_geom")
ax[0].legend()

for tipo, color in [("normal", "#4C78A8"), ("escoliosis", "#F58518")]:
    vals = df_features_val_pred.loc[df_features_val_pred["tipo_real"] == tipo, "score_escoliosis_geom"]
    ax[1].hist(vals, bins=20, alpha=0.65, label=tipo, color=color)

ax[1].axvline(UMBRAL_ESCOLIOSIS_GEOM, color="black", linestyle="--", label="umbral")
ax[1].set_title("Val: score geometrico")
ax[1].set_xlabel("score_escoliosis_geom")
ax[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Comparar metricas de cajas por grupo normal/escoliosis
# ============================================================

if "df_cmp_pacientes" in globals() and not df_cmp_pacientes.empty:
    df_cmp_pacientes = df_cmp_pacientes.copy()
    df_cmp_pacientes["tipo_real"] = df_cmp_pacientes["patient_id"].apply(tipo_desde_patient_id)

    df_cmp_grupo = (
        df_cmp_pacientes
        .groupby(["estrategia", "tipo_real"], as_index=False)
        .agg(
            pacientes=("patient_id", "nunique"),
            bbox_iou_promedio=("bbox_iou_promedio", "mean"),
            bbox_recall_promedio=("bbox_recall_promedio", "mean"),
            bbox_precision_promedio=("bbox_precision_promedio", "mean"),
            bbox_area_ratio_promedio=("bbox_area_ratio_promedio", "mean"),
            dx_abs_promedio=("dx_abs_promedio", "mean"),
            dy_abs_promedio=("dy_abs_promedio", "mean"),
        )
        .sort_values(["tipo_real", "bbox_iou_promedio"], ascending=[True, False])
        .reset_index(drop=True)
    )

    print("Top estrategias para normales")
    display(df_cmp_grupo[df_cmp_grupo["tipo_real"] == "normal"].head(10))

    print("Top estrategias para escoliosis")
    display(df_cmp_grupo[df_cmp_grupo["tipo_real"] == "escoliosis"].head(10))
else:
    print("Ejecuta primero la grilla de estrategias de cajas para ver metricas por grupo.")


## 10. Comparacion de estrategias de cajas
Esta seccion compara configuraciones de cajas automaticas usando metricas geometricas contra GT. Es rapida porque no ejecuta MedSAM; sirve para decidir que parametros valen la pena probar cualitativamente y luego con segmentacion.

In [ ]:
# ============================================================
# Comparacion rapida de estrategias de cajas automaticas
# ============================================================

def evaluar_cajas_automaticas_muestra(
    split,
    patient_id,
    template_bbox,
    escala_w=1.90,
    escala_h=1.35,
    peso_eje=0.90,
    escala_pos_y=0.91,
    offset_y=-16,
    offset_x=-18,
    mid_lift=22,
    metodo_eje="bordes",
):
    path_img, path_mask = resolver_paths_muestra(split, patient_id)
    imagen = cargar_imagen(path_img)
    mascara = cargar_mascara(path_mask)

    prompts_auto, info_curva = generar_prompts_auto_por_bordes(
        img_rgb=imagen,
        template_bbox=template_bbox,
        escala_w=escala_w,
        escala_h=escala_h,
        peso_eje=peso_eje,
        escala_pos_y=escala_pos_y,
        offset_y=offset_y,
        offset_x=offset_x,
        mid_lift=mid_lift,
        metodo_eje=metodo_eje,
        debug=False,
    )

    df_cajas = evaluar_cobertura_cajas_por_clase(prompts_auto, mascara)
    df_diag = diagnosticar_desplazamiento_cajas(prompts_auto, mascara)

    resumen = {
        "split": split,
        "patient_id": patient_id,
        "tipo_real": "escoliosis" if str(patient_id).startswith("S_") else "normal",
        "bbox_recall_promedio": df_cajas["bbox_recall"].mean(),
        "bbox_recall_min": df_cajas["bbox_recall"].min(),
        "bbox_precision_promedio": df_cajas["bbox_precision"].mean(),
        "bbox_precision_min": df_cajas["bbox_precision"].min(),
        "bbox_iou_promedio": df_cajas["bbox_iou"].mean(),
        "bbox_iou_min": df_cajas["bbox_iou"].min(),
        "bbox_area_ratio_promedio": df_cajas["bbox_area_ratio"].mean(),
        "bbox_area_ratio_max": df_cajas["bbox_area_ratio"].max(),
        "dx_abs_promedio": df_diag["dx_box_gt"].abs().mean() if len(df_diag) else np.nan,
        "dy_abs_promedio": df_diag["dy_box_gt"].abs().mean() if len(df_diag) else np.nan,
        "n_vertebras_recall_mayor_08": int((df_cajas["bbox_recall"] >= 0.80).sum()),
        "n_vertebras_precision_mayor_025": int((df_cajas["bbox_precision"] >= 0.25).sum()),
        "n_vertebras_iou_mayor_02": int((df_cajas["bbox_iou"] >= 0.20).sum()),
        "escala_w": escala_w,
        "escala_h": escala_h,
        "peso_eje": peso_eje,
        "escala_pos_y": escala_pos_y,
        "offset_y": offset_y,
        "offset_x": offset_x,
        "mid_lift": mid_lift,
        "metodo_eje": metodo_eje,
    }

    return resumen, df_cajas, df_diag, prompts_auto, info_curva, imagen, mascara


def comparar_estrategias_cajas(
    estrategias,
    split="val",
    max_items=10,
    patient_ids=None,
    ordenar_por="bbox_iou_promedio",
):
    if patient_ids is None:
        patient_ids = sorted(list(PROMPTS_DICC[split].keys()))
        if max_items is not None:
            patient_ids = patient_ids[:max_items]

    resumenes = []
    detalles = []
    errores = []

    for estrategia in tqdm(estrategias, desc="Estrategias de cajas"):
        nombre = estrategia.get("nombre", "sin_nombre")
        params = {k: v for k, v in estrategia.items() if k != "nombre"}

        for patient_id in patient_ids:
            try:
                resumen, df_cajas, df_diag, *_ = evaluar_cajas_automaticas_muestra(
                    split=split,
                    patient_id=patient_id,
                    template_bbox=template_bbox_auto,
                    **params,
                )
                resumen["estrategia"] = nombre
                resumenes.append(resumen)

                df_det = df_cajas.copy()
                df_det["estrategia"] = nombre
                df_det["split"] = split
                df_det["patient_id"] = patient_id
                detalles.append(df_det)
            except Exception as e:
                errores.append({
                    "estrategia": nombre,
                    "split": split,
                    "patient_id": patient_id,
                    "error": str(e),
                })

    df_resumen = pd.DataFrame(resumenes)
    df_detalles = pd.concat(detalles, ignore_index=True) if detalles else pd.DataFrame()
    df_errores = pd.DataFrame(errores)

    if df_resumen.empty:
        return pd.DataFrame(), df_resumen, df_detalles, df_errores

    df_estrategias = (
        df_resumen
        .groupby("estrategia", as_index=False)
        .agg(
            pacientes=("patient_id", "nunique"),
            bbox_recall_promedio=("bbox_recall_promedio", "mean"),
            bbox_recall_min_promedio=("bbox_recall_min", "mean"),
            bbox_precision_promedio=("bbox_precision_promedio", "mean"),
            bbox_iou_promedio=("bbox_iou_promedio", "mean"),
            bbox_iou_min_promedio=("bbox_iou_min", "mean"),
            bbox_area_ratio_promedio=("bbox_area_ratio_promedio", "mean"),
            bbox_area_ratio_max_promedio=("bbox_area_ratio_max", "mean"),
            dx_abs_promedio=("dx_abs_promedio", "mean"),
            dy_abs_promedio=("dy_abs_promedio", "mean"),
            vertebras_recall_08_prom=("n_vertebras_recall_mayor_08", "mean"),
            vertebras_precision_025_prom=("n_vertebras_precision_mayor_025", "mean"),
            vertebras_iou_02_prom=("n_vertebras_iou_mayor_02", "mean"),
        )
        .sort_values(ordenar_por, ascending=False)
        .reset_index(drop=True)
    )

    return df_estrategias, df_resumen, df_detalles, df_errores


In [ ]:
# ============================================================
# Grilla de estrategias alrededor de la mejor forma observada
# ============================================================

from itertools import product


def generar_grilla_estrategias_cajas(
    escala_w_vals=(1.40, 1.50, 1.60, 1.70),
    escala_h_vals=(1.20, 1.25, 1.30, 1.35),
    mid_lift_vals=(15, 22),
    offset_x_vals=(-18, -10),
    metodo_eje_vals=("bordes", "central", "central_robusto", "ruta_dinamica"),
    peso_eje=0.90,
    escala_pos_y=0.91,
    offset_y=-16,
):
    estrategias = []
    for escala_w, escala_h, mid_lift, offset_x, metodo_eje in product(
        escala_w_vals,
        escala_h_vals,
        mid_lift_vals,
        offset_x_vals,
        metodo_eje_vals,
    ):
        estrategias.append({
            "nombre": f"{metodo_eje}_w{int(escala_w*100):03d}_h{int(escala_h*100):03d}_lift{mid_lift}_x{offset_x}",
            "escala_w": escala_w,
            "escala_h": escala_h,
            "peso_eje": peso_eje,
            "escala_pos_y": escala_pos_y,
            "offset_y": offset_y,
            "offset_x": offset_x,
            "mid_lift": mid_lift,
            "metodo_eje": metodo_eje,
        })
    return estrategias


ESTRATEGIAS_CAJAS = generar_grilla_estrategias_cajas()
print(f"Estrategias en grilla: {len(ESTRATEGIAS_CAJAS)}")

# Empieza pequeno para iterar rapido. Sube max_items a None cuando quieras usar todo validation.
MAX_ITEMS_GRILLA = None

df_cmp_estrategias, df_cmp_pacientes, df_cmp_detalles, df_cmp_errores = comparar_estrategias_cajas(
    estrategias=ESTRATEGIAS_CAJAS,
    split="val",
    max_items=MAX_ITEMS_GRILLA,
    ordenar_por="bbox_iou_promedio",
)

# Ranking principal: prioriza bbox_iou porque penaliza tanto cajas que se quedan cortas como cajas demasiado grandes.
display(df_cmp_estrategias.head(15))
display(df_cmp_errores)

# Vista compacta de trade-off: mejor promedio por escala_w / escala_h, colapsando lift y offset_x.
if not df_cmp_pacientes.empty:
    df_tradeoff_wh = (
        df_cmp_pacientes
        .groupby(["escala_w", "escala_h"], as_index=False)
        .agg(
            bbox_iou_promedio=("bbox_iou_promedio", "mean"),
            bbox_recall_promedio=("bbox_recall_promedio", "mean"),
            bbox_precision_promedio=("bbox_precision_promedio", "mean"),
            bbox_area_ratio_promedio=("bbox_area_ratio_promedio", "mean"),
        )
        .sort_values("bbox_iou_promedio", ascending=False)
        .reset_index(drop=True)
    )
    display(df_tradeoff_wh)

    fig, ax = plt.subplots(1, 3, figsize=(18, 4))
    metricas_heatmap = [
        ("bbox_iou_promedio", "BBox IoU"),
        ("bbox_recall_promedio", "BBox recall"),
        ("bbox_precision_promedio", "BBox precision"),
    ]
    for k, (metrica, titulo) in enumerate(metricas_heatmap):
        pivot = df_tradeoff_wh.pivot(index="escala_h", columns="escala_w", values=metrica).sort_index(ascending=False)
        im = ax[k].imshow(pivot.values, cmap="viridis", aspect="auto")
        ax[k].set_title(titulo)
        ax[k].set_xlabel("escala_w")
        ax[k].set_ylabel("escala_h")
        ax[k].set_xticks(range(len(pivot.columns)))
        ax[k].set_xticklabels([f"{v:.2f}" for v in pivot.columns])
        ax[k].set_yticks(range(len(pivot.index)))
        ax[k].set_yticklabels([f"{v:.2f}" for v in pivot.index])
        for y in range(pivot.shape[0]):
            for x in range(pivot.shape[1]):
                ax[k].text(x, y, f"{pivot.values[y, x]:.3f}", ha="center", va="center", color="white", fontsize=8)
        fig.colorbar(im, ax=ax[k], fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# Visualizar mejores estrategias y casos dificiles
# ============================================================

if not df_cmp_estrategias.empty:
    top_nombres = df_cmp_estrategias["estrategia"].head(3).tolist()
    mejor_nombre = top_nombres[0]

    df_mejor_pacientes = df_cmp_pacientes[df_cmp_pacientes["estrategia"] == mejor_nombre].copy()
    paciente_mejor = df_mejor_pacientes.sort_values("bbox_iou_promedio", ascending=False)["patient_id"].iloc[0]
    paciente_peor = df_mejor_pacientes.sort_values("bbox_iou_promedio", ascending=True)["patient_id"].iloc[0]

    pacientes_a_ver = []
    for pid in ["N_12", paciente_mejor, paciente_peor]:
        if pid in df_mejor_pacientes["patient_id"].tolist() and pid not in pacientes_a_ver:
            pacientes_a_ver.append(pid)

    print("Top estrategias por bbox_iou:", top_nombres)
    print("Pacientes a revisar:", pacientes_a_ver)

    for paciente_comparativo in pacientes_a_ver:
        print("=" * 80)
        print(f"Paciente: {paciente_comparativo}")

        for nombre in top_nombres:
            estrategia = next(e for e in ESTRATEGIAS_CAJAS if e["nombre"] == nombre)
            params = {k: v for k, v in estrategia.items() if k != "nombre"}

            resumen, df_cajas, df_diag, prompts_auto_cmp, info_curva_cmp, imagen_cmp, mascara_cmp = evaluar_cajas_automaticas_muestra(
                split="val",
                patient_id=paciente_comparativo,
                template_bbox=template_bbox_auto,
                **params,
            )

            visualizar_cajas_auto_bordes(
                img_rgb=imagen_cmp,
                prompts_auto=prompts_auto_cmp,
                info_eje=info_curva_cmp,
                mascara_gt=mascara_cmp,
                titulo=(
                    f"{paciente_comparativo} | {nombre} | "
                    f"recall={resumen['bbox_recall_promedio']:.3f} | "
                    f"precision={resumen['bbox_precision_promedio']:.3f} | "
                    f"IoU={resumen['bbox_iou_promedio']:.3f}"
                ),
            )

            cols = [
                "vertebra", "bbox_recall", "bbox_precision", "bbox_iou",
                "bbox_area_ratio", "bbox_auto"
            ]
            display(df_cajas[cols])


## 10B. Multi-ruta para escoliosis y corridas largas seguras
 
 Esta seccion prueba varias rutas laterales candidatas. La idea es verificar si el problema de S_187 es que el eje correcto existe, pero la estrategia actual esta escogiendo otra banda de la radiografia.


In [ ]:
# ============================================================
# Multi-ruta: varias hipotesis laterales para escoliosis
# ============================================================

PARAMS_MULTIRUTA_BASE = {
    "escala_w": 1.70,
    "escala_h": 1.35,
    "peso_eje": 1.00,
    "escala_pos_y": 0.91,
    "offset_y": -16,
    "offset_x": -10,
    "mid_lift": 15,
    "metodo_eje": "ruta_dinamica_multi",
}

OFFSETS_MULTIRUTA_FRAC = (-0.30, -0.20, -0.10, 0.00, 0.10, 0.20, 0.30)


def generar_prompts_auto_desde_info_eje(
    img_rgb,
    template_bbox,
    info_eje,
    escala_w=1.70,
    escala_h=1.35,
    peso_eje=1.00,
    escala_pos_y=0.91,
    offset_y=-16,
    offset_x=-10,
    mid_lift=15,
    min_w_frac=0.10,
    min_h_frac=0.035,
    max_w_frac=0.34,
    max_h_frac=0.13,
    limitar_rango_y_anatomico=True,
):
    """Genera cajas usando un eje ya calculado, sin volver a estimarlo."""

    H, W = img_rgb.shape[:2]

    if limitar_rango_y_anatomico:
        info_eje = limitar_info_eje_a_template(
            info_eje,
            img_rgb.shape,
            template_bbox=template_bbox,
            margen_rel=0.05,
        )

    curva = info_eje["polinomio"]
    prompts_auto = {}
    filas_debug = []
    template_ordenado = template_bbox.sort_values("id_real").reset_index(drop=True)

    y_anchor = float(template_ordenado.iloc[0]["cy_rel"] * H)
    y_min_template = float(template_ordenado["cy_rel"].min() * H)
    y_max_template = float(template_ordenado["cy_rel"].max() * H)

    for _, row in template_ordenado.iterrows():
        vertebra = row["vertebra"]
        if vertebra not in CLASES_OBJETIVO:
            continue

        id_real = int(row["id_real"])
        cy_original = float(row["cy_rel"] * H)
        t = (cy_original - y_min_template) / (y_max_template - y_min_template + 1e-6)
        t = float(np.clip(t, 0, 1))
        correccion_media = mid_lift * np.sin(np.pi * t)

        cy = y_anchor + escala_pos_y * (cy_original - y_anchor) + offset_y - correccion_media
        cy = float(np.clip(cy, 0, H - 1))

        cx_eje = float(curva(cy))
        cx_template = float(row["cx_rel"] * W)
        cx = peso_eje * cx_eje + (1 - peso_eje) * cx_template + offset_x
        cx = float(np.clip(cx, 0, W - 1))

        bw = float(row["w_rel"] * W * escala_w)
        bh = float(row["h_rel"] * H * escala_h)
        bw = float(np.clip(bw, W * min_w_frac, W * max_w_frac))
        bh = float(np.clip(bh, H * min_h_frac, H * max_h_frac))

        x0 = int(round(cx - bw / 2))
        x1 = int(round(cx + bw / 2))
        y0 = int(round(cy - bh / 2))
        y1 = int(round(cy + bh / 2))

        x0 = max(0, x0)
        y0 = max(0, y0)
        x1 = min(W - 1, x1)
        y1 = min(H - 1, y1)

        prompts_auto[vertebra] = {
            "vertebra": vertebra,
            "id_real": id_real,
            "bbox_xyxy": [x0, y0, x1, y1],
            "prompt_origen": "bbox_auto_multi_ruta",
        }

        filas_debug.append({
            "vertebra": vertebra,
            "id_real": id_real,
            "cy_final": cy,
            "cx_eje": cx_eje,
            "cx_template": cx_template,
            "cx_final": cx,
            "bbox_xyxy": [x0, y0, x1, y1],
        })

    return prompts_auto, info_eje, pd.DataFrame(filas_debug)


def puntuar_prompts_por_respuesta_visual(prompts_auto, score_img):
    """Score sin GT: respuesta promedio del mapa visual dentro de las cajas."""

    if score_img is None:
        return np.nan

    vals = []
    for info in prompts_auto.values():
        x0, y0, x1, y1 = info["bbox_xyxy"]
        crop = score_img[y0:y1 + 1, x0:x1 + 1]
        if crop.size:
            vals.append(float(np.mean(crop)))

    return float(np.mean(vals)) if vals else np.nan


def penalizar_ruta_por_saltos(info_eje, ancho_img):
    pts = np.asarray(info_eje.get("puntos_smooth", []))
    if pts.ndim != 2 or len(pts) < 5:
        return np.nan

    dx = np.diff(pts[:, 1])
    return float(np.percentile(np.abs(dx), 90) / max(1, ancho_img))


def evaluar_multi_ruta_muestra(
    split,
    patient_id,
    params_base=None,
    offsets_frac=OFFSETS_MULTIRUTA_FRAC,
    search_half_frac=0.16,
    n_franjas=64,
):
    """Evalua varias rutas laterales en una muestra. Usa GT solo para diagnostico/ranking."""

    if params_base is None:
        params_base = PARAMS_MULTIRUTA_BASE

    path_img, path_mask = resolver_paths_muestra(split, patient_id)
    imagen = cargar_imagen(path_img)
    mascara = cargar_mascara(path_mask)
    H, W = imagen.shape[:2]

    x_prior_base = float(np.median(template_bbox_auto["cx_rel"].to_numpy(dtype=float) * W))
    filas = []
    detalles = {}

    for offset_frac in offsets_frac:
        nombre_ruta = f"multi_{offset_frac:+.2f}"
        x_prior = x_prior_base + float(offset_frac) * W

        try:
            info_eje = estimar_eje_por_ruta_dinamica(
                imagen,
                template_bbox=template_bbox_auto,
                search_half_frac=search_half_frac,
                n_franjas=n_franjas,
                x_prior_override=x_prior,
                ruta_nombre=nombre_ruta,
            )

            params = {k: v for k, v in params_base.items() if k != "metodo_eje"}
            prompts_auto, info_eje_lim, df_debug = generar_prompts_auto_desde_info_eje(
                imagen,
                template_bbox_auto,
                info_eje,
                **params,
            )

            df_cajas = evaluar_cobertura_cajas_por_clase(prompts_auto, mascara)
            df_diag = diagnosticar_desplazamiento_cajas(prompts_auto, mascara)
            score_visual = puntuar_prompts_por_respuesta_visual(
                prompts_auto,
                info_eje.get("score_img"),
            )
            penal_salto = penalizar_ruta_por_saltos(info_eje_lim, W)

            fila = {
                "split": split,
                "patient_id": patient_id,
                "tipo_real": tipo_desde_patient_id(patient_id),
                "ruta": nombre_ruta,
                "offset_frac": offset_frac,
                "x_prior": float(info_eje.get("x_prior_usado", x_prior)),
                "bbox_iou_promedio": df_cajas["bbox_iou"].mean(),
                "bbox_recall_promedio": df_cajas["bbox_recall"].mean(),
                "bbox_precision_promedio": df_cajas["bbox_precision"].mean(),
                "bbox_iou_min": df_cajas["bbox_iou"].min(),
                "n_vertebras_iou_mayor_02": int((df_cajas["bbox_iou"] >= 0.20).sum()),
                "n_vertebras_recall_mayor_08": int((df_cajas["bbox_recall"] >= 0.80).sum()),
                "dx_abs_promedio": df_diag["dx_box_gt"].abs().mean() if len(df_diag) else np.nan,
                "dy_abs_promedio": df_diag["dy_box_gt"].abs().mean() if len(df_diag) else np.nan,
                "score_visual_cajas": score_visual,
                "penal_salto_ruta": penal_salto,
                "score_sin_gt": score_visual - 0.35 * penal_salto if pd.notna(score_visual) and pd.notna(penal_salto) else np.nan,
                "error": "",
            }

            detalles[nombre_ruta] = {
                "prompts": prompts_auto,
                "info_eje": info_eje_lim,
                "df_cajas": df_cajas,
                "df_diag": df_diag,
                "df_debug": df_debug,
                "imagen": imagen,
                "mascara": mascara,
            }

        except Exception as exc:
            fila = {
                "split": split,
                "patient_id": patient_id,
                "tipo_real": tipo_desde_patient_id(patient_id),
                "ruta": nombre_ruta,
                "offset_frac": offset_frac,
                "x_prior": x_prior,
                "bbox_iou_promedio": np.nan,
                "bbox_recall_promedio": np.nan,
                "bbox_precision_promedio": np.nan,
                "bbox_iou_min": np.nan,
                "n_vertebras_iou_mayor_02": 0,
                "n_vertebras_recall_mayor_08": 0,
                "dx_abs_promedio": np.nan,
                "dy_abs_promedio": np.nan,
                "score_visual_cajas": np.nan,
                "penal_salto_ruta": np.nan,
                "score_sin_gt": np.nan,
                "error": repr(exc),
            }

        filas.append(fila)

    df_rutas = pd.DataFrame(filas)
    df_rutas = df_rutas.sort_values(
        ["bbox_iou_promedio", "n_vertebras_recall_mayor_08", "score_sin_gt"],
        ascending=[False, False, False],
    ).reset_index(drop=True)

    return df_rutas, detalles


def visualizar_multi_ruta_muestra(
    split="val",
    patient_id="S_187",
    top_k=4,
    ordenar_por="bbox_iou_promedio",
):
    df_rutas, detalles = evaluar_multi_ruta_muestra(split=split, patient_id=patient_id)
    display(df_rutas)

    rutas_validas = df_rutas[df_rutas["ruta"].isin(detalles.keys())].copy()
    rutas_validas = rutas_validas.sort_values(ordenar_por, ascending=False).head(top_k)

    for _, row in rutas_validas.iterrows():
        ruta = row["ruta"]
        det = detalles[ruta]
        visualizar_cajas_auto_bordes(
            img_rgb=det["imagen"],
            prompts_auto=det["prompts"],
            info_eje=det["info_eje"],
            mascara_gt=det["mascara"],
            titulo=(
                f"{patient_id} | {ruta} | "
                f"IoU={row['bbox_iou_promedio']:.3f} | "
                f"recall={row['bbox_recall_promedio']:.3f} | "
                f"score_sin_gt={row['score_sin_gt']:.3f}"
            ),
        )

    return df_rutas, detalles


def correr_multi_ruta_largo(
    split="val",
    patient_ids=None,
    solo_escoliosis=False,
    checkpoint_csv="resultados_multi_ruta_checkpoint.csv",
    guardar_cada=5,
):
    """
    Corrida larga robusta.
    - No se cae por un paciente: guarda el error y continua.
    - Va guardando checkpoint CSV para revisar progreso si lo dejas corriendo.
    """

    if patient_ids is None:
        patient_ids = sorted(list(PROMPTS_DICC[split].keys()))

    if solo_escoliosis:
        patient_ids = [p for p in patient_ids if tipo_desde_patient_id(p) == "escoliosis"]

    acumulado = []
    errores = []

    for i, patient_id in enumerate(tqdm(patient_ids, desc="multi-ruta")):
        try:
            df_rutas, _ = evaluar_multi_ruta_muestra(split=split, patient_id=patient_id)
            acumulado.append(df_rutas)
        except Exception as exc:
            errores.append({
                "split": split,
                "patient_id": patient_id,
                "error": repr(exc),
            })

        if acumulado and ((i + 1) % guardar_cada == 0):
            pd.concat(acumulado, ignore_index=True).to_csv(checkpoint_csv, index=False)

    df_all = pd.concat(acumulado, ignore_index=True) if acumulado else pd.DataFrame()
    df_err = pd.DataFrame(errores)

    if not df_all.empty:
        df_all.to_csv(checkpoint_csv, index=False)

    return df_all, df_err


EJECUTAR_DIAGNOSTICO_S187_MULTIRUTA = False

if EJECUTAR_DIAGNOSTICO_S187_MULTIRUTA:
    df_s187_multiruta, detalles_s187_multiruta = visualizar_multi_ruta_muestra(
        split="val",
        patient_id="S_187",
        top_k=4,
        ordenar_por="bbox_iou_promedio",
    )
else:
    print("Diagnostico S_187 multi-ruta apagado; activar solo si quieres revisar esa figura puntual.")

# Corrida sugerida para la noche:
# df_multiruta_largo, df_multiruta_errores = correr_multi_ruta_largo(
#     split="val",
#     solo_escoliosis=True,
#     checkpoint_csv="resultados_multi_ruta_escoliosis_val.csv",
#     guardar_cada=3,
# )
# display(df_multiruta_largo.sort_values(["patient_id", "bbox_iou_promedio"], ascending=[True, False]))
# display(df_multiruta_errores)

## 10C. Calibracion vertical larga y seleccion de escenarios
 
 Esta seccion descarta rutas muy laterales como apuesta principal y explora el error que vimos en S_187: cajas cerca en X pero desplazadas en Y. La corrida larga guarda checkpoint y permite reanudar.


In [ ]:
# ============================================================
# Calibracion vertical larga: aprender que mueve realmente las cajas
# ============================================================

# Leccion de S_187: dx moderado, dy muy alto. Probamos offsets verticales amplios.
OFFSET_Y_CALIBRACION = (-170, -145, -120, -95, -70, -45, -20, 5)
ESCALA_POS_Y_CALIBRACION = (0.82, 0.88, 0.94, 1.00, 1.06)
MID_LIFT_CALIBRACION = (0, 12, 24, 36)
ESCALA_W_H_CALIBRACION = (
    (1.35, 1.15),  # cajas mas precisas
    (1.55, 1.25),  # balance
    (1.75, 1.35),  # tolerancia para escoliosis
)

# Ejes que vale la pena probar segun lo observado:
# - central_robusto: venia siendo el mejor promedio en grillas previas.
# - bordes: baseline estable, especialmente normal.
# - multi cercano: S_187 mostro que el fallo no era irse a extremos laterales.
EJES_CALIBRACION = [
    # Baselines que han rendido mejor para normales o promedio global.
    {"nombre": "central_robusto", "tipo": "metodo", "metodo": "central_robusto"},
    {"nombre": "bordes", "tipo": "metodo", "metodo": "bordes"},
    {"nombre": "central", "tipo": "metodo", "metodo": "central"},

    # Mantener ruta dinamica: visualmente puede capturar mejor la curva escoliotica.
    {"nombre": "ruta_dinamica", "tipo": "metodo", "metodo": "ruta_dinamica"},

    # Multi-ruta cercana al eje. Evitamos extremos como apuesta principal, pero conservamos
    # suficientes variantes para no perder curvas prometedoras en escoliosis.
    {"nombre": "multi_-0.15", "tipo": "multi", "offset_frac": -0.15},
    {"nombre": "multi_-0.10", "tipo": "multi", "offset_frac": -0.10},
    {"nombre": "multi_-0.05", "tipo": "multi", "offset_frac": -0.05},
    {"nombre": "multi_+0.00", "tipo": "multi", "offset_frac": 0.00},
    {"nombre": "multi_+0.05", "tipo": "multi", "offset_frac": 0.05},
    {"nombre": "multi_+0.10", "tipo": "multi", "offset_frac": 0.10},
    {"nombre": "multi_+0.15", "tipo": "multi", "offset_frac": 0.15},
]


def estimar_info_eje_calibracion(img_rgb, eje_cfg, search_half_frac=0.16):
    H, W = img_rgb.shape[:2]

    if eje_cfg["tipo"] == "metodo":
        return estimar_eje_columna(
            img_rgb,
            template_bbox=template_bbox_auto,
            metodo=eje_cfg["metodo"],
        )

    if eje_cfg["tipo"] == "multi":
        x_prior_base = float(np.median(template_bbox_auto["cx_rel"].to_numpy(dtype=float) * W))
        x_prior = x_prior_base + float(eje_cfg["offset_frac"]) * W
        return estimar_eje_por_ruta_dinamica(
            img_rgb,
            template_bbox=template_bbox_auto,
            search_half_frac=search_half_frac,
            x_prior_override=x_prior,
            ruta_nombre=eje_cfg["nombre"],
        )

    raise ValueError(f"Tipo de eje no reconocido: {eje_cfg}")


def generar_grid_calibracion_vertical(
    offset_y_vals=OFFSET_Y_CALIBRACION,
    escala_pos_y_vals=ESCALA_POS_Y_CALIBRACION,
    mid_lift_vals=MID_LIFT_CALIBRACION,
    escala_w_h_vals=ESCALA_W_H_CALIBRACION,
):
    filas = []
    for offset_y in offset_y_vals:
        for escala_pos_y in escala_pos_y_vals:
            for mid_lift in mid_lift_vals:
                for escala_w, escala_h in escala_w_h_vals:
                    filas.append({
                        "escala_w": escala_w,
                        "escala_h": escala_h,
                        "peso_eje": 1.00,
                        "escala_pos_y": escala_pos_y,
                        "offset_y": offset_y,
                        "offset_x": -10,
                        "mid_lift": mid_lift,
                    })
    return filas


GRID_CALIBRACION_VERTICAL = generar_grid_calibracion_vertical()
print("Ejes calibracion:", [e["nombre"] for e in EJES_CALIBRACION])
print("Combinaciones verticales por eje:", len(GRID_CALIBRACION_VERTICAL))
print("Escenarios por paciente:", len(EJES_CALIBRACION) * len(GRID_CALIBRACION_VERTICAL))


def evaluar_calibracion_vertical_muestra(
    split,
    patient_id,
    ejes_calibracion=EJES_CALIBRACION,
    grid_params=GRID_CALIBRACION_VERTICAL,
):
    """
    Evalua muchos escenarios de cajas sin MedSAM.
    Usa GT para diagnosticar que familia de parametros corrige cada paciente.
    """

    path_img, path_mask = resolver_paths_muestra(split, patient_id)
    imagen = cargar_imagen(path_img)
    mascara = cargar_mascara(path_mask)
    H, W = imagen.shape[:2]

    filas = []
    detalles = {}
    eje_cache = {}

    for eje_cfg in ejes_calibracion:
        eje_nombre = eje_cfg["nombre"]

        try:
            eje_cache[eje_nombre] = estimar_info_eje_calibracion(imagen, eje_cfg)
        except Exception as exc:
            filas.append({
                "split": split,
                "patient_id": patient_id,
                "tipo_real": tipo_desde_patient_id(patient_id),
                "eje": eje_nombre,
                "escenario": f"{eje_nombre}_ERROR_EJE",
                "error": repr(exc),
            })
            continue

        for params in grid_params:
            try:
                prompts_auto, info_eje_lim, df_debug = generar_prompts_auto_desde_info_eje(
                    imagen,
                    template_bbox_auto,
                    eje_cache[eje_nombre],
                    **params,
                )
                df_cajas = evaluar_cobertura_cajas_por_clase(prompts_auto, mascara)
                df_diag = diagnosticar_desplazamiento_cajas(prompts_auto, mascara)
                score_visual = puntuar_prompts_por_respuesta_visual(
                    prompts_auto,
                    eje_cache[eje_nombre].get("score_img"),
                )
                penal_salto = penalizar_ruta_por_saltos(info_eje_lim, W)
                score_sin_gt = score_visual - 0.35 * penal_salto if pd.notna(score_visual) and pd.notna(penal_salto) else np.nan

                escenario = (
                    f"{eje_nombre}"
                    f"_oy{params['offset_y']}"
                    f"_sy{params['escala_pos_y']:.2f}"
                    f"_ml{params['mid_lift']}"
                    f"_w{params['escala_w']:.2f}"
                    f"_h{params['escala_h']:.2f}"
                )

                fila = {
                    "split": split,
                    "patient_id": patient_id,
                    "tipo_real": tipo_desde_patient_id(patient_id),
                    "eje": eje_nombre,
                    "escenario": escenario,
                    "bbox_iou_promedio": df_cajas["bbox_iou"].mean(),
                    "bbox_recall_promedio": df_cajas["bbox_recall"].mean(),
                    "bbox_precision_promedio": df_cajas["bbox_precision"].mean(),
                    "bbox_iou_min": df_cajas["bbox_iou"].min(),
                    "n_vertebras_iou_mayor_02": int((df_cajas["bbox_iou"] >= 0.20).sum()),
                    "n_vertebras_recall_mayor_08": int((df_cajas["bbox_recall"] >= 0.80).sum()),
                    "dx_abs_promedio": df_diag["dx_box_gt"].abs().mean() if len(df_diag) else np.nan,
                    "dy_abs_promedio": df_diag["dy_box_gt"].abs().mean() if len(df_diag) else np.nan,
                    "score_visual_cajas": score_visual,
                    "penal_salto_ruta": penal_salto,
                    "score_sin_gt": score_sin_gt,
                    "error": "",
                    **params,
                }

                filas.append(fila)

                # Guardar detalles solo de candidatos prometedores para no inflar memoria.
                if fila["bbox_iou_promedio"] >= 0.15 or fila["n_vertebras_iou_mayor_02"] >= 6:
                    detalles[escenario] = {
                        "prompts": prompts_auto,
                        "info_eje": info_eje_lim,
                        "df_cajas": df_cajas,
                        "df_diag": df_diag,
                        "df_debug": df_debug,
                        "imagen": imagen,
                        "mascara": mascara,
                    }

            except Exception as exc:
                fila = {
                    "split": split,
                    "patient_id": patient_id,
                    "tipo_real": tipo_desde_patient_id(patient_id),
                    "eje": eje_nombre,
                    "escenario": f"{eje_nombre}_ERROR_PARAMS",
                    "bbox_iou_promedio": np.nan,
                    "bbox_recall_promedio": np.nan,
                    "bbox_precision_promedio": np.nan,
                    "bbox_iou_min": np.nan,
                    "n_vertebras_iou_mayor_02": 0,
                    "n_vertebras_recall_mayor_08": 0,
                    "dx_abs_promedio": np.nan,
                    "dy_abs_promedio": np.nan,
                    "score_visual_cajas": np.nan,
                    "penal_salto_ruta": np.nan,
                    "score_sin_gt": np.nan,
                    "error": repr(exc),
                    **params,
                }
                filas.append(fila)

    df = pd.DataFrame(filas)
    if "bbox_iou_promedio" in df.columns:
        df = df.sort_values(
            ["bbox_iou_promedio", "n_vertebras_iou_mayor_02", "bbox_recall_promedio", "bbox_precision_promedio"],
            ascending=[False, False, False, False],
        ).reset_index(drop=True)

    return df, detalles


def resumir_calibracion_vertical(df_all):
    if df_all.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    df_ok = df_all[df_all["error"].fillna("") == ""].copy()
    df_ok = df_ok.dropna(subset=["bbox_iou_promedio"])

    idx_best = df_ok.groupby("patient_id")["bbox_iou_promedio"].idxmax()
    df_oracle = df_ok.loc[idx_best].sort_values("bbox_iou_promedio", ascending=False).reset_index(drop=True)

    df_por_eje = (
        df_ok
        .groupby(["tipo_real", "eje"], as_index=False)
        .agg(
            bbox_iou_promedio=("bbox_iou_promedio", "mean"),
            bbox_recall_promedio=("bbox_recall_promedio", "mean"),
            bbox_precision_promedio=("bbox_precision_promedio", "mean"),
            dy_abs_promedio=("dy_abs_promedio", "mean"),
            escenarios=("escenario", "count"),
        )
        .sort_values(["tipo_real", "bbox_iou_promedio"], ascending=[True, False])
    )

    df_top_escenarios = (
        df_ok
        .groupby(["tipo_real", "eje", "offset_y", "escala_pos_y", "mid_lift", "escala_w", "escala_h"], as_index=False)
        .agg(
            pacientes=("patient_id", "nunique"),
            bbox_iou_promedio=("bbox_iou_promedio", "mean"),
            bbox_recall_promedio=("bbox_recall_promedio", "mean"),
            bbox_precision_promedio=("bbox_precision_promedio", "mean"),
            dy_abs_promedio=("dy_abs_promedio", "mean"),
        )
        .sort_values(["tipo_real", "bbox_iou_promedio"], ascending=[True, False])
    )

    return df_oracle, df_por_eje, df_top_escenarios


def correr_calibracion_vertical_larga(
    split="val",
    patient_ids=None,
    solo_escoliosis=False,
    checkpoint_csv="resultados_calibracion_vertical_val.csv",
    errores_csv="errores_calibracion_vertical_val.csv",
    guardar_cada=1,
    reanudar=True,
):
    """
    Corrida larga robusta para dejar de noche.
    - Reanuda si existe checkpoint_csv.
    - Escribe resultados despues de cada paciente por defecto.
    - Si un paciente falla, lo registra y continua.
    """

    if patient_ids is None:
        patient_ids = sorted(list(PROMPTS_DICC[split].keys()))

    if solo_escoliosis:
        patient_ids = [p for p in patient_ids if tipo_desde_patient_id(p) == "escoliosis"]

    acumulado = []
    errores = []
    ya_hechos = set()

    if reanudar and Path(checkpoint_csv).exists():
        df_prev = pd.read_csv(checkpoint_csv)
        acumulado.append(df_prev)
        if "patient_id" in df_prev.columns:
            ya_hechos = set(df_prev["patient_id"].astype(str).unique())
        print(f"Reanudando: {len(ya_hechos)} pacientes ya estaban en {checkpoint_csv}")

    pendientes = [p for p in patient_ids if str(p) not in ya_hechos]
    print(f"Pacientes pendientes: {len(pendientes)} / {len(patient_ids)}")

    for i, patient_id in enumerate(tqdm(pendientes, desc="calibracion vertical")):
        try:
            df_muestra, _ = evaluar_calibracion_vertical_muestra(split=split, patient_id=patient_id)
            acumulado.append(df_muestra)
        except Exception as exc:
            errores.append({"split": split, "patient_id": patient_id, "error": repr(exc)})

        if acumulado and ((i + 1) % guardar_cada == 0):
            pd.concat(acumulado, ignore_index=True).to_csv(checkpoint_csv, index=False)
            if errores:
                pd.DataFrame(errores).to_csv(errores_csv, index=False)

    df_all = pd.concat(acumulado, ignore_index=True) if acumulado else pd.DataFrame()
    df_err = pd.DataFrame(errores)

    if not df_all.empty:
        df_all.to_csv(checkpoint_csv, index=False)
    if not df_err.empty:
        df_err.to_csv(errores_csv, index=False)

    df_oracle, df_por_eje, df_top_escenarios = resumir_calibracion_vertical(df_all)
    return df_all, df_err, df_oracle, df_por_eje, df_top_escenarios


def visualizar_mejores_calibracion_muestra(split="val", patient_id="S_187", top_k=4):
    df_muestra, detalles = evaluar_calibracion_vertical_muestra(split=split, patient_id=patient_id)
    display(df_muestra.head(20))

    for _, row in df_muestra.head(top_k).iterrows():
        escenario = row["escenario"]
        if escenario not in detalles:
            # Regenerar detalle si el candidato no habia superado el umbral de guardado.
            eje_cfg = next(e for e in EJES_CALIBRACION if e["nombre"] == row["eje"])
            path_img, path_mask = resolver_paths_muestra(split, patient_id)
            imagen = cargar_imagen(path_img)
            mascara = cargar_mascara(path_mask)
            info_eje = estimar_info_eje_calibracion(imagen, eje_cfg)
            params = {
                "escala_w": row["escala_w"],
                "escala_h": row["escala_h"],
                "peso_eje": row["peso_eje"],
                "escala_pos_y": row["escala_pos_y"],
                "offset_y": row["offset_y"],
                "offset_x": row["offset_x"],
                "mid_lift": row["mid_lift"],
            }
            prompts_auto, info_eje_lim, _ = generar_prompts_auto_desde_info_eje(
                imagen,
                template_bbox_auto,
                info_eje,
                **params,
            )
            det = {"imagen": imagen, "mascara": mascara, "prompts": prompts_auto, "info_eje": info_eje_lim}
        else:
            det = detalles[escenario]

        visualizar_cajas_auto_bordes(
            img_rgb=det["imagen"],
            prompts_auto=det["prompts"],
            info_eje=det["info_eje"],
            mascara_gt=det["mascara"],
            titulo=(
                f"{patient_id} | {escenario} | "
                f"IoU={row['bbox_iou_promedio']:.3f} | "
                f"recall={row['bbox_recall_promedio']:.3f} | "
                f"dy={row['dy_abs_promedio']:.1f}"
            ),
        )

    return df_muestra, detalles


EJECUTAR_DIAGNOSTICO_S187_CALIBRACION = False

if EJECUTAR_DIAGNOSTICO_S187_CALIBRACION:
    df_s187_calibracion, detalles_s187_calibracion = visualizar_mejores_calibracion_muestra(
        split="val",
        patient_id="S_187",
        top_k=4,
    )
else:
    print("Diagnostico S_187 calibracion amplia apagado; el pipeline dirigido es la corrida recomendada.")

# Corrida recomendada para dejar varias horas:
# df_calib_val, df_calib_err, df_calib_oracle, df_calib_ejes, df_calib_top = correr_calibracion_vertical_larga(
#     split="val",
#     solo_escoliosis=False,
#     checkpoint_csv="resultados_calibracion_vertical_val.csv",
#     errores_csv="errores_calibracion_vertical_val.csv",
#     guardar_cada=1,
#     reanudar=True,
# )
# display(df_calib_oracle)
# display(df_calib_ejes)
# display(df_calib_top.groupby("tipo_real").head(15))

## 10D. Prueba final corta MedSAM con escenarios calibrados
 
 Esta seccion toma los mejores escenarios de cajas de 10C y ejecuta MedSAM en pocos casos. Compara caja sola contra puntos positivos/negativos generados sin usar GT.


In [ ]:
# ============================================================
# Prueba final corta: cajas calibradas + MedSAM + puntos
# ============================================================

POINT_STRATEGIES_MEDSAM = ("box_only", "centro", "respuesta_local")
MAX_PACIENTES_MEDSAM_FINAL = 8
TOP_ESCENARIOS_POR_PACIENTE_MEDSAM = 3


def obtener_eje_cfg_por_nombre(eje_nombre):
    for eje_cfg in EJES_CALIBRACION:
        if eje_cfg["nombre"] == eje_nombre:
            return eje_cfg
    raise ValueError(f"No encuentro eje en EJES_CALIBRACION: {eje_nombre}")


def prompts_desde_fila_calibracion(split, patient_id, fila_escenario):
    path_img, path_mask = resolver_paths_muestra(split, patient_id)
    imagen = cargar_imagen(path_img)
    mascara = cargar_mascara(path_mask)

    eje_cfg = obtener_eje_cfg_por_nombre(fila_escenario["eje"])
    info_eje = estimar_info_eje_calibracion(imagen, eje_cfg)
    params = {
        "escala_w": float(fila_escenario["escala_w"]),
        "escala_h": float(fila_escenario["escala_h"]),
        "peso_eje": float(fila_escenario.get("peso_eje", 1.0)),
        "escala_pos_y": float(fila_escenario["escala_pos_y"]),
        "offset_y": float(fila_escenario["offset_y"]),
        "offset_x": float(fila_escenario.get("offset_x", -10)),
        "mid_lift": float(fila_escenario["mid_lift"]),
    }
    prompts_auto, info_eje_lim, df_debug = generar_prompts_auto_desde_info_eje(
        imagen,
        template_bbox_auto,
        info_eje,
        **params,
    )
    return imagen, mascara, prompts_auto, info_eje_lim, df_debug


def construir_score_prompt_visual(imagen):
    gray = imagen_a_gris_uint8(imagen)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray_eq = clahe.apply(gray)
    grad_x = np.abs(cv2.Sobel(gray_eq, cv2.CV_32F, 1, 0, ksize=3))
    grad_y = np.abs(cv2.Sobel(gray_eq, cv2.CV_32F, 0, 1, ksize=3))
    lap = np.abs(cv2.Laplacian(gray_eq, cv2.CV_32F, ksize=3))
    return (
        0.35 * normalizar_01(gray_eq) +
        0.25 * normalizar_01(grad_x) +
        0.25 * normalizar_01(grad_y) +
        0.15 * normalizar_01(lap)
    )


def generar_puntos_prompt_bbox(imagen, bbox_xyxy, strategy="box_only", score_img=None):
    """
    Genera puntos sin GT.
    - box_only: solo caja.
    - centro: punto positivo en el centro de la caja.
    - respuesta_local: positivo en maxima respuesta dentro de zona central; negativos en esquinas internas.
    """
    if strategy == "box_only":
        return None, None

    H, W = imagen.shape[:2]
    x0, y0, x1, y1 = [int(v) for v in bbox_xyxy]
    x0, x1 = max(0, x0), min(W - 1, x1)
    y0, y1 = max(0, y0), min(H - 1, y1)
    bw = max(1, x1 - x0 + 1)
    bh = max(1, y1 - y0 + 1)

    if strategy == "centro":
        pts = np.array([[(x0 + x1) / 2, (y0 + y1) / 2]], dtype=np.float32)
        labels = np.array([1], dtype=np.int32)
        return pts, labels

    if score_img is None:
        score_img = construir_score_prompt_visual(imagen)

    # Positivo: maxima respuesta en la region central para evitar esquinas/costillas.
    cx0 = int(round(x0 + 0.25 * bw))
    cx1 = int(round(x0 + 0.75 * bw))
    cy0 = int(round(y0 + 0.20 * bh))
    cy1 = int(round(y0 + 0.80 * bh))
    crop = score_img[cy0:cy1 + 1, cx0:cx1 + 1]
    if crop.size == 0:
        pos = [(x0 + x1) / 2, (y0 + y1) / 2]
    else:
        yy, xx = np.unravel_index(np.argmax(crop), crop.shape)
        pos = [float(cx0 + xx), float(cy0 + yy)]

    # Negativos: esquinas internas, donde usualmente queremos desalentar costillas/fondo dentro de cajas grandes.
    negs = [
        [x0 + 0.12 * bw, y0 + 0.12 * bh],
        [x1 - 0.12 * bw, y0 + 0.12 * bh],
        [x0 + 0.12 * bw, y1 - 0.12 * bh],
        [x1 - 0.12 * bw, y1 - 0.12 * bh],
    ]
    pts = np.array([pos] + negs, dtype=np.float32)
    labels = np.array([1, 0, 0, 0, 0], dtype=np.int32)
    return pts, labels


def reconstruir_mascara_semantica_medsam_con_puntos(
    imagen,
    mascara_gt_multiclase,
    prompts_sample,
    predictor,
    point_strategy="box_only",
    frac_x=0.04,
    frac_y=0.06,
    return_quality=False,
):
    predictor.set_image(imagen)
    h, w = imagen.shape[:2]
    mask_sem_pred = np.zeros((h, w), dtype=np.uint8)
    score_map = np.zeros((h, w), dtype=np.float32)
    pred_count = np.zeros((h, w), dtype=np.uint8)
    mask_sem_gt = construir_mask_multiclase_t1_l5(mascara_gt_multiclase)
    score_img = construir_score_prompt_visual(imagen) if point_strategy == "respuesta_local" else None
    detalles = []

    for vertebra_objetivo in CLASES_OBJETIVO:
        if vertebra_objetivo not in prompts_sample:
            continue

        prompt_info = prompts_sample[vertebra_objetivo]
        bbox_original = prompt_info["bbox_xyxy"]
        bbox_expandida = expand_bbox_xyxy_pequena(
            bbox_original,
            imagen.shape,
            frac_x=frac_x,
            frac_y=frac_y,
        )
        box_np = np.array(bbox_expandida, dtype=np.float32)[None, :]
        point_coords, point_labels = generar_puntos_prompt_bbox(
            imagen,
            bbox_expandida,
            strategy=point_strategy,
            score_img=score_img,
        )

        predict_kwargs = {"box": box_np, "multimask_output": False}
        if point_coords is not None:
            predict_kwargs["point_coords"] = point_coords
            predict_kwargs["point_labels"] = point_labels

        masks, scores, logits = predictor.predict(**predict_kwargs)
        mask_pred_bin = masks[0].astype(np.uint8)
        score_pred = float(scores[0])
        pred_count += mask_pred_bin

        id_local = CLASS_TO_ID[vertebra_objetivo]
        update_idx = (mask_pred_bin == 1) & (score_pred > score_map)
        mask_sem_pred[update_idx] = id_local
        score_map[update_idx] = score_pred

        mask_gt_bin = construir_mask_binaria_vertebra(mascara_gt_multiclase, vertebra_objetivo)
        inter = np.logical_and(mask_pred_bin == 1, mask_gt_bin == 1).sum()
        union = np.logical_or(mask_pred_bin == 1, mask_gt_bin == 1).sum()
        dice_v = (2 * inter + 1e-8) / (mask_pred_bin.sum() + mask_gt_bin.sum() + 1e-8)
        iou_v = (inter + 1e-8) / (union + 1e-8)

        detalles.append({
            "vertebra": vertebra_objetivo,
            "id_local": id_local,
            "id_real": VERTEBRA_TO_ID[vertebra_objetivo],
            "bbox_original": bbox_original,
            "bbox_expandida": bbox_expandida,
            "point_strategy": point_strategy,
            "n_points": 0 if point_coords is None else int(len(point_coords)),
            "score_medsam": score_pred,
            "pix_gt": int(mask_gt_bin.sum()),
            "pix_pred": int(mask_pred_bin.sum()),
            "dice": float(dice_v),
            "iou": float(iou_v),
            "pred_vacia": bool(mask_pred_bin.sum() == 0),
            "gt_vacia": bool(mask_gt_bin.sum() == 0),
        })

    quality = {
        "overlap_pixels": int((pred_count > 1).sum()),
        "pred_pixels": int((pred_count > 0).sum()),
        "overlap_fraction_pred": float((pred_count > 1).sum() / max((pred_count > 0).sum(), 1)),
        "n_predicciones_vacias": int(sum(d["pred_vacia"] for d in detalles)),
        "n_gt_vacias": int(sum(d["gt_vacia"] for d in detalles)),
    }

    if return_quality:
        return mask_sem_pred, mask_sem_gt, detalles, score_map, quality
    return mask_sem_pred, mask_sem_gt, detalles, score_map


def seleccionar_escenarios_medsam_final(
    df_calibracion,
    max_pacientes=MAX_PACIENTES_MEDSAM_FINAL,
    top_por_paciente=TOP_ESCENARIOS_POR_PACIENTE_MEDSAM,
):
    """
    Selecciona pocos escenarios para MedSAM:
    - top por paciente segun GT de cajas, para medir techo posible;
    - conserva normales y escoliosis;
    - prioriza pacientes con mejor evidencia geometrica.
    """
    if df_calibracion is None or df_calibracion.empty:
        return pd.DataFrame()

    df = df_calibracion.copy()
    df = df[df["error"].fillna("") == ""].dropna(subset=["bbox_iou_promedio"])
    if df.empty:
        return df

    mejores_paciente = (
        df.sort_values(["patient_id", "bbox_iou_promedio", "n_vertebras_iou_mayor_02"], ascending=[True, False, False])
        .groupby("patient_id")
        .head(top_por_paciente)
        .copy()
    )

    ranking_pacientes = (
        mejores_paciente.groupby(["patient_id", "tipo_real"], as_index=False)
        .agg(mejor_iou=("bbox_iou_promedio", "max"))
        .sort_values("mejor_iou", ascending=False)
    )

    seleccion_ids = []
    for tipo in ["normal", "escoliosis"]:
        ids_tipo = ranking_pacientes[ranking_pacientes["tipo_real"] == tipo]["patient_id"].head(max_pacientes // 2).tolist()
        seleccion_ids.extend(ids_tipo)

    if len(seleccion_ids) < max_pacientes:
        extra = [p for p in ranking_pacientes["patient_id"].tolist() if p not in seleccion_ids]
        seleccion_ids.extend(extra[:max_pacientes - len(seleccion_ids)])

    return mejores_paciente[mejores_paciente["patient_id"].isin(seleccion_ids)].reset_index(drop=True)


def evaluar_medsam_escenarios_calibrados(
    df_escenarios,
    predictor,
    point_strategies=POINT_STRATEGIES_MEDSAM,
    checkpoint_csv="resultados_medsam_final_calibrado.csv",
    guardar_cada=5,
):
    resultados = []
    detalles_all = []
    errores = []
    n = 0

    for _, row in tqdm(df_escenarios.iterrows(), total=len(df_escenarios), desc="MedSAM escenarios"):
        split = row["split"]
        patient_id = row["patient_id"]
        try:
            imagen, mascara, prompts_auto, info_eje, df_debug = prompts_desde_fila_calibracion(split, patient_id, row)
            df_cobertura = evaluar_cobertura_cajas_por_clase(prompts_auto, mascara)

            for point_strategy in point_strategies:
                out = reconstruir_mascara_semantica_medsam_con_puntos(
                    imagen=imagen,
                    mascara_gt_multiclase=mascara,
                    prompts_sample=prompts_auto,
                    predictor=predictor,
                    point_strategy=point_strategy,
                    frac_x=0.04,
                    frac_y=0.06,
                    return_quality=True,
                )
                mask_pred, mask_gt, detalles, score_map, quality = out
                dice_macro = dice_multiclase_promedio(mask_pred, mask_gt, list(range(1, N_CLASES + 1)))
                iou_macro = iou_multiclase_promedio(mask_pred, mask_gt, list(range(1, N_CLASES + 1)))

                res = {
                    "split": split,
                    "patient_id": patient_id,
                    "tipo_real": row["tipo_real"],
                    "escenario": row["escenario"],
                    "eje": row["eje"],
                    "point_strategy": point_strategy,
                    "dice_macro": dice_macro,
                    "iou_macro": iou_macro,
                    "bbox_iou_promedio": df_cobertura["bbox_iou"].mean(),
                    "bbox_recall_promedio": df_cobertura["bbox_recall"].mean(),
                    "bbox_precision_promedio": df_cobertura["bbox_precision"].mean(),
                    "quality_overlap_fraction_pred": quality["overlap_fraction_pred"],
                    "quality_pred_pixels": quality["pred_pixels"],
                    "quality_vacias": quality["n_predicciones_vacias"],
                    "offset_y": row["offset_y"],
                    "escala_pos_y": row["escala_pos_y"],
                    "mid_lift": row["mid_lift"],
                    "escala_w": row["escala_w"],
                    "escala_h": row["escala_h"],
                }
                resultados.append(res)

                df_det = pd.DataFrame(detalles)
                df_det["split"] = split
                df_det["patient_id"] = patient_id
                df_det["tipo_real"] = row["tipo_real"]
                df_det["escenario"] = row["escenario"]
                df_det["eje"] = row["eje"]
                detalles_all.append(df_det)

        except Exception as exc:
            errores.append({
                "split": row.get("split", ""),
                "patient_id": row.get("patient_id", ""),
                "escenario": row.get("escenario", ""),
                "error": repr(exc),
            })

        n += 1
        if resultados and n % guardar_cada == 0:
            pd.DataFrame(resultados).to_csv(checkpoint_csv, index=False)

    df_res = pd.DataFrame(resultados)
    df_det = pd.concat(detalles_all, ignore_index=True) if detalles_all else pd.DataFrame()
    df_err = pd.DataFrame(errores)
    if not df_res.empty:
        df_res.to_csv(checkpoint_csv, index=False)
    return df_res, df_det, df_err


# Flujo recomendado:
# 1) Corre primero 10C completo para obtener df_calib_val.
# 2) Luego activa esta bandera para evaluar MedSAM en pocos escenarios prometedores.
EJECUTAR_MEDSAM_FINAL_CALIBRADO = False

if EJECUTAR_MEDSAM_FINAL_CALIBRADO:
    if "df_calib_val" not in globals():
        if Path("resultados_calibracion_vertical_val.csv").exists():
            df_calib_val = pd.read_csv("resultados_calibracion_vertical_val.csv")
        else:
            raise RuntimeError("Primero corre 10C o genera resultados_calibracion_vertical_val.csv")

    df_escenarios_medsam_final = seleccionar_escenarios_medsam_final(
        df_calib_val,
        max_pacientes=MAX_PACIENTES_MEDSAM_FINAL,
        top_por_paciente=TOP_ESCENARIOS_POR_PACIENTE_MEDSAM,
    )
    display(df_escenarios_medsam_final)

    df_medsam_final, df_medsam_final_detalles, df_medsam_final_errores = evaluar_medsam_escenarios_calibrados(
        df_escenarios=df_escenarios_medsam_final,
        predictor=predictor,
        point_strategies=POINT_STRATEGIES_MEDSAM,
        checkpoint_csv="resultados_medsam_final_calibrado.csv",
        guardar_cada=3,
    )

    display(
        df_medsam_final
        .groupby(["tipo_real", "point_strategy"], as_index=False)
        .agg(
            pacientes=("patient_id", "nunique"),
            escenarios=("escenario", "nunique"),
            dice_macro=("dice_macro", "mean"),
            iou_macro=("iou_macro", "mean"),
            bbox_iou_promedio=("bbox_iou_promedio", "mean"),
            overlap=("quality_overlap_fraction_pred", "mean"),
        )
        .sort_values(["tipo_real", "dice_macro"], ascending=[True, False])
    )
    display(df_medsam_final.sort_values("dice_macro", ascending=False).head(30))
    display(df_medsam_final_errores)
else:
    print("EJECUTAR_MEDSAM_FINAL_CALIBRADO=False. Activalo despues de correr 10C o cargar su checkpoint.")

## 10E. Pipeline final nocturno
 
 Activa esta celda cuando quieras dejar corriendo la busqueda larga y una prueba corta de MedSAM sobre los mejores escenarios encontrados.


In [ ]:
# ============================================================
# Pipeline final nocturno: calibracion larga + MedSAM corto
# ============================================================

# Cambia a True cuando quieras dejarlo corriendo varias horas.
EJECUTAR_PIPELINE_FINAL_NOCTURNO = False

# Recomendacion: empezar con val completo. Si quieres concentrarte solo en escoliosis, cambia a True.
PIPELINE_SOLO_ESCOLIOSIS = False
PIPELINE_CHECKPOINT_CAJAS = "resultados_calibracion_vertical_val.csv"
PIPELINE_ERRORES_CAJAS = "errores_calibracion_vertical_val.csv"
PIPELINE_CHECKPOINT_MEDSAM = "resultados_medsam_final_calibrado.csv"

if EJECUTAR_PIPELINE_FINAL_NOCTURNO:
    print("Iniciando calibracion vertical larga...")
    df_calib_val, df_calib_err, df_calib_oracle, df_calib_ejes, df_calib_top = correr_calibracion_vertical_larga(
        split="val",
        solo_escoliosis=PIPELINE_SOLO_ESCOLIOSIS,
        checkpoint_csv=PIPELINE_CHECKPOINT_CAJAS,
        errores_csv=PIPELINE_ERRORES_CAJAS,
        guardar_cada=1,
        reanudar=True,
    )

    print("Resumen oracle por paciente: mejores cajas posibles dentro de la grilla.")
    display(df_calib_oracle)
    print("Resumen por familia de eje.")
    display(df_calib_ejes)
    print("Top escenarios por tipo real.")
    display(df_calib_top.groupby("tipo_real").head(20))
    display(df_calib_err)

    print("Seleccionando escenarios para MedSAM corto...")
    df_escenarios_medsam_final = seleccionar_escenarios_medsam_final(
        df_calib_val,
        max_pacientes=MAX_PACIENTES_MEDSAM_FINAL,
        top_por_paciente=TOP_ESCENARIOS_POR_PACIENTE_MEDSAM,
    )
    display(df_escenarios_medsam_final)

    print("Ejecutando MedSAM con box_only, centro y respuesta_local...")
    df_medsam_final, df_medsam_final_detalles, df_medsam_final_errores = evaluar_medsam_escenarios_calibrados(
        df_escenarios=df_escenarios_medsam_final,
        predictor=predictor,
        point_strategies=POINT_STRATEGIES_MEDSAM,
        checkpoint_csv=PIPELINE_CHECKPOINT_MEDSAM,
        guardar_cada=3,
    )

    df_medsam_resumen = (
        df_medsam_final
        .groupby(["tipo_real", "point_strategy"], as_index=False)
        .agg(
            pacientes=("patient_id", "nunique"),
            escenarios=("escenario", "nunique"),
            dice_macro=("dice_macro", "mean"),
            iou_macro=("iou_macro", "mean"),
            bbox_iou_promedio=("bbox_iou_promedio", "mean"),
            overlap=("quality_overlap_fraction_pred", "mean"),
            vacias=("quality_vacias", "mean"),
        )
        .sort_values(["tipo_real", "dice_macro"], ascending=[True, False])
    )

    display(df_medsam_resumen)
    display(df_medsam_final.sort_values("dice_macro", ascending=False).head(40))
    display(df_medsam_final_errores)
else:
    print("EJECUTAR_PIPELINE_FINAL_NOCTURNO=False. Cambialo a True para correr calibracion larga + MedSAM corto.")

## Configuracion recomendada para corrida final
 
 La corrida recomendada es 10F: busqueda dirigida por tipo + refinamiento + MedSAM corto. Mantiene ruta dinamica para escoliosis, evita rutas poco probables en normales y guarda checkpoints para reanudar.


## 10F. Busqueda dirigida por tipo y refinamiento
 
 Esta seccion reduce escenarios poco probables: normales usan estrategias estables; escoliosis conserva rutas dinamicas prometedoras y calibra verticalmente. Luego refina alrededor de los mejores candidatos.


In [ ]:
# ============================================================
# Busqueda dirigida: preservar curvas dinamicas y no gastar en escenarios poco probables
# ============================================================

# Normales: las curvas suelen ser suaves; conviene probar recetas estables y pocos offsets.
EJES_PROBABLES_NORMAL = [
    {"nombre": "bordes", "tipo": "metodo", "metodo": "bordes"},
    {"nombre": "central_robusto", "tipo": "metodo", "metodo": "central_robusto"},
]

GRID_PROBABLE_NORMAL = generar_grid_calibracion_vertical(
    offset_y_vals=(-55, -35, -15, 5),
    escala_pos_y_vals=(0.88, 0.94, 1.00),
    mid_lift_vals=(0, 12, 24),
    escala_w_h_vals=((1.35, 1.15), (1.55, 1.25)),
)

# Escoliosis: conservar dinamica como hipotesis principal de trayecto.
# central_robusto queda como fallback porque fue fuerte en metricas previas.
EJES_PROBABLES_ESCOLIOSIS = [
    {"nombre": "ruta_dinamica", "tipo": "metodo", "metodo": "ruta_dinamica"},
    {"nombre": "multi_-0.10", "tipo": "multi", "offset_frac": -0.10},
    {"nombre": "multi_-0.05", "tipo": "multi", "offset_frac": -0.05},
    {"nombre": "multi_+0.00", "tipo": "multi", "offset_frac": 0.00},
    {"nombre": "multi_+0.05", "tipo": "multi", "offset_frac": 0.05},
    {"nombre": "multi_+0.10", "tipo": "multi", "offset_frac": 0.10},
    {"nombre": "central_robusto", "tipo": "metodo", "metodo": "central_robusto"},
]

GRID_PROBABLE_ESCOLIOSIS = generar_grid_calibracion_vertical(
    offset_y_vals=(-170, -145, -120, -95, -70, -45, -20),
    escala_pos_y_vals=(0.82, 0.88, 0.94, 1.00),
    mid_lift_vals=(0, 12, 24, 36),
    escala_w_h_vals=((1.35, 1.15), (1.55, 1.25), (1.75, 1.35)),
)

print("Escenarios probables normal:", len(EJES_PROBABLES_NORMAL) * len(GRID_PROBABLE_NORMAL))
print("Escenarios probables escoliosis:", len(EJES_PROBABLES_ESCOLIOSIS) * len(GRID_PROBABLE_ESCOLIOSIS))


def ejes_y_grid_probables_por_tipo(tipo_real):
    if tipo_real == "escoliosis":
        return EJES_PROBABLES_ESCOLIOSIS, GRID_PROBABLE_ESCOLIOSIS
    return EJES_PROBABLES_NORMAL, GRID_PROBABLE_NORMAL


def clip_param(valor, minimo, maximo):
    return max(minimo, min(maximo, valor))


def generar_grid_refinamiento_alrededor(fila, tipo_real):
    """
    Segunda pasada: no explora todo de nuevo; refina alrededor de un candidato prometedor.
    """
    offset_base = float(fila["offset_y"])
    escala_y_base = float(fila["escala_pos_y"])
    mid_base = float(fila["mid_lift"])
    w_base = float(fila["escala_w"])
    h_base = float(fila["escala_h"])

    if tipo_real == "escoliosis":
        offset_vals = sorted(set(int(round(offset_base + d)) for d in (-24, -12, 0, 12, 24)))
        escala_y_vals = sorted(set(round(clip_param(escala_y_base + d, 0.76, 1.08), 2) for d in (-0.04, -0.02, 0, 0.02, 0.04)))
        mid_vals = sorted(set(int(round(clip_param(mid_base + d, 0, 48))) for d in (-12, 0, 12)))
    else:
        offset_vals = sorted(set(int(round(offset_base + d)) for d in (-12, 0, 12)))
        escala_y_vals = sorted(set(round(clip_param(escala_y_base + d, 0.84, 1.04), 2) for d in (-0.02, 0, 0.02)))
        mid_vals = sorted(set(int(round(clip_param(mid_base + d, 0, 36))) for d in (-12, 0, 12)))

    escala_w_h_vals = sorted(set([
        (round(w_base, 2), round(h_base, 2)),
        (1.35, 1.15),
        (1.55, 1.25),
        (1.75, 1.35),
    ]))

    return generar_grid_calibracion_vertical(
        offset_y_vals=offset_vals,
        escala_pos_y_vals=escala_y_vals,
        mid_lift_vals=mid_vals,
        escala_w_h_vals=escala_w_h_vals,
    )


def evaluar_calibracion_dirigida_muestra(
    split,
    patient_id,
    top_n_refinar=5,
):
    tipo_real = tipo_desde_patient_id(patient_id)
    ejes_probables, grid_probable = ejes_y_grid_probables_por_tipo(tipo_real)

    df_coarse, detalles_coarse = evaluar_calibracion_vertical_muestra(
        split=split,
        patient_id=patient_id,
        ejes_calibracion=ejes_probables,
        grid_params=grid_probable,
    )
    df_coarse = df_coarse.copy()
    df_coarse["fase_busqueda"] = "probable_coarse"

    dfs = [df_coarse]
    detalles = dict(detalles_coarse)

    df_top = df_coarse[df_coarse["error"].fillna("") == ""].dropna(subset=["bbox_iou_promedio"]).head(top_n_refinar)

    for _, fila in df_top.iterrows():
        eje_cfg = next(e for e in ejes_probables if e["nombre"] == fila["eje"])
        grid_ref = generar_grid_refinamiento_alrededor(fila, tipo_real)
        df_ref, detalles_ref = evaluar_calibracion_vertical_muestra(
            split=split,
            patient_id=patient_id,
            ejes_calibracion=[eje_cfg],
            grid_params=grid_ref,
        )
        df_ref = df_ref.copy()
        df_ref["fase_busqueda"] = "refinamiento_top"
        df_ref["escenario_padre"] = fila["escenario"]
        dfs.append(df_ref)
        detalles.update(detalles_ref)

    df_all = pd.concat(dfs, ignore_index=True)
    df_all = df_all.sort_values(
        ["bbox_iou_promedio", "n_vertebras_iou_mayor_02", "bbox_recall_promedio", "bbox_precision_promedio"],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)

    return df_all, detalles


def correr_calibracion_dirigida_larga(
    split="val",
    patient_ids=None,
    checkpoint_csv="resultados_calibracion_dirigida_val.csv",
    errores_csv="errores_calibracion_dirigida_val.csv",
    guardar_cada=1,
    reanudar=True,
    top_n_refinar=5,
):
    if patient_ids is None:
        patient_ids = sorted(list(PROMPTS_DICC[split].keys()))

    acumulado = []
    errores = []
    ya_hechos = set()

    if reanudar and Path(checkpoint_csv).exists():
        df_prev = pd.read_csv(checkpoint_csv)
        acumulado.append(df_prev)
        if "patient_id" in df_prev.columns:
            ya_hechos = set(df_prev["patient_id"].astype(str).unique())
        print(f"Reanudando dirigida: {len(ya_hechos)} pacientes ya estaban en {checkpoint_csv}")

    pendientes = [p for p in patient_ids if str(p) not in ya_hechos]
    print(f"Pacientes pendientes dirigida: {len(pendientes)} / {len(patient_ids)}")

    for i, patient_id in enumerate(tqdm(pendientes, desc="calibracion dirigida")):
        try:
            df_muestra, _ = evaluar_calibracion_dirigida_muestra(
                split=split,
                patient_id=patient_id,
                top_n_refinar=top_n_refinar,
            )
            acumulado.append(df_muestra)
        except Exception as exc:
            errores.append({"split": split, "patient_id": patient_id, "error": repr(exc)})

        if acumulado and ((i + 1) % guardar_cada == 0):
            pd.concat(acumulado, ignore_index=True).to_csv(checkpoint_csv, index=False)
            if errores:
                pd.DataFrame(errores).to_csv(errores_csv, index=False)

    df_all = pd.concat(acumulado, ignore_index=True) if acumulado else pd.DataFrame()
    df_err = pd.DataFrame(errores)

    if not df_all.empty:
        df_all.to_csv(checkpoint_csv, index=False)
    if not df_err.empty:
        df_err.to_csv(errores_csv, index=False)

    df_oracle, df_por_eje, df_top_escenarios = resumir_calibracion_vertical(df_all)
    return df_all, df_err, df_oracle, df_por_eje, df_top_escenarios


def visualizar_dirigida_muestra(split="val", patient_id="S_187", top_k=4):
    df_muestra, detalles = evaluar_calibracion_dirigida_muestra(split=split, patient_id=patient_id)
    display(df_muestra.head(25))

    for _, row in df_muestra.head(top_k).iterrows():
        imagen, mascara, prompts_auto, info_eje, _ = prompts_desde_fila_calibracion(split, patient_id, row)
        visualizar_cajas_auto_bordes(
            img_rgb=imagen,
            prompts_auto=prompts_auto,
            info_eje=info_eje,
            mascara_gt=mascara,
            titulo=(
                f"{patient_id} | {row['fase_busqueda']} | {row['escenario']} | "
                f"IoU={row['bbox_iou_promedio']:.3f} | dy={row['dy_abs_promedio']:.1f}"
            ),
        )
    return df_muestra, detalles


def guardar_resumenes_decision_deteccion(
    df_all,
    prefijo="resultados_calibracion_dirigida_val",
):
    """Guarda tablas de decision para que la corrida larga deje norte aunque no miremos graficas."""
    if df_all is None or df_all.empty:
        print("No hay resultados para resumir.")
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    df_oracle, df_por_eje, df_top_escenarios = resumir_calibracion_vertical(df_all)

    df_ok = df_all[df_all["error"].fillna("") == ""].copy()
    df_ok = df_ok.dropna(subset=["bbox_iou_promedio"])

    df_decision = (
        df_ok
        .groupby(["tipo_real", "eje", "fase_busqueda"], as_index=False)
        .agg(
            pacientes=("patient_id", "nunique"),
            bbox_iou_promedio=("bbox_iou_promedio", "mean"),
            bbox_recall_promedio=("bbox_recall_promedio", "mean"),
            bbox_precision_promedio=("bbox_precision_promedio", "mean"),
            n_iou02=("n_vertebras_iou_mayor_02", "mean"),
            dy_abs_promedio=("dy_abs_promedio", "mean"),
        )
        .sort_values(["tipo_real", "bbox_iou_promedio"], ascending=[True, False])
    )

    df_mejor_por_paciente = (
        df_ok
        .sort_values(["patient_id", "bbox_iou_promedio", "n_vertebras_iou_mayor_02"], ascending=[True, False, False])
        .groupby("patient_id")
        .head(1)
        .reset_index(drop=True)
    )

    df_oracle.to_csv(f"{prefijo}_oracle_por_paciente.csv", index=False)
    df_por_eje.to_csv(f"{prefijo}_resumen_por_eje.csv", index=False)
    df_top_escenarios.to_csv(f"{prefijo}_top_escenarios.csv", index=False)
    df_decision.to_csv(f"{prefijo}_decision_por_tipo_eje.csv", index=False)
    df_mejor_por_paciente.to_csv(f"{prefijo}_mejor_por_paciente.csv", index=False)

    print("Resumenes guardados:")
    print(f"- {prefijo}_oracle_por_paciente.csv")
    print(f"- {prefijo}_resumen_por_eje.csv")
    print(f"- {prefijo}_top_escenarios.csv")
    print(f"- {prefijo}_decision_por_tipo_eje.csv")
    print(f"- {prefijo}_mejor_por_paciente.csv")

    return df_oracle, df_por_eje, df_top_escenarios, df_decision


EJECUTAR_DIAGNOSTICO_S187_DIRIGIDO = False

if EJECUTAR_DIAGNOSTICO_S187_DIRIGIDO:
    df_s187_dirigida, detalles_s187_dirigida = visualizar_dirigida_muestra(
        split="val",
        patient_id="S_187",
        top_k=4,
    )
else:
    print("Diagnostico S_187 dirigido apagado para priorizar la corrida larga.")

# Pipeline preferido para hoy: deteccion dirigida. MedSAM queda opcional y apagado.
EJECUTAR_PIPELINE_DIRIGIDO_8H = False
PIPELINE_DIRIGIDO_EJECUTAR_MEDSAM_CORTO = False
PIPELINE_DIRIGIDO_TOP_N_REFINAR = 8
PIPELINE_DIRIGIDO_PREFIJO = "resultados_calibracion_dirigida_val"

if EJECUTAR_PIPELINE_DIRIGIDO_8H:
    print("Corrida final de deteccion: dirigida por tipo + refinamiento alrededor de mejores candidatos.")
    df_dirigida_val, df_dirigida_err, df_dirigida_oracle, df_dirigida_ejes, df_dirigida_top = correr_calibracion_dirigida_larga(
        split="val",
        checkpoint_csv=f"{PIPELINE_DIRIGIDO_PREFIJO}.csv",
        errores_csv="errores_calibracion_dirigida_val.csv",
        guardar_cada=1,
        reanudar=True,
        top_n_refinar=PIPELINE_DIRIGIDO_TOP_N_REFINAR,
    )

    df_dirigida_oracle, df_dirigida_ejes, df_dirigida_top, df_dirigida_decision = guardar_resumenes_decision_deteccion(
        df_dirigida_val,
        prefijo=PIPELINE_DIRIGIDO_PREFIJO,
    )

    print("Oracle por paciente: techo posible de cajas dentro de esta busqueda.")
    display(df_dirigida_oracle)
    print("Decision por tipo/eje: tabla principal para escoger estrategia del proyecto.")
    display(df_dirigida_decision)
    print("Top escenarios por tipo real.")
    display(df_dirigida_top.groupby("tipo_real").head(20))
    display(df_dirigida_err)

    if PIPELINE_DIRIGIDO_EJECUTAR_MEDSAM_CORTO:
        df_escenarios_medsam_dirigido = seleccionar_escenarios_medsam_final(
            df_dirigida_val,
            max_pacientes=MAX_PACIENTES_MEDSAM_FINAL,
            top_por_paciente=TOP_ESCENARIOS_POR_PACIENTE_MEDSAM,
        )
        display(df_escenarios_medsam_dirigido)

        df_medsam_dirigido, df_medsam_dirigido_detalles, df_medsam_dirigido_errores = evaluar_medsam_escenarios_calibrados(
            df_escenarios=df_escenarios_medsam_dirigido,
            predictor=predictor,
            point_strategies=POINT_STRATEGIES_MEDSAM,
            checkpoint_csv="resultados_medsam_dirigido_final.csv",
            guardar_cada=3,
        )
        display(
            df_medsam_dirigido
            .groupby(["tipo_real", "point_strategy"], as_index=False)
            .agg(
                pacientes=("patient_id", "nunique"),
                dice_macro=("dice_macro", "mean"),
                iou_macro=("iou_macro", "mean"),
                bbox_iou_promedio=("bbox_iou_promedio", "mean"),
                overlap=("quality_overlap_fraction_pred", "mean"),
                vacias=("quality_vacias", "mean"),
            )
            .sort_values(["tipo_real", "dice_macro"], ascending=[True, False])
        )
        display(df_medsam_dirigido.sort_values("dice_macro", ascending=False).head(40))
        display(df_medsam_dirigido_errores)
    else:
        print("MedSAM corto apagado: esta corrida se concentra en deteccion de vertebras/cajas.")
else:
    print("EJECUTAR_PIPELINE_DIRIGIDO_8H=False. Cambialo a True para correr la busqueda mas probable de deteccion.")

## 10G. Estrategia final compacta de deteccion
 
 Implementa lo aprendido: normales con bordes/central robusto; escoliosis con curva dinamica/multi-ruta. Agrega correccion lumbar para casos donde L5 queda demasiado abajo y deja una prueba basica de MedSAM.


In [ ]:
# ============================================================
# Estrategia final compacta: deteccion de vertebras con correccion por tramos
# ============================================================

# Esta es la corrida recomendada ahora: corta, enfocada y basada en los resultados previos.
EJECUTAR_PRUEBA_FINAL_CORTA_DETECCION = True
EJECUTAR_MEDSAM_BASICO_FINAL = False

PREFIJO_FINAL_CORTA = "resultados_final_corta_deteccion_val"

# Lo aprendido:
# - normales: bordes/central_robusto, cajas pequenas.
# - escoliosis: dinamica/multi-ruta cercana + calibracion vertical.
# - L5 bajo sugiere desfase lumbar, no necesariamente mala curva.
EJES_FINAL_NORMAL = [
    {"nombre": "bordes", "tipo": "metodo", "metodo": "bordes"},
    {"nombre": "central_robusto", "tipo": "metodo", "metodo": "central_robusto"},
]

EJES_FINAL_ESCOLIOSIS = [
    {"nombre": "ruta_dinamica", "tipo": "metodo", "metodo": "ruta_dinamica"},
    {"nombre": "multi_-0.05", "tipo": "multi", "offset_frac": -0.05},
    {"nombre": "multi_+0.00", "tipo": "multi", "offset_frac": 0.00},
    {"nombre": "multi_+0.05", "tipo": "multi", "offset_frac": 0.05},
    {"nombre": "central_robusto", "tipo": "metodo", "metodo": "central_robusto"},
]


def generar_prompts_final_desde_info_eje(
    img_rgb,
    template_bbox,
    info_eje,
    escala_w=1.35,
    escala_h=1.15,
    peso_eje=1.00,
    escala_pos_y=1.00,
    offset_y=-20,
    offset_x=-10,
    mid_lift=24,
    global_lift=0,
    thoracic_lift=0,
    mid_spine_lift=0,
    lumbar_lift=0,
    min_w_frac=0.10,
    min_h_frac=0.035,
    max_w_frac=0.30,
    max_h_frac=0.12,
    limitar_rango_y_anatomico=True,
):
    """
    Genera cajas finales con correccion vertical por tramos.

    La curva puede estar bien pero las cajas pueden quedar corridas en varias zonas.
    Por eso se prueban cuatro componentes:
    - global_lift: sube todas las vertebras.
    - thoracic_lift: sube mas T1-T6.
    - mid_spine_lift: sube mas T7-L1.
    - lumbar_lift: sube mas L2-L5.
    """
    H, W = img_rgb.shape[:2]

    if limitar_rango_y_anatomico:
        info_eje = limitar_info_eje_a_template(
            info_eje,
            img_rgb.shape,
            template_bbox=template_bbox,
            margen_rel=0.05,
        )

    curva = info_eje["polinomio"]
    prompts_auto = {}
    filas_debug = []
    template_ordenado = template_bbox.sort_values("id_real").reset_index(drop=True)

    y_anchor = float(template_ordenado.iloc[0]["cy_rel"] * H)
    y_min_template = float(template_ordenado["cy_rel"].min() * H)
    y_max_template = float(template_ordenado["cy_rel"].max() * H)

    for _, row in template_ordenado.iterrows():
        vertebra = row["vertebra"]
        if vertebra not in CLASES_OBJETIVO:
            continue

        id_real = int(row["id_real"])
        cy_original = float(row["cy_rel"] * H)
        t = (cy_original - y_min_template) / (y_max_template - y_min_template + 1e-6)
        t = float(np.clip(t, 0, 1))

        correccion_media = mid_lift * np.sin(np.pi * t)

        # Correcciones por tramos: no asumimos que solo L5 esta desplazada.
        thoracic_w = float(np.clip((0.42 - t) / 0.42, 0, 1)) ** 1.2
        mid_w = float(np.clip(1.0 - abs(t - 0.58) / 0.30, 0, 1)) ** 1.2
        lumbar_w = float(np.clip((t - 0.62) / 0.38, 0, 1)) ** 1.3

        correccion_global = global_lift
        correccion_toracica = thoracic_lift * thoracic_w
        correccion_media_tramo = mid_spine_lift * mid_w
        correccion_lumbar = lumbar_lift * lumbar_w
        correccion_tramos = correccion_global + correccion_toracica + correccion_media_tramo + correccion_lumbar

        cy = y_anchor + escala_pos_y * (cy_original - y_anchor) + offset_y - correccion_media - correccion_tramos
        cy = float(np.clip(cy, 0, H - 1))

        cx_eje = float(curva(cy))
        cx_template = float(row["cx_rel"] * W)
        cx = peso_eje * cx_eje + (1 - peso_eje) * cx_template + offset_x
        cx = float(np.clip(cx, 0, W - 1))

        bw = float(row["w_rel"] * W * escala_w)
        bh = float(row["h_rel"] * H * escala_h)
        bw = float(np.clip(bw, W * min_w_frac, W * max_w_frac))
        bh = float(np.clip(bh, H * min_h_frac, H * max_h_frac))

        x0 = int(round(cx - bw / 2))
        x1 = int(round(cx + bw / 2))
        y0 = int(round(cy - bh / 2))
        y1 = int(round(cy + bh / 2))

        x0 = max(0, x0)
        y0 = max(0, y0)
        x1 = min(W - 1, x1)
        y1 = min(H - 1, y1)

        prompts_auto[vertebra] = {
            "vertebra": vertebra,
            "id_real": id_real,
            "bbox_xyxy": [x0, y0, x1, y1],
            "prompt_origen": "bbox_final_compacta_lumbar",
        }

        filas_debug.append({
            "vertebra": vertebra,
            "id_real": id_real,
            "t_vertical": t,
            "cy_final": cy,
            "cx_eje": cx_eje,
            "cx_template": cx_template,
            "cx_final": cx,
            "correccion_media": correccion_media,
            "correccion_global": correccion_global,
            "correccion_toracica": correccion_toracica,
            "correccion_media_tramo": correccion_media_tramo,
            "correccion_lumbar": correccion_lumbar,
            "correccion_tramos": correccion_tramos,
            "global_lift": global_lift,
            "thoracic_lift": thoracic_lift,
            "mid_spine_lift": mid_spine_lift,
            "lumbar_lift": lumbar_lift,
            "bbox_xyxy": [x0, y0, x1, y1],
        })

    return prompts_auto, info_eje, pd.DataFrame(filas_debug)


def generar_grid_final_normal():
    filas = []
    lift_sets = [
        {"global_lift": 0, "thoracic_lift": 0, "mid_spine_lift": 0, "lumbar_lift": 0},
        {"global_lift": 8, "thoracic_lift": 0, "mid_spine_lift": 0, "lumbar_lift": 0},
        {"global_lift": 0, "thoracic_lift": 0, "mid_spine_lift": 0, "lumbar_lift": 12},
    ]
    for offset_y in (-47, -35, -27, -23, -15, -7, 5):
        for escala_pos_y in (0.92, 0.96, 1.00, 1.02):
            for mid_lift in (0, 24, 36):
                for lifts in lift_sets:
                    filas.append({
                        "escala_w": 1.35,
                        "escala_h": 1.15,
                        "peso_eje": 1.00,
                        "escala_pos_y": escala_pos_y,
                        "offset_y": offset_y,
                        "offset_x": -10,
                        "mid_lift": mid_lift,
                        **lifts,
                    })
    return filas


def generar_grid_final_escoliosis():
    filas = []
    # Correcciones por tramos: varias vertebras pueden estar corridas, no solo L5.
    lift_sets = [
        {"global_lift": 0, "thoracic_lift": 0, "mid_spine_lift": 0, "lumbar_lift": 0},
        {"global_lift": 16, "thoracic_lift": 0, "mid_spine_lift": 0, "lumbar_lift": 0},
        {"global_lift": 32, "thoracic_lift": 0, "mid_spine_lift": 0, "lumbar_lift": 0},
        {"global_lift": 0, "thoracic_lift": 16, "mid_spine_lift": 0, "lumbar_lift": 0},
        {"global_lift": 0, "thoracic_lift": 0, "mid_spine_lift": 16, "lumbar_lift": 0},
        {"global_lift": 0, "thoracic_lift": 0, "mid_spine_lift": 0, "lumbar_lift": 24},
        {"global_lift": 16, "thoracic_lift": 0, "mid_spine_lift": 16, "lumbar_lift": 0},
        {"global_lift": 16, "thoracic_lift": 0, "mid_spine_lift": 0, "lumbar_lift": 24},
        {"global_lift": 0, "thoracic_lift": 12, "mid_spine_lift": 12, "lumbar_lift": 24},
    ]
    # Offsets concentrados en lo que salvo S_187 y otros casos, mas algunos cercanos a cero.
    for offset_y in (-194, -182, -170, -157, -145, -133, -120, -107, -95, -69, -45, -33, -20, -8, 4):
        for escala_pos_y in (0.84, 0.96, 1.00, 1.04):
            for mid_lift in (0, 24, 36, 48):
                for lifts in lift_sets:
                    for escala_w, escala_h in ((1.35, 1.15), (1.55, 1.25)):
                        filas.append({
                            "escala_w": escala_w,
                            "escala_h": escala_h,
                            "peso_eje": 1.00,
                            "escala_pos_y": escala_pos_y,
                            "offset_y": offset_y,
                            "offset_x": -10,
                            "mid_lift": mid_lift,
                            **lifts,
                        })
    return filas


GRID_FINAL_NORMAL = generar_grid_final_normal()
GRID_FINAL_ESCOLIOSIS = generar_grid_final_escoliosis()

print("Escenarios finales normal:", len(EJES_FINAL_NORMAL) * len(GRID_FINAL_NORMAL))
print("Escenarios finales escoliosis:", len(EJES_FINAL_ESCOLIOSIS) * len(GRID_FINAL_ESCOLIOSIS))


def ejes_y_grid_final_por_tipo(tipo_real):
    if tipo_real == "escoliosis":
        return EJES_FINAL_ESCOLIOSIS, GRID_FINAL_ESCOLIOSIS
    return EJES_FINAL_NORMAL, GRID_FINAL_NORMAL


def evaluar_final_corta_muestra(split, patient_id):
    tipo_real = tipo_desde_patient_id(patient_id)
    ejes_final, grid_final = ejes_y_grid_final_por_tipo(tipo_real)

    path_img, path_mask = resolver_paths_muestra(split, patient_id)
    imagen = cargar_imagen(path_img)
    mascara = cargar_mascara(path_mask)
    H, W = imagen.shape[:2]

    filas = []
    detalles = {}
    eje_cache = {}

    for eje_cfg in ejes_final:
        eje_nombre = eje_cfg["nombre"]
        try:
            eje_cache[eje_nombre] = estimar_info_eje_calibracion(imagen, eje_cfg)
        except Exception as exc:
            filas.append({
                "split": split,
                "patient_id": patient_id,
                "tipo_real": tipo_real,
                "eje": eje_nombre,
                "escenario": f"{eje_nombre}_ERROR_EJE",
                "error": repr(exc),
            })
            continue

        for params in grid_final:
            try:
                prompts_auto, info_eje_lim, df_debug = generar_prompts_final_desde_info_eje(
                    imagen,
                    template_bbox_auto,
                    eje_cache[eje_nombre],
                    **params,
                )
                df_cajas = evaluar_cobertura_cajas_por_clase(prompts_auto, mascara)
                df_diag = diagnosticar_desplazamiento_cajas(prompts_auto, mascara)
                score_visual = puntuar_prompts_por_respuesta_visual(
                    prompts_auto,
                    eje_cache[eje_nombre].get("score_img"),
                )
                penal_salto = penalizar_ruta_por_saltos(info_eje_lim, W)
                score_sin_gt = score_visual - 0.35 * penal_salto if pd.notna(score_visual) and pd.notna(penal_salto) else np.nan

                escenario = (
                    f"{eje_nombre}"
                    f"_oy{params['offset_y']}"
                    f"_sy{params['escala_pos_y']:.2f}"
                    f"_ml{params['mid_lift']}"
                    f"_gl{params.get('global_lift', 0)}"
                    f"_tl{params.get('thoracic_lift', 0)}"
                    f"_sl{params.get('mid_spine_lift', 0)}"
                    f"_ll{params.get('lumbar_lift', 0)}"
                    f"_w{params['escala_w']:.2f}"
                    f"_h{params['escala_h']:.2f}"
                )

                fila = {
                    "split": split,
                    "patient_id": patient_id,
                    "tipo_real": tipo_real,
                    "eje": eje_nombre,
                    "escenario": escenario,
                    "bbox_iou_promedio": df_cajas["bbox_iou"].mean(),
                    "bbox_recall_promedio": df_cajas["bbox_recall"].mean(),
                    "bbox_precision_promedio": df_cajas["bbox_precision"].mean(),
                    "bbox_iou_min": df_cajas["bbox_iou"].min(),
                    "n_vertebras_iou_mayor_02": int((df_cajas["bbox_iou"] >= 0.20).sum()),
                    "n_vertebras_recall_mayor_08": int((df_cajas["bbox_recall"] >= 0.80).sum()),
                    "dx_abs_promedio": df_diag["dx_box_gt"].abs().mean() if len(df_diag) else np.nan,
                    "dy_abs_promedio": df_diag["dy_box_gt"].abs().mean() if len(df_diag) else np.nan,
                    "dy_promedio_firmado": df_diag["dy_box_gt"].mean() if len(df_diag) else np.nan,
                    "score_visual_cajas": score_visual,
                    "penal_salto_ruta": penal_salto,
                    "score_sin_gt": score_sin_gt,
                    "error": "",
                    **params,
                }
                filas.append(fila)

                if fila["bbox_iou_promedio"] >= 0.25 or fila["n_vertebras_iou_mayor_02"] >= 14:
                    detalles[escenario] = {
                        "prompts": prompts_auto,
                        "info_eje": info_eje_lim,
                        "df_cajas": df_cajas,
                        "df_diag": df_diag,
                        "df_debug": df_debug,
                        "imagen": imagen,
                        "mascara": mascara,
                    }

            except Exception as exc:
                filas.append({
                    "split": split,
                    "patient_id": patient_id,
                    "tipo_real": tipo_real,
                    "eje": eje_nombre,
                    "escenario": f"{eje_nombre}_ERROR_PARAMS",
                    "bbox_iou_promedio": np.nan,
                    "bbox_recall_promedio": np.nan,
                    "bbox_precision_promedio": np.nan,
                    "bbox_iou_min": np.nan,
                    "n_vertebras_iou_mayor_02": 0,
                    "n_vertebras_recall_mayor_08": 0,
                    "dx_abs_promedio": np.nan,
                    "dy_abs_promedio": np.nan,
                    "dy_promedio_firmado": np.nan,
                    "score_visual_cajas": np.nan,
                    "penal_salto_ruta": np.nan,
                    "score_sin_gt": np.nan,
                    "error": repr(exc),
                    **params,
                })

    df = pd.DataFrame(filas)
    if "bbox_iou_promedio" in df.columns:
        df = df.sort_values(
            ["bbox_iou_promedio", "n_vertebras_iou_mayor_02", "bbox_recall_promedio", "bbox_precision_promedio"],
            ascending=[False, False, False, False],
        ).reset_index(drop=True)

    return df, detalles


def resumir_final_corta(df_all, prefijo=PREFIJO_FINAL_CORTA):
    if df_all is None or df_all.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    df_ok = df_all[df_all["error"].fillna("") == ""].copy()
    df_ok = df_ok.dropna(subset=["bbox_iou_promedio"])

    df_best = (
        df_ok
        .sort_values(["patient_id", "bbox_iou_promedio", "n_vertebras_iou_mayor_02"], ascending=[True, False, False])
        .groupby("patient_id")
        .head(1)
        .reset_index(drop=True)
    )

    df_decision = (
        df_ok
        .groupby(["tipo_real", "eje"], as_index=False)
        .agg(
            pacientes=("patient_id", "nunique"),
            bbox_iou_promedio=("bbox_iou_promedio", "mean"),
            bbox_recall_promedio=("bbox_recall_promedio", "mean"),
            bbox_precision_promedio=("bbox_precision_promedio", "mean"),
            n_iou02=("n_vertebras_iou_mayor_02", "mean"),
            dy_abs_promedio=("dy_abs_promedio", "mean"),
        )
        .sort_values(["tipo_real", "bbox_iou_promedio"], ascending=[True, False])
    )

    df_top_params = (
        df_ok
        .groupby(["tipo_real", "eje", "offset_y", "escala_pos_y", "mid_lift", "global_lift", "thoracic_lift", "mid_spine_lift", "lumbar_lift", "escala_w", "escala_h"], as_index=False)
        .agg(
            pacientes=("patient_id", "nunique"),
            bbox_iou_promedio=("bbox_iou_promedio", "mean"),
            bbox_recall_promedio=("bbox_recall_promedio", "mean"),
            bbox_precision_promedio=("bbox_precision_promedio", "mean"),
            dy_abs_promedio=("dy_abs_promedio", "mean"),
        )
        .sort_values(["tipo_real", "bbox_iou_promedio"], ascending=[True, False])
    )

    df_lumbar = (
        df_ok
        .groupby(["tipo_real", "global_lift", "thoracic_lift", "mid_spine_lift", "lumbar_lift"], as_index=False)
        .agg(
            escenarios=("escenario", "count"),
            bbox_iou_promedio=("bbox_iou_promedio", "mean"),
            dy_abs_promedio=("dy_abs_promedio", "mean"),
            dy_promedio_firmado=("dy_promedio_firmado", "mean"),
        )
        .sort_values(["tipo_real", "bbox_iou_promedio"], ascending=[True, False])
    )

    df_all.to_csv(f"{prefijo}.csv", index=False)
    df_best.to_csv(f"{prefijo}_mejor_por_paciente.csv", index=False)
    df_decision.to_csv(f"{prefijo}_decision_por_eje.csv", index=False)
    df_top_params.to_csv(f"{prefijo}_top_parametros.csv", index=False)
    df_lumbar.to_csv(f"{prefijo}_efecto_lifts_tramos.csv", index=False)

    print("Resumen final corto guardado:")
    print(f"- {prefijo}.csv")
    print(f"- {prefijo}_mejor_por_paciente.csv")
    print(f"- {prefijo}_decision_por_eje.csv")
    print(f"- {prefijo}_top_parametros.csv")
    print(f"- {prefijo}_efecto_lifts_tramos.csv")

    return df_best, df_decision, df_top_params, df_lumbar


def correr_prueba_final_corta_deteccion(
    split="val",
    patient_ids=None,
    checkpoint_csv=None,
    errores_csv="errores_final_corta_deteccion_val.csv",
    guardar_cada=1,
    reanudar=True,
):
    if checkpoint_csv is None:
        checkpoint_csv = f"{PREFIJO_FINAL_CORTA}.csv"

    if patient_ids is None:
        patient_ids = sorted(list(PROMPTS_DICC[split].keys()))

    acumulado = []
    errores = []
    ya_hechos = set()

    if reanudar and Path(checkpoint_csv).exists():
        df_prev = pd.read_csv(checkpoint_csv)
        acumulado.append(df_prev)
        if "patient_id" in df_prev.columns:
            ya_hechos = set(df_prev["patient_id"].astype(str).unique())
        print(f"Reanudando prueba final corta: {len(ya_hechos)} pacientes ya estaban en {checkpoint_csv}")

    pendientes = [p for p in patient_ids if str(p) not in ya_hechos]
    print(f"Pacientes pendientes final corta: {len(pendientes)} / {len(patient_ids)}")

    for i, patient_id in enumerate(tqdm(pendientes, desc="final corta deteccion")):
        try:
            df_muestra, _ = evaluar_final_corta_muestra(split=split, patient_id=patient_id)
            acumulado.append(df_muestra)
        except Exception as exc:
            errores.append({"split": split, "patient_id": patient_id, "error": repr(exc)})

        if acumulado and ((i + 1) % guardar_cada == 0):
            pd.concat(acumulado, ignore_index=True).to_csv(checkpoint_csv, index=False)
            if errores:
                pd.DataFrame(errores).to_csv(errores_csv, index=False)

    df_all = pd.concat(acumulado, ignore_index=True) if acumulado else pd.DataFrame()
    df_err = pd.DataFrame(errores)

    if not df_all.empty:
        df_all.to_csv(checkpoint_csv, index=False)
    if not df_err.empty:
        df_err.to_csv(errores_csv, index=False)

    df_best, df_decision, df_top_params, df_lumbar = resumir_final_corta(df_all, prefijo=PREFIJO_FINAL_CORTA)
    return df_all, df_err, df_best, df_decision, df_top_params, df_lumbar


def prompts_desde_fila_final(split, patient_id, fila_escenario):
    path_img, path_mask = resolver_paths_muestra(split, patient_id)
    imagen = cargar_imagen(path_img)
    mascara = cargar_mascara(path_mask)

    tipo_real = fila_escenario.get("tipo_real", tipo_desde_patient_id(patient_id))
    ejes_final, _ = ejes_y_grid_final_por_tipo(tipo_real)
    eje_cfg = next(e for e in ejes_final if e["nombre"] == fila_escenario["eje"])
    info_eje = estimar_info_eje_calibracion(imagen, eje_cfg)

    params = {
        "escala_w": float(fila_escenario["escala_w"]),
        "escala_h": float(fila_escenario["escala_h"]),
        "peso_eje": float(fila_escenario.get("peso_eje", 1.0)),
        "escala_pos_y": float(fila_escenario["escala_pos_y"]),
        "offset_y": float(fila_escenario["offset_y"]),
        "offset_x": float(fila_escenario.get("offset_x", -10)),
        "mid_lift": float(fila_escenario["mid_lift"]),
        "global_lift": float(fila_escenario.get("global_lift", 0)),
        "thoracic_lift": float(fila_escenario.get("thoracic_lift", 0)),
        "mid_spine_lift": float(fila_escenario.get("mid_spine_lift", 0)),
        "lumbar_lift": float(fila_escenario.get("lumbar_lift", 0)),
    }

    prompts_auto, info_eje_lim, df_debug = generar_prompts_final_desde_info_eje(
        imagen,
        template_bbox_auto,
        info_eje,
        **params,
    )
    return imagen, mascara, prompts_auto, info_eje_lim, df_debug


def evaluar_medsam_basico_final(
    df_best,
    predictor,
    max_pacientes=6,
    point_strategy="box_only",
    checkpoint_csv="resultados_medsam_basico_final.csv",
):
    resultados = []
    detalles_all = []
    errores = []

    df_eval = df_best.sort_values("bbox_iou_promedio", ascending=False).head(max_pacientes).copy()

    for _, row in tqdm(df_eval.iterrows(), total=len(df_eval), desc="MedSAM basico final"):
        try:
            imagen, mascara, prompts_auto, info_eje, df_debug = prompts_desde_fila_final(row["split"], row["patient_id"], row)
            df_cobertura = evaluar_cobertura_cajas_por_clase(prompts_auto, mascara)

            out = reconstruir_mascara_semantica_medsam_con_puntos(
                imagen=imagen,
                mascara_gt_multiclase=mascara,
                prompts_sample=prompts_auto,
                predictor=predictor,
                point_strategy=point_strategy,
                frac_x=0.04,
                frac_y=0.06,
                return_quality=True,
            )
            mask_pred, mask_gt, detalles, score_map, quality = out
            dice_macro = dice_multiclase_promedio(mask_pred, mask_gt, list(range(1, N_CLASES + 1)))
            iou_macro = iou_multiclase_promedio(mask_pred, mask_gt, list(range(1, N_CLASES + 1)))

            resultados.append({
                "split": row["split"],
                "patient_id": row["patient_id"],
                "tipo_real": row["tipo_real"],
                "eje": row["eje"],
                "escenario": row["escenario"],
                "point_strategy": point_strategy,
                "dice_macro": dice_macro,
                "iou_macro": iou_macro,
                "bbox_iou_promedio": df_cobertura["bbox_iou"].mean(),
                "bbox_recall_promedio": df_cobertura["bbox_recall"].mean(),
                "bbox_precision_promedio": df_cobertura["bbox_precision"].mean(),
                "quality_overlap_fraction_pred": quality["overlap_fraction_pred"],
                "quality_vacias": quality["n_predicciones_vacias"],
            })

            df_det = pd.DataFrame(detalles)
            df_det["patient_id"] = row["patient_id"]
            df_det["tipo_real"] = row["tipo_real"]
            df_det["eje"] = row["eje"]
            df_det["escenario"] = row["escenario"]
            detalles_all.append(df_det)

        except Exception as exc:
            errores.append({
                "patient_id": row.get("patient_id", ""),
                "escenario": row.get("escenario", ""),
                "error": repr(exc),
            })

    df_res = pd.DataFrame(resultados)
    df_det = pd.concat(detalles_all, ignore_index=True) if detalles_all else pd.DataFrame()
    df_err = pd.DataFrame(errores)

    if not df_res.empty:
        df_res.to_csv(checkpoint_csv, index=False)
    return df_res, df_det, df_err


if EJECUTAR_PRUEBA_FINAL_CORTA_DETECCION:
    df_final_corta, df_final_corta_err, df_final_corta_best, df_final_corta_decision, df_final_corta_top, df_final_corta_lumbar = correr_prueba_final_corta_deteccion(
        split="val",
        checkpoint_csv=f"{PREFIJO_FINAL_CORTA}.csv",
        errores_csv="errores_final_corta_deteccion_val.csv",
        guardar_cada=1,
        reanudar=True,
    )
    display(df_final_corta_best)
    display(df_final_corta_decision)
    display(df_final_corta_top.groupby("tipo_real").head(20))
    display(df_final_corta_lumbar)
    display(df_final_corta_err)
else:
    print("EJECUTAR_PRUEBA_FINAL_CORTA_DETECCION=False. Activalo para correr la estrategia final compacta.")

if EJECUTAR_MEDSAM_BASICO_FINAL:
    if "df_final_corta_best" not in globals():
        df_final_corta_best = pd.read_csv(f"{PREFIJO_FINAL_CORTA}_mejor_por_paciente.csv")

    df_medsam_basico_final, df_medsam_basico_final_detalles, df_medsam_basico_final_errores = evaluar_medsam_basico_final(
        df_best=df_final_corta_best,
        predictor=predictor,
        max_pacientes=6,
        point_strategy="box_only",
        checkpoint_csv="resultados_medsam_basico_final.csv",
    )
    display(df_medsam_basico_final)
    display(df_medsam_basico_final_errores)
else:
    print("MedSAM basico final apagado; activalo despues de revisar deteccion.")

## 11. Estrategia adaptativa por tipo geometrico
Esta seccion permite usar parametros distintos segun la prediccion normal/escoliosis antes de MedSAM. Es el puente entre clasificar la forma y generar cajas con una receta diferente.

In [ ]:
# ============================================================
# Estrategias adaptativas normal/escoliosis
# ============================================================

PARAMS_CAJA_NORMAL = {
    "escala_w": 1.40,
    "escala_h": 1.20,
    "peso_eje": 0.90,
    "escala_pos_y": 0.91,
    "offset_y": -16,
    "offset_x": -10,
    "mid_lift": 15,
    "metodo_eje": "bordes",
}

PARAMS_CAJA_ESCOLIOSIS = {
    # Punto de partida: mas peso al eje y un poco mas de tolerancia lateral.
    # Debe calibrarse con df_cmp_grupo cuando haya resultados por grupo.
    "escala_w": 1.70,
    "escala_h": 1.35,
    "peso_eje": 1.00,
    "escala_pos_y": 0.91,
    "offset_y": -16,
    "offset_x": -10,
    "mid_lift": 15,
    "metodo_eje": "ruta_dinamica",
}


def predecir_tipo_geom_imagen(img_rgb, umbral=UMBRAL_ESCOLIOSIS_GEOM):
    feats, info = extraer_features_eje_columna(img_rgb, template_bbox=template_bbox_auto)
    tipo_pred = "escoliosis" if feats["score_escoliosis_geom"] >= umbral else "normal"
    return tipo_pred, feats, info


def generar_prompts_auto_adaptativo(img_rgb, debug=False, usar_multiruta_escoliosis=True):
    tipo_pred, feats, info_pred = predecir_tipo_geom_imagen(img_rgb)

    if tipo_pred == "escoliosis" and usar_multiruta_escoliosis:
        H, W = img_rgb.shape[:2]
        x_prior_base = float(np.median(template_bbox_auto["cx_rel"].to_numpy(dtype=float) * W))
        candidatos = []

        for offset_frac in OFFSETS_MULTIRUTA_FRAC:
            nombre_ruta = f"adapt_multi_{offset_frac:+.2f}"
            x_prior = x_prior_base + float(offset_frac) * W
            info_eje = estimar_eje_por_ruta_dinamica(
                img_rgb,
                template_bbox=template_bbox_auto,
                search_half_frac=0.16,
                x_prior_override=x_prior,
                ruta_nombre=nombre_ruta,
            )
            params = {k: v for k, v in PARAMS_CAJA_ESCOLIOSIS.items() if k != "metodo_eje"}
            prompts_auto, info_eje_lim, df_debug = generar_prompts_auto_desde_info_eje(
                img_rgb,
                template_bbox_auto,
                info_eje,
                **params,
            )
            score_visual = puntuar_prompts_por_respuesta_visual(prompts_auto, info_eje.get("score_img"))
            penal_salto = penalizar_ruta_por_saltos(info_eje_lim, W)
            score_sin_gt = score_visual - 0.35 * penal_salto if pd.notna(score_visual) and pd.notna(penal_salto) else -np.inf
            candidatos.append({
                "score_sin_gt": score_sin_gt,
                "score_visual_cajas": score_visual,
                "penal_salto_ruta": penal_salto,
                "offset_frac": offset_frac,
                "prompts_auto": prompts_auto,
                "info_eje": info_eje_lim,
                "df_debug": df_debug,
            })

        mejor = sorted(candidatos, key=lambda x: x["score_sin_gt"], reverse=True)[0]
        df_debug = mejor["df_debug"].copy()
        df_debug["tipo_pred_geom"] = tipo_pred
        df_debug["score_escoliosis_geom"] = feats["score_escoliosis_geom"]
        df_debug["estrategia_adaptativa"] = "multi_ruta_sin_gt"
        df_debug["offset_frac_multiruta"] = mejor["offset_frac"]
        df_debug["score_sin_gt_multiruta"] = mejor["score_sin_gt"]

        if debug:
            return mejor["prompts_auto"], mejor["info_eje"], df_debug, tipo_pred, feats

        return mejor["prompts_auto"], mejor["info_eje"], tipo_pred, feats

    params = PARAMS_CAJA_ESCOLIOSIS if tipo_pred == "escoliosis" else PARAMS_CAJA_NORMAL
    salida = generar_prompts_auto_por_bordes(
        img_rgb=img_rgb,
        template_bbox=template_bbox_auto,
        debug=debug,
        **params,
    )

    if debug:
        prompts_auto, info_eje, df_debug = salida
        df_debug = df_debug.copy()
        df_debug["tipo_pred_geom"] = tipo_pred
        df_debug["score_escoliosis_geom"] = feats["score_escoliosis_geom"]
        df_debug["estrategia_adaptativa"] = "ruta_unica"
        return prompts_auto, info_eje, df_debug, tipo_pred, feats

    prompts_auto, info_eje = salida
    return prompts_auto, info_eje, tipo_pred, feats


def evaluar_cajas_adaptativas_muestra(split, patient_id):
    path_img, path_mask = resolver_paths_muestra(split, patient_id)
    imagen = cargar_imagen(path_img)
    mascara = cargar_mascara(path_mask)

    prompts_auto, info_eje, tipo_pred, feats = generar_prompts_auto_adaptativo(imagen, debug=False)
    df_cajas = evaluar_cobertura_cajas_por_clase(prompts_auto, mascara)
    df_diag = diagnosticar_desplazamiento_cajas(prompts_auto, mascara)

    resumen = {
        "split": split,
        "patient_id": patient_id,
        "tipo_real": tipo_desde_patient_id(patient_id),
        "tipo_pred_geom": tipo_pred,
        "score_escoliosis_geom": feats["score_escoliosis_geom"],
        "bbox_iou_promedio": df_cajas["bbox_iou"].mean(),
        "bbox_recall_promedio": df_cajas["bbox_recall"].mean(),
        "bbox_precision_promedio": df_cajas["bbox_precision"].mean(),
        "bbox_area_ratio_promedio": df_cajas["bbox_area_ratio"].mean(),
        "dx_abs_promedio": df_diag["dx_box_gt"].abs().mean() if len(df_diag) else np.nan,
        "dy_abs_promedio": df_diag["dy_box_gt"].abs().mean() if len(df_diag) else np.nan,
    }
    return resumen, df_cajas, prompts_auto, info_eje, imagen, mascara


def evaluar_cajas_adaptativas_split(split="val", max_items=None):
    patient_ids = sorted(list(PROMPTS_DICC[split].keys()))
    if max_items is not None:
        patient_ids = patient_ids[:max_items]

    resumenes = []
    detalles = []
    errores = []

    for patient_id in tqdm(patient_ids, desc=f"Cajas adaptativas - {split}"):
        try:
            resumen, df_cajas, *_ = evaluar_cajas_adaptativas_muestra(split, patient_id)
            resumenes.append(resumen)
            df_det = df_cajas.copy()
            df_det["split"] = split
            df_det["patient_id"] = patient_id
            df_det["tipo_real"] = resumen["tipo_real"]
            df_det["tipo_pred_geom"] = resumen["tipo_pred_geom"]
            detalles.append(df_det)
        except Exception as e:
            errores.append({"split": split, "patient_id": patient_id, "error": str(e)})

    return (
        pd.DataFrame(resumenes),
        pd.concat(detalles, ignore_index=True) if detalles else pd.DataFrame(),
        pd.DataFrame(errores),
    )


EJECUTAR_ADAPTATIVO_VAL = False  # Ya evaluado: bajo rendimiento en escoliosis; evitar gastar tiempo en Run All.

if EJECUTAR_ADAPTATIVO_VAL:
    df_adaptativo_val, df_adaptativo_detalles_val, df_adaptativo_errores_val = evaluar_cajas_adaptativas_split(
        split="val",
        max_items=None,
    )
    display(df_adaptativo_val)
    display(
        df_adaptativo_val
        .groupby(["tipo_real", "tipo_pred_geom"], as_index=False)
        .agg(
            pacientes=("patient_id", "nunique"),
            bbox_iou_promedio=("bbox_iou_promedio", "mean"),
            bbox_recall_promedio=("bbox_recall_promedio", "mean"),
            bbox_precision_promedio=("bbox_precision_promedio", "mean"),
        )
    )
    display(df_adaptativo_errores_val)
else:
    print("EJECUTAR_ADAPTATIVO_VAL=False. Activalo despues de revisar el clasificador y df_cmp_grupo.")


## 12. Evaluacion MedSAM de las mejores cajas
Esta seccion toma las mejores estrategias geometricas y, si activas la bandera, corre MedSAM sobre ellas. Es mas lenta que la grilla de cajas, pero conecta la calidad geometrica con Dice/IoU de segmentacion.

In [ ]:
# ============================================================
# Evaluacion opcional con MedSAM para las mejores estrategias
# ============================================================

EJECUTAR_TOP_MEDSAM = True
N_TOP_MEDSAM = 3
MAX_ITEMS_TOP_MEDSAM = 5

if EJECUTAR_TOP_MEDSAM:
    top_estrategias_medsam = df_cmp_estrategias["estrategia"].head(N_TOP_MEDSAM).tolist()
    resultados_top = []
    detalles_top = []
    errores_top = []

    for nombre in top_estrategias_medsam:
        estrategia = next(e for e in ESTRATEGIAS_CAJAS if e["nombre"] == nombre)
        params = {k: v for k, v in estrategia.items() if k != "nombre"}

        df_res, df_cob, df_det, df_err = evaluar_experimento12_split(
            split="val",
            max_items=MAX_ITEMS_TOP_MEDSAM,
            **params,
            frac_x=0.04,
            frac_y=0.06,
        )

        df_res["estrategia"] = nombre
        df_cob["estrategia"] = nombre
        df_det["estrategia"] = nombre
        if not df_err.empty:
            df_err["estrategia"] = nombre

        resultados_top.append(df_res)
        detalles_top.append(df_det)
        errores_top.append(df_err)

    df_top_medsam = pd.concat(resultados_top, ignore_index=True) if resultados_top else pd.DataFrame()
    df_top_medsam_detalles = pd.concat(detalles_top, ignore_index=True) if detalles_top else pd.DataFrame()
    df_top_medsam_errores = pd.concat(errores_top, ignore_index=True) if errores_top else pd.DataFrame()

    df_top_medsam_resumen = (
        df_top_medsam
        .groupby("estrategia", as_index=False)
        .agg(
            pacientes=("patient_id", "nunique"),
            dice_macro=("dice_macro", "mean"),
            iou_macro=("iou_macro", "mean"),
            bbox_iou_promedio=("bbox_iou_promedio", "mean"),
            bbox_precision_promedio=("bbox_precision_promedio", "mean"),
            bbox_recall_promedio=("bbox_recall_promedio", "mean"),
            quality_overlap_fraction_pred=("quality_overlap_fraction_pred", "mean"),
        )
        .sort_values("dice_macro", ascending=False)
        .reset_index(drop=True)
    )

    display(df_top_medsam_resumen)
    display(df_top_medsam)
    display(df_top_medsam_errores)
else:
    print("EJECUTAR_TOP_MEDSAM=False. Activalo cuando quieras probar MedSAM en las mejores estrategias geometricas.")


In [ ]:
EJECUTAR_VAL_COMPLETO = False

if EJECUTAR_VAL_COMPLETO:
    df_exp12_resumen_val, df_exp12_cobertura_val, df_exp12_detalles_val, df_exp12_errores_val = evaluar_experimento12_split(
        split="val",
        max_items=None,
        escala_w=1.90,
        escala_h=1.35,
        peso_eje=0.90,
        escala_pos_y=0.91,
        offset_y=-16,
        offset_x=-18,
        mid_lift=22,
        frac_x=0.04,
        frac_y=0.06,
    )

    display(df_exp12_resumen_val)
    display(df_exp12_cobertura_val)
    display(df_exp12_errores_val)

    print("Resumen experimento 12 en validacion")
    print(f"Dice macro promedio      : {df_exp12_resumen_val['dice_macro'].mean():.4f}")
    print(f"IoU macro promedio       : {df_exp12_resumen_val['iou_macro'].mean():.4f}")
    print(f"BBox recall promedio     : {df_exp12_resumen_val['bbox_recall_promedio'].mean():.4f}")
    print(f"BBox recall minimo prom. : {df_exp12_resumen_val['bbox_recall_min'].mean():.4f}")
else:
    print("EJECUTAR_VAL_COMPLETO=False. Activalo solo cuando quieras correr todo validation.")
